In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
import os
import json
import random
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILES = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json"
}

SEED = 42
random.seed(SEED)

source_paths = {}

for source_name,file_name in TRAIN_FILES.items():
    print("Downloading:",source_name)

    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=file_name,
        repo_type="dataset",
        token=HF_TOKEN
    )

    source_paths[source_name] = path

    print("Path:",path)
    print()
    
print("All four files downloaded/accessed successfully.")

Downloading: acts


Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json

Downloading: high_courts


Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json

Downloading: supreme_court


Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json

Downloading: tribunals


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json

All four files downloaded/accessed successfully.


In [81]:
TRAIN_FILES = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json"
}

source_paths = {}

for source_name,file_name in TRAIN_FILES.items():
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=file_name,
        repo_type="dataset",
        token=HF_TOKEN
    )

    source_paths[source_name] = path

    print(source_name)
    print(path)
    print()

acts
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json

high_courts
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json

supreme_court
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json

tribunals
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json



In [3]:
import json

def read_first_record(path):
    with open(path,"r",encoding="utf-8") as f:
        decoder = json.JSONDecoder()

        buffer = ""
        
        while True:
            chunk = f.read(1024 * 1024)
            
            if not chunk:
                break
            buffer += chunk
            buffer = buffer.lstrip()
            
            if buffer.startswith("["):
                buffer = buffer[1:]
                buffer = buffer.lstrip()
            
            try:
                record,end = decoder.raw_decode(buffer)
                return record
                
            except json.JSONDecodeError:
                continue

    return None

#Creating Manifest which stores the document id using streaming approach

In [83]:
import json
import random
import os

SEED = 42
VALID_RATIO = 0.10

random.seed(SEED)

OUTPUT_DIR = "/kaggle/working/legal_preprocessing"
os.makedirs(OUTPUT_DIR,exist_ok=True)

TRAIN_MANIFEST_PATH = os.path.join(OUTPUT_DIR,"train_manifest.json")

VALID_MANIFEST_PATH = os.path.join(OUTPUT_DIR,"valid_manifest.json")

In [84]:
def stream_json_array(path,chunk_size=1024*1024):
    decoder = json.JSONDecoder()

    with open(path,"r",encoding="utf-8") as f:
        buffer = ""
        started = False
        finished = False

        while not finished:
            chunk = f.read(chunk_size)

            if chunk:
                buffer += chunk
            else:
                finished = True

            while True:
                buffer = buffer.lstrip()

                if not started:
                    if len(buffer) == 0:
                        break

                    if buffer[0] != "[":
                        raise ValueError("Expected a JSON array.")

                    buffer = buffer[1:]
                    started = True

                buffer = buffer.lstrip()

                if len(buffer) == 0:
                    break

                if buffer[0] == "]":
                    return

                try:
                    record,end = decoder.raw_decode(buffer)
                except json.JSONDecodeError:
                    break

                yield record

                buffer = buffer[end:]
                buffer = buffer.lstrip()

                if len(buffer) > 0 and buffer[0] == ",":
                    buffer = buffer[1:]

In [8]:
train_manifest = []
valid_manifest = []

source_counts = {}

for source_name,path in source_paths.items():

    print("\nProcessing:",source_name)

    train_count = 0
    valid_count = 0
    document_number = 0

    for document in stream_json_array(path):

        document_id = source_name + "_" + str(document_number).zfill(6)

        entry = {
            "document_id": document_id,
            "source": source_name
        }

        if random.random() < VALID_RATIO:
            valid_manifest.append(entry)
            valid_count += 1
        else:
            train_manifest.append(entry)
            train_count += 1

        document_number += 1

    source_counts[source_name] = {
        "total": document_number,
        "train": train_count,
        "valid": valid_count
    }

    print("Total:",document_number)
    print("Train:",train_count)
    print("Valid:",valid_count)


Processing: acts
Total: 18826
Train: 16970
Valid: 1856

Processing: high_courts
Total: 2536738
Train: 2282377
Valid: 254361

Processing: supreme_court
Total: 55549
Train: 49895
Valid: 5654

Processing: tribunals
Total: 869132
Train: 782395
Valid: 86737


In [14]:
import os
import shutil

TRAIN_MANIFEST_SRC="/kaggle/input/datasets/spikie/capstone-split/train_manifest.json"
VALID_MANIFEST_SRC="/kaggle/input/datasets/spikie/capstone-split/valid_manifest.json"

WORK_DIR="/kaggle/working/legal_preprocessing"
os.makedirs(WORK_DIR,exist_ok=True)

shutil.copy2(TRAIN_MANIFEST_SRC,os.path.join(WORK_DIR,"train_manifest.json"))
shutil.copy2(VALID_MANIFEST_SRC,os.path.join(WORK_DIR,"valid_manifest.json"))

print("Manifests restored.")
print(os.listdir(WORK_DIR))

Manifests restored.
['valid_manifest.json', 'train_manifest.json']


In [85]:
import os
import json
import shutil
from collections import Counter

TRAIN_MANIFEST_SRC="/kaggle/input/datasets/spikie/capstone-split/train_manifest.json"
VALID_MANIFEST_SRC="/kaggle/input/datasets/spikie/capstone-split/valid_manifest.json"

WORK_DIR="/kaggle/working/legal_preprocessing"
os.makedirs(WORK_DIR,exist_ok=True)

TRAIN_MANIFEST_PATH=os.path.join(WORK_DIR,"train_manifest.json")
VALID_MANIFEST_PATH=os.path.join(WORK_DIR,"valid_manifest.json")

shutil.copy2(TRAIN_MANIFEST_SRC,TRAIN_MANIFEST_PATH)
shutil.copy2(VALID_MANIFEST_SRC,VALID_MANIFEST_PATH)

with open(TRAIN_MANIFEST_PATH,"r",encoding="utf-8") as f:
    train_manifest=json.load(f)

with open(VALID_MANIFEST_PATH,"r",encoding="utf-8") as f:
    valid_manifest=json.load(f)

source_counts=Counter(x["source"] for x in train_manifest)

print("TRAIN DOCUMENTS")
print("="*60)
print("Total:",len(train_manifest))

for source_name,count in source_counts.items():
    print(source_name,":",count)

print("\nVALID DOCUMENTS")
print("="*60)
print("Total:",len(valid_manifest))

TRAIN DOCUMENTS
Total: 3131637
acts : 16970
high_courts : 2282377
supreme_court : 49895
tribunals : 782395

VALID DOCUMENTS
Total: 348608


In [15]:
import os

for root,dirs,files in os.walk("/kaggle/input"):
    for file in files:
        if "manifest" in file.lower():
            path=os.path.join(root,file)
            print(path,os.path.getsize(path),"bytes")

/kaggle/input/datasets/spikie/capstone-split/train_manifest.json 242482340 bytes
/kaggle/input/datasets/spikie/capstone-split/valid_manifest.json 26995616 bytes


In [18]:
with open(TRAIN_MANIFEST_PATH,"w",encoding="utf-8") as f:
    json.dump(train_manifest,f,ensure_ascii=False,indent=2)

with open(VALID_MANIFEST_PATH,"w",encoding="utf-8") as f:
    json.dump(valid_manifest,f,ensure_ascii=False,indent=2)

print("Train manifest:")
print(TRAIN_MANIFEST_PATH)

print("\nValidation manifest:")
print(VALID_MANIFEST_PATH)

Train manifest:
/kaggle/working/legal_preprocessing/train_manifest.json

Validation manifest:
/kaggle/working/legal_preprocessing/valid_manifest.json


In [20]:
train_counts=Counter(x["source"] for x in train_manifest)
valid_counts=Counter(x["source"] for x in valid_manifest)

print("SOURCE STATISTICS")
print("="*60)

for source_name in train_counts:
    total=train_counts[source_name]+valid_counts[source_name]
    print(source_name,"total =",total,"train =",train_counts[source_name],"valid =",valid_counts[source_name])

SOURCE STATISTICS
acts total = 18826 train = 16970 valid = 1856
high_courts total = 2536738 train = 2282377 valid = 254361
supreme_court total = 55549 train = 49895 valid = 5654
tribunals total = 869132 train = 782395 valid = 86737


In [25]:
#Custom BPE Tokenizer adapted from A-1
from collections import Counter,defaultdict
import heapq
import re
from tqdm.auto import tqdm

class ByteLevelBPE:
    SPECIALS=["<eos>"]

    def __init__(self,vocab_size=32000):
        self.target_vocab_size=vocab_size
        self._merge_id_pairs=[]
        self.merges=[]
        self._token_bytes=[]
        self._bytes_to_id={}
        self._merge_ranks={}
        self._merge_to_new_id={}
        self.itos=[]
        self.stoi={}
        self.vocab_size=0
        self.eos_id=0

    @staticmethod
    def _get_word_pairs(symbols):
        return Counter(zip(symbols,symbols[1:]))

    @staticmethod
    def _merge_word(symbols,pair,new_id):
        left,right=pair
        merged=[]
        i=0

        while i<len(symbols):
            if i+1<len(symbols) and symbols[i]==left and symbols[i+1]==right:
                merged.append(new_id)
                i+=2
            else:
                merged.append(symbols[i])
                i+=1

        return merged

    def train_from_word_counts(self,word_freqs):
        self._token_bytes=[(byte,) for byte in range(256)]
        words=[]
        freqs=[]
        for word,freq in tqdm(word_freqs.items(),desc="Preparing BPE corpus"):
            symbols=list(word.encode("utf-8"))
            if len(symbols)>=2:
                words.append(symbols)
                freqs.append(freq)

        pair_counts=Counter()
        pair_to_words=defaultdict(set)
        for word_idx,symbols in tqdm(enumerate(words),total=len(words),desc="Building pair statistics"):
            word_freq=freqs[word_idx]
            for pair,occurrences in self._get_word_pairs(symbols).items():
                pair_counts[pair]+=occurrences*word_freq
                pair_to_words[pair].add(word_idx)

        heap=[]
        for pair,count in pair_counts.items():
            left,right=pair
            heap.append((-count,self._token_bytes[left],self._token_bytes[right],pair))
        heapq.heapify(heap)
        max_merges=self.target_vocab_size-256-len(self.SPECIALS)
        for _ in tqdm(range(max_merges),desc="Learning BPE merges"):
            best_pair=None
            while heap:
                neg_count,_,_,pair=heapq.heappop(heap)
                count=pair_counts.get(pair,0)
                if count==-neg_count and count>1:
                    best_pair=pair
                    break

            if best_pair is None:
                break

            left,right=best_pair
            new_id=len(self._token_bytes)
            self._token_bytes.append(self._token_bytes[left]+self._token_bytes[right])
            self._merge_id_pairs.append(best_pair)
            self.merges.append((self._token_bytes[left],self._token_bytes[right]))
            affected_words=list(pair_to_words.get(best_pair,set()))
            changed_pairs=set()
            for word_idx in affected_words:
                symbols=words[word_idx]
                word_freq=freqs[word_idx]
                new_symbols=[]
                i=0
                while i<len(symbols):
                    if i+1<len(symbols) and symbols[i]==left and symbols[i+1]==right:
                        new_symbols.append(new_id)
                        i+=2
                    else:
                        new_symbols.append(symbols[i])
                        i+=1

                if new_symbols==symbols:
                    continue
                old_pairs=self._get_word_pairs(symbols)
                new_pairs=self._get_word_pairs(new_symbols)
                for pair,occurrences in old_pairs.items():
                    pair_counts[pair]-=occurrences*word_freq
                    changed_pairs.add(pair)

                for pair,occurrences in new_pairs.items():
                    pair_counts[pair]+=occurrences*word_freq
                    changed_pairs.add(pair)

                for pair in old_pairs:
                    if pair not in new_pairs:
                        pair_to_words[pair].discard(word_idx)

                for pair in new_pairs:
                    pair_to_words[pair].add(word_idx)

                words[word_idx]=new_symbols
            pair_counts.pop(best_pair,None)
            pair_to_words.pop(best_pair,None)
            for pair in changed_pairs:
                count=pair_counts.get(pair,0)

                if count<=0:
                    pair_counts.pop(pair,None)
                    pair_to_words.pop(pair,None)
                else:
                    left_pair,right_pair=pair
                    heapq.heappush(heap,(-count,self._token_bytes[left_pair],self._token_bytes[right_pair],pair))

        self._bytes_to_id={token_bytes:idx for idx,token_bytes in enumerate(self._token_bytes)}
        self._merge_ranks={pair:rank for rank,pair in enumerate(self._merge_id_pairs)}
        self._merge_to_new_id={pair:256+rank for rank,pair in enumerate(self._merge_id_pairs)}
        self.itos=self.SPECIALS.copy()
        self.itos.extend(self._token_bytes)
        self.stoi={token:i for i,token in enumerate(self.itos)}
        self.eos_id=0
        self.vocab_size=len(self.itos)
        print("Learned merges:",len(self.merges))
        print("Final vocabulary size:",self.vocab_size)
        return self

    def encode_word(self,word):
        symbols=[byte for byte in word.encode("utf-8")]
        while len(symbols)>=2:
            best_rank=None
            best_pair=None
            for i in range(len(symbols)-1):
                pair=(symbols[i],symbols[i+1])
                rank=self._merge_ranks.get(pair)
                if rank is not None and (best_rank is None or rank<best_rank):
                    best_rank=rank
                    best_pair=pair

            if best_pair is None:
                break

            symbols=self._merge_word(symbols,best_pair,self._merge_to_new_id[best_pair])
        return [token_id+1 for token_id in symbols]

    def encode_text(self,text,add_eos=True):
        tokens=[]

        for part in re.findall(r"\s+|\S+",text):
            tokens.extend(self.encode_word(part))

        if add_eos:
            tokens.append(self.eos_id)

        return tokens

In [27]:
    def encode_text(self,text,add_eos=True):
        tokens=[]

        for part in re.findall(r"\s+|\S+",text):
            tokens.extend(self.encode_word(part))

        if add_eos:
            tokens.append(self.eos_id)

        return tokens

In [41]:
from collections import Counter
from huggingface_hub import hf_hub_download
import os
import re

SOURCE_FILES={
    "acts":"Phase_1_Pretraining/Train/train_acts.json",
    "high_courts":"Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court":"Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals":"Phase_1_Pretraining/Train/train_tribunals.json"
}

def get_words(text):
    return re.findall(r"\S+",text)

full_word_freqs=Counter()

for source_name,repo_path in SOURCE_FILES.items():
    path=hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=repo_path,
        repo_type="dataset"
    )

    for idx,document in enumerate(stream_json_array(path)):
        text=document.get("text","")
        full_word_freqs.update(get_words(text))

        if (idx+1)%100000==0:
            print(source_name,idx+1)

print("Unique words:",len(full_word_freqs))
print("Total word occurrences:",sum(full_word_freqs.values()))

high_courts 100000
high_courts 200000
high_courts 300000
high_courts 400000
high_courts 500000
high_courts 600000
high_courts 700000
high_courts 800000
high_courts 900000
high_courts 1000000
high_courts 1100000
high_courts 1200000
high_courts 1300000
high_courts 1400000
high_courts 1500000
high_courts 1600000
high_courts 1700000
high_courts 1800000
high_courts 1900000
high_courts 2000000
high_courts 2100000
high_courts 2200000
high_courts 2300000
high_courts 2400000
high_courts 2500000
tribunals 100000
tribunals 200000
tribunals 300000
tribunals 400000
tribunals 500000
tribunals 600000
tribunals 700000
tribunals 800000
Unique words: 19295686
Total word occurrences: 2900680187


In [42]:
top_2m_word_freqs=dict(full_word_freqs.most_common(2000000))

covered=sum(top_2m_word_freqs.values())
total=sum(full_word_freqs.values())

print("Unique words for BPE:",len(top_2m_word_freqs))
print("Occurrences represented:",covered)
print("Coverage:",round(100*covered/total,2),"%")

Unique words for BPE: 2000000
Occurrences represented: 2872094484
Coverage: 99.01 %


In [43]:
bpe_32k=ByteLevelBPE(vocab_size=32000)
bpe_32k.train_from_word_counts(top_2m_word_freqs)

Preparing BPE corpus:   0%|          | 0/2000000 [00:00<?, ?it/s]

Building pair statistics:   0%|          | 0/1999899 [00:00<?, ?it/s]

Learning BPE merges:   0%|          | 0/31743 [00:00<?, ?it/s]

Learned merges: 31743
Final vocabulary size: 32000


In [39]:
bpe=ByteLevelBPE(vocab_size=32000)
bpe.train_from_word_counts(top_word_freqs)

Preparing BPE corpus:   0%|          | 0/500000 [00:00<?, ?it/s]

Building pair statistics:   0%|          | 0/499903 [00:00<?, ?it/s]

Learning BPE merges:   0%|          | 0/31743 [00:00<?, ?it/s]

Learned merges: 31743
Final vocabulary size: 32000


In [44]:
bpe_64k=ByteLevelBPE(vocab_size=64000)
bpe_64k.train_from_word_counts(top_2m_word_freqs)

Preparing BPE corpus:   0%|          | 0/2000000 [00:00<?, ?it/s]

Building pair statistics:   0%|          | 0/1999899 [00:00<?, ?it/s]

Learning BPE merges:   0%|          | 0/63743 [00:00<?, ?it/s]

Learned merges: 63743
Final vocabulary size: 64000


In [46]:
import random

words=list(full_word_freqs.keys())
freqs=list(full_word_freqs.values())

rng=random.Random(42)
sample_words=rng.choices(words,weights=freqs,k=100000)

tokens_32=0
tokens_64=0

for word in sample_words:
    tokens_32+=len(bpe_32k.encode_word(word))
    tokens_64+=len(bpe_64k.encode_word(word))

print("Frequency-weighted words:",len(sample_words))
print("32K tokens:",tokens_32)
print("64K tokens:",tokens_64)
print("32K tokens/word:",round(tokens_32/len(sample_words),3))
print("64K tokens/word:",round(tokens_64/len(sample_words),3))
print("64K reduction:",round(100*(1-tokens_64/tokens_32),2),"%")

Frequency-weighted words: 100000
32K tokens: 116746
64K tokens: 111198
32K tokens/word: 1.167
64K tokens/word: 1.112
64K reduction: 4.75 %


In [40]:
import pickle

with open("/kaggle/working/legal_bpe_32k.pkl","wb") as f:
    pickle.dump(bpe,f)

print("Saved 32K BPE tokenizer")

Saved 32K BPE tokenizer


In [47]:
import pickle

with open("/kaggle/working/legal_bpe_32k.pkl","wb") as f:
    pickle.dump(bpe_32k,f)

with open("/kaggle/working/legal_bpe_64k.pkl","wb") as f:
    pickle.dump(bpe_64k,f)

print("32K tokenizer saved")
print("64K tokenizer saved")

32K tokenizer saved
64K tokenizer saved


In [125]:
#Preprocessing the datasets
import re,time
from tqdm.auto import tqdm

BENCH_DOCS=2000
cache={}
total_tokens=0
total_words=0
start=time.time()

for source in ["acts","high_courts","supreme_court","tribunals"]:
    path=source_paths[source]
    for i,doc in enumerate(tqdm(stream_json_array(path),desc=source)):
        if i>=BENCH_DOCS:
            break

        for part in re.findall(r"\s+|\S+",doc["text"]):
            if part not in cache:
                cache[part]=bpe.encode_word(part)
            total_tokens+=len(cache[part])
            total_words+=1

        total_tokens+=1

elapsed=time.time()-start
docs=BENCH_DOCS*4

print("\nBenchmark")
print("Documents:",docs)
print("Tokens:",total_tokens)
print("Unique segments:",len(cache))
print("Time:",round(elapsed,2),"sec")
print("Documents/sec:",round(docs/elapsed,2))
print("Tokens/sec:",round(total_tokens/elapsed,2))

total_train_docs=3131637
estimated_sec=total_train_docs/(docs/elapsed)

print("\nEstimated full TRAIN preprocessing time:")
print(round(estimated_sec/3600,2),"hours")
print(round(estimated_sec/60,2),"minutes")

acts: 0it [00:00, ?it/s]

high_courts: 0it [00:00, ?it/s]

supreme_court: 0it [00:00, ?it/s]

tribunals: 0it [00:00, ?it/s]


Benchmark
Documents: 8000
Tokens: 78098404
Unique segments: 592884
Time: 61.17 sec
Documents/sec: 130.79
Tokens/sec: 1276814.71

Estimated full TRAIN preprocessing time:
6.65 hours
399.07 minutes


In [126]:
import re,time,json,os
import numpy as np
from tqdm.auto import tqdm

BLOCK_SIZE=2048
OUT_DIR="/kaggle/working/legal_tokens"
os.makedirs(OUT_DIR,exist_ok=True)

cache={}
stats={"train":{},"valid":{}}
start_all=time.time()

def preprocess_split(manifest,split_name):
    out_path=os.path.join(OUT_DIR,f"{split_name}_tokens.bin")
    total_docs=0
    total_tokens=0
    total_blocks=0
    buffer=[]
    start=time.time()

    with open(out_path,"wb") as f:
        for source in ["acts","high_courts","supreme_court","tribunals"]:
            source_ids=[]
            for item in manifest:
                if item["source"]==source:
                    source_ids.append(int(item["document_id"].rsplit("_",1)[1]))

            source_ids.sort()
            ptr=0

            pbar=tqdm(total=len(source_ids),desc=f"{split_name} {source}")

            for i,doc in enumerate(stream_json_array(source_paths[source])):
                if ptr>=len(source_ids):
                    break

                if i!=source_ids[ptr]:
                    continue

                for part in re.findall(r"\s+|\S+",doc["text"]):
                    if part not in cache:
                        cache[part]=bpe.encode_word(part)
                    buffer.extend(cache[part])

                buffer.append(bpe.eos_id)

                while len(buffer)>=BLOCK_SIZE:
                    arr=np.asarray(buffer[:BLOCK_SIZE],dtype=np.uint16)
                    f.write(arr.tobytes())
                    del buffer[:BLOCK_SIZE]
                    total_blocks+=1
                    total_tokens+=BLOCK_SIZE

                total_docs+=1
                ptr+=1
                pbar.update(1)

            pbar.close()

    elapsed=time.time()-start

    stats[split_name]={
        "documents":total_docs,
        "tokens_written":total_tokens,
        "blocks":total_blocks,
        "leftover_tokens":len(buffer),
        "time_sec":elapsed,
        "cache_size":len(cache),
        "file_size_bytes":os.path.getsize(out_path)
    }

    print(f"\n{split_name} complete")
    print("Documents:",total_docs)
    print("Blocks:",total_blocks)
    print("Tokens written:",total_tokens)
    print("Leftover tokens:",len(buffer))
    print("File size:",round(os.path.getsize(out_path)/(1024**3),2),"GB")
    print("Time:",round(elapsed/3600,2),"hours")

print("Starting TRAIN preprocessing...")
preprocess_split(train_manifest,"train")

print("\nStarting VALID preprocessing...")
preprocess_split(valid_manifest,"valid")

stats["total_time_sec"]=time.time()-start_all

with open(os.path.join(OUT_DIR,"metadata.json"),"w") as f:
    json.dump({
        "vocab_size":VOCAB_SIZE,
        "block_size":BLOCK_SIZE,
        "dtype":"uint16",
        "eos_id":bpe.eos_id,
        "train":stats["train"],
        "valid":stats["valid"],
        "total_time_sec":stats["total_time_sec"]
    },f,indent=2)

print("\n========== DONE ==========")
print("Total time:",round(stats["total_time_sec"]/3600,2),"hours")
print("Cache size:",len(cache))
print("Output:",OUT_DIR)
print("\nFiles:")
for name in os.listdir(OUT_DIR):
    path=os.path.join(OUT_DIR,name)
    print(name,round(os.path.getsize(path)/(1024**3),2),"GB")

Starting TRAIN preprocessing...


train acts:   0%|          | 0/16970 [00:00<?, ?it/s]

train high_courts:   0%|          | 0/2282377 [00:00<?, ?it/s]

train supreme_court:   0%|          | 0/49895 [00:00<?, ?it/s]

train tribunals:   0%|          | 0/782395 [00:00<?, ?it/s]


train complete
Documents: 3131637
Blocks: 2799571
Tokens written: 5733521408
Leftover tokens: 1624
File size: 10.68 GB
Time: 1.17 hours

Starting VALID preprocessing...


valid acts:   0%|          | 0/1856 [00:00<?, ?it/s]

valid high_courts:   0%|          | 0/254361 [00:00<?, ?it/s]

valid supreme_court:   0%|          | 0/5654 [00:00<?, ?it/s]

valid tribunals:   0%|          | 0/86737 [00:00<?, ?it/s]


valid complete
Documents: 348608
Blocks: 312739
Tokens written: 640489472
Leftover tokens: 1890
File size: 1.19 GB
Time: 0.34 hours

========== DONE ==========
Total time: 1.51 hours
Cache size: 19295799
Output: /kaggle/working/legal_tokens

Files:
train_tokens.bin 10.68 GB
valid_tokens.bin 1.19 GB
metadata.json 0.0 GB


In [143]:
!kaggle datasets version \
    -p /kaggle/working/capstone_common_version \
    -m "Add pre-tokenized legal corpus for Phase 1" \
    -t \
    -r skip

Starting upload for file legal_bpe_64k.pkl
100%|██████████████████████████████████████| 4.40M/4.40M [00:02<00:00, 1.66MB/s]
Upload successful: legal_bpe_64k.pkl (4MB)
Starting upload for file train_manifest.json
100%|████████████████████████████████████████| 231M/231M [00:17<00:00, 14.1MB/s]
Upload successful: train_manifest.json (231MB)
Starting upload for file legal_bpe_32k.pkl
100%|███████████████████████████████████████| 2.16M/2.16M [00:02<00:00, 756kB/s]
Upload successful: legal_bpe_32k.pkl (2MB)
Starting upload for file train_tokens.bin
100%|██████████████████████████████████████| 10.7G/10.7G [08:24<00:00, 22.7MB/s]
Upload successful: train_tokens.bin (11GB)
Starting upload for file valid_tokens.bin
100%|██████████████████████████████████████| 1.19G/1.19G [01:04<00:00, 19.9MB/s]
Upload successful: valid_tokens.bin (1GB)
Starting upload for file metadata.json
100%|████████████████████████████████████████████| 575/575 [00:01<00:00, 364B/s]
Upload successful: metadata.json (575B)
St

In [ ]:
import os
import json
import math
import time
import random
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader,IterableDataset
from torch.amp import autocast,GradScaler

SEED=42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark=True

print("PyTorch:",torch.__version__)
print("CUDA:",torch.version.cuda)
print("GPU count:",torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i,torch.cuda.get_device_name(i))
    print("Memory:",round(torch.cuda.get_device_properties(i).total_memory/1024**3,2),"GB")


VOCAB_SIZE=32000
BLOCK_SIZE=2048

LEARNING_RATE=3e-4
WEIGHT_DECAY=0.1
WARMUP_RATIO=0.03
MAX_GRAD_NORM=1.0

DROPOUT=0.2

TRAIN_DOCS=None
VALID_DOCS=None

NUM_WORKERS=2

import sys
sys.path.insert(0,"/kaggle/working")
with open(BPE_PATH,"rb") as f:
    bpe=pickle.load(f)

print("Vocabulary size:",bpe.vocab_size)
print("EOS id:",bpe.eos_id)

In [87]:
COMMON_PATH=None

for root,dirs,files in os.walk("/kaggle/input"):
    if "legal_bpe_32k.pkl" in files and "train_manifest.json" in files:
        COMMON_PATH=root
        break

print("Common dataset:",COMMON_PATH)

with open(os.path.join(COMMON_PATH,"train_manifest.json"),"r",encoding="utf-8") as f:
    train_manifest=json.load(f)

with open(os.path.join(COMMON_PATH,"valid_manifest.json"),"r",encoding="utf-8") as f:
    valid_manifest=json.load(f)

print("Train documents:",len(train_manifest))
print("Validation documents:",len(valid_manifest))
print("Train sample:",train_manifest[0])
print("Valid sample:",valid_manifest[0])

Common dataset: /kaggle/input/datasets/spikie/capstone-split
Train documents: 3131637
Validation documents: 348608
Train sample: {'document_id': 'acts_000000', 'source': 'acts'}
Valid sample: {'document_id': 'acts_000001', 'source': 'acts'}


In [88]:
train_indices={
    "acts":set(),
    "high_courts":set(),
    "supreme_court":set(),
    "tribunals":set()
}

valid_indices={
    "acts":set(),
    "high_courts":set(),
    "supreme_court":set(),
    "tribunals":set()
}

for item in train_manifest:
    source=item["source"]
    index=int(item["document_id"].split("_")[-1])
    train_indices[source].add(index)

for item in valid_manifest:
    source=item["source"]
    index=int(item["document_id"].split("_")[-1])
    valid_indices[source].add(index)

for source in train_indices:
    print(
        source,
        "train:",len(train_indices[source]),
        "valid:",len(valid_indices[source])
    )

acts train: 16970 valid: 1856
high_courts train: 2282377 valid: 254361
supreme_court train: 49895 valid: 5654
tribunals train: 782395 valid: 86737


In [93]:
def stream_token_blocks(indices,max_docs=None):
    buffer=[]
    docs=0

    for text in stream_split_documents(indices,max_docs=max_docs):
        tokens=bpe.encode_text(text,add_eos=True)
        buffer.extend(tokens)
        while len(buffer)>=BLOCK_SIZE+1:
            yield (buffer[:BLOCK_SIZE],buffer[1:BLOCK_SIZE+1])

            buffer=buffer[BLOCK_SIZE:]

        docs+=1

    if len(buffer)>=BLOCK_SIZE+1:
        yield (buffer[:BLOCK_SIZE],buffer[1:BLOCK_SIZE+1])

blocks=stream_token_blocks(train_indices,max_docs=3)

x,y=next(blocks)

print("Input length:",len(x))
print("Target length:",len(y))
print("First 20 input:",x[:20])
print("First 20 target:",y[:20])
print("Shift correct:",x[1:]==y[:-1])

Input length: 2048
Target length: 2048
First 20 input: [4970, 33, 6627, 33, 10332, 33, 7266, 33, 10408, 33, 5378, 33, 11018, 2782, 33, 276, 33, 21307, 19773, 33]
First 20 target: [33, 6627, 33, 10332, 33, 7266, 33, 10408, 33, 5378, 33, 11018, 2782, 33, 276, 33, 21307, 19773, 33, 807]
Shift correct: True


In [ ]:
def get_document_text(source,doc):
    return doc["text"]

def stream_split_documents(indices,max_docs=None):
    count=0

    for source in source_paths:
        required=indices[source]

        if not required:
            continue

        for doc_idx,doc in enumerate(stream_json_array(source_paths[source])):
            if doc_idx not in required:
                continue

            text=get_document_text(source,doc)

            if text:
                yield text

            count+=1

            if max_docs is not None and count>=max_docs:
                return


for i,text in enumerate(stream_split_documents(train_indices,max_docs=3)):
    print("Document",i)
    print("Characters:",len(text))
    print(text[:300])
    print("-"*80)

In [94]:
import torch.distributed as dist

def get_ddp_info():
    if dist.is_available() and dist.is_initialized():
        return dist.get_rank(),dist.get_world_size()
    return 0,1

def stream_token_blocks_ddp(indices,max_docs=None):
    rank,world_size=get_ddp_info()
    indices=indices[rank::world_size]
    buffer=[]
    docs=0
    for text in stream_split_documents(indices,max_docs=max_docs):
        tokens=bpe.encode_text(text,add_eos=True)
        buffer.extend(tokens)
        while len(buffer)>=BLOCK_SIZE+1:
            yield buffer[:BLOCK_SIZE],buffer[1:BLOCK_SIZE+1]
            buffer=buffer[BLOCK_SIZE:]
        docs+=1
    if len(buffer)>=BLOCK_SIZE+1:
        yield buffer[:BLOCK_SIZE],buffer[1:BLOCK_SIZE+1]

class LegalBlockDatasetDDP(IterableDataset):
    def __init__(self,indices,max_docs=None):
        self.indices=indices
        self.max_docs=max_docs
    def __iter__(self):
        for x,y in stream_token_blocks_ddp(self.indices,max_docs=self.max_docs):
            yield torch.tensor(x,dtype=torch.long),torch.tensor(y,dtype=torch.long)

print("DDP data pipeline ready")
print("World size:",dist.get_world_size() if dist.is_initialized() else 1)

DDP data pipeline ready
World size: 1


In [95]:
def create_optimizer(model):
    return torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY,betas=(0.9,0.95),eps=1e-8)

def get_lr(step,total_steps):
    warmup_steps=max(1,int(WARMUP_RATIO*total_steps))
    if step<warmup_steps:
        return LEARNING_RATE*(step+1)/warmup_steps
    progress=(step-warmup_steps)/max(1,total_steps-warmup_steps)
    return LEARNING_RATE*0.5*(1+math.cos(math.pi*progress))

def set_lr(optimizer,lr):
    for param_group in optimizer.param_groups:
        param_group["lr"]=lr

print("Optimizer and LR helpers ready")

Optimizer and LR helpers ready


In [2]:
#MOdel 1
N_LAYERS=12
D_MODEL=768
N_HEADS=12
HEAD_DIM=D_MODEL//N_HEADS
FFN_DIM=3072

BATCH_SIZE=5
GLOBAL_BATCH_SIZE=10
BLOCK_SIZE=2048
VOCAB_SIZE=32000

LEARNING_RATE=3e-4
WEIGHT_DECAY=0.1
WARMUP_RATIO=0.03
MAX_GRAD_NORM=1.0
DROPOUT=0.2
NUM_WORKERS=0

TRAIN_PATH="/kaggle/input/datasets/spikie/capstone-split/train_tokens.bin"
VALID_PATH="/kaggle/input/datasets/spikie/capstone-split/valid_tokens.bin"

FP16=True
GPUS=2

class RMSNorm(nn.Module):
    def __init__(self,dim,eps=1e-6):
        super().__init__()
        self.weight=nn.Parameter(torch.ones(dim))
        self.eps=eps

    def forward(self,x):
        rms=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(rms+self.eps)
        return self.weight*x

class RotaryEmbedding(nn.Module):
    def __init__(self,dim,max_seq_len=BLOCK_SIZE,base=10000):
        super().__init__()

        inv_freq=1.0/(base**(torch.arange(0,dim,2).float()/dim))
        positions=torch.arange(max_seq_len).float()
        freqs=torch.outer(positions,inv_freq)
        self.register_buffer("cos",freqs.cos()[None,None,:,:])
        self.register_buffer("sin",freqs.sin()[None,None,:,:])
    def forward(self,q,k):
        seq_len=q.size(-2)
        cos=self.cos[:,:,:seq_len,:]
        sin=self.sin[:,:,:seq_len,:]
        q1=q[...,::2]
        q2=q[...,1::2]
        k1=k[...,::2]
        k2=k[...,1::2]
        q_rot=torch.stack((q1*cos-q2*sin,q1*sin+q2*cos),dim=-1).flatten(-2)
        k_rot=torch.stack((k1*cos-k2*sin,k1*sin+k2*cos),dim=-1).flatten(-2)
        return q_rot,k_rot

In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self,d_model,n_heads,dropout=0.2):
        super().__init__()
        self.n_heads=n_heads
        self.head_dim=d_model//n_heads
        self.qkv=nn.Linear(d_model,3*d_model)
        self.out_proj=nn.Linear(d_model,d_model)
        self.rope=RotaryEmbedding(self.head_dim)
        self.dropout=dropout
        
    def forward(self,x):
        B,T,C=x.shape
        qkv=self.qkv(x)
        q,k,v=qkv.chunk(3,dim=-1)
        q=q.view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        k=k.view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        v=v.view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        q,k=self.rope(q,k)
        out=F.scaled_dot_product_attention(q,k,v,dropout_p=self.dropout if self.training else 0.0,is_causal=True)

        out=out.transpose(1,2).contiguous()
        out=out.view(B,T,C)

        return self.out_proj(out)

In [4]:
class FeedForward(nn.Module):
    def __init__(self,d_model,ffn_dim,dropout=0.2):
        super().__init__()
        self.fc1=nn.Linear(d_model,ffn_dim)
        self.fc2=nn.Linear(ffn_dim,d_model)
        self.dropout=nn.Dropout(dropout)

    def forward(self,x):
        x=self.fc1(x)
        x=F.gelu(x)
        x=self.fc2(x)
        return self.dropout(x)

In [5]:
class TransformerBlock(nn.Module):
    def __init__(self,d_model,n_heads,ffn_dim,dropout=0.2):
        super().__init__()
        self.norm1=RMSNorm(d_model)
        self.attn=CausalSelfAttention(d_model,n_heads,dropout)
        self.norm2=RMSNorm(d_model)
        self.ffn=FeedForward(d_model,ffn_dim,dropout)

    def forward(self,x):
        x=x+self.attn(self.norm1(x))
        x=x+self.ffn(self.norm2(x))
        return x

In [6]:
class GPTModel(nn.Module):
    def __init__(self,vocab_size,d_model,n_layers,n_heads,ffn_dim,dropout=0.2):
        super().__init__()
        self.token_embedding=nn.Embedding(vocab_size,d_model)
        nn.init.normal_(self.token_embedding.weight,mean=0.0,std=0.02)
        self.blocks=nn.ModuleList([TransformerBlock(d_model,n_heads,ffn_dim,dropout) for _ in range(n_layers)])
        self.norm=RMSNorm(d_model)
        self.lm_head=nn.Linear(d_model,vocab_size,bias=False)
        self.lm_head.weight=self.token_embedding.weight
        self.dropout=nn.Dropout(dropout)
        
    def forward(self,x):
        x=self.token_embedding(x)
        x=self.dropout(x)
        for block in self.blocks:
            x=block(x)
        x=self.norm(x)
        return self.lm_head(x)

In [7]:
model=GPTModel(VOCAB_SIZE,D_MODEL,N_LAYERS,N_HEADS,FFN_DIM,DROPOUT).cuda()
num_params=sum(p.numel() for p in model.parameters())
print("Parameters:",num_params)
print("Parameters (M):",num_params/1e6)

Parameters: 109612800
Parameters (M): 109.6128


In [ ]:
BATCH_SIZE=5
GRAD_ACCUM_STEPS=1
optimizer=create_optimizer(model)
scaler=GradScaler("cuda")

In [6]:
%%writefile /kaggle/working/train_m1.py
import os,gc,json,math,time
import torch
import torch.nn.functional as F
import torch.distributed as dist
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.amp import autocast,GradScaler
from tqdm import tqdm

from legal_m1 import GPTModel
from train_data import TokenBlockDataset

# =========================
# CONFIG
# =========================
DATA_DIR="/kaggle/working"
CKPT_DIR="/kaggle/working/m1_checkpoints"
os.makedirs(CKPT_DIR,exist_ok=True)

VOCAB_SIZE=32000
BLOCK_SIZE=2048
BATCH_SIZE=5
NUM_WORKERS=2

D_MODEL=768
N_LAYERS=12
N_HEADS=12
FFN_DIM=3072
DROPOUT=0.2

TARGET_TOKENS=500_000_000
LR=3e-4
MIN_LR=3e-5
WEIGHT_DECAY=0.1
GRAD_CLIP=1.0
WARMUP_FRAC=0.03

VAL_EVERY=2500
VAL_BATCHES=100
SAVE_EVERY=2500

TRAIN_BIN=f"{DATA_DIR}/train_tokens.bin"
VALID_BIN=f"{DATA_DIR}/valid_tokens.bin"

# =========================
# DDP
# =========================
def setup():
    dist.init_process_group("nccl")
    rank=int(os.environ["RANK"])
    local_rank=int(os.environ["LOCAL_RANK"])
    world_size=int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(local_rank)
    return rank,local_rank,world_size

def cleanup():
    dist.destroy_process_group()

# =========================
# LR SCHEDULE
# =========================
def get_lr(step,total_steps,warmup_steps):
    if step<warmup_steps:
        return LR*(step+1)/warmup_steps
    progress=(step-warmup_steps)/max(1,total_steps-warmup_steps)
    progress=min(1.0,progress)
    return MIN_LR+0.5*(LR-MIN_LR)*(1+math.cos(math.pi*progress))

# =========================
# VALIDATION
# =========================
@torch.no_grad()
def validate(model,loader,device,val_batches):
    model.eval()
    total_loss=torch.tensor(0.0,device=device)
    total_count=torch.tensor(0.0,device=device)

    for i,(x,y) in enumerate(loader):
        if i>=val_batches:
            break

        x=x.to(device,non_blocking=True)
        y=y.to(device,non_blocking=True)

        with autocast(device_type="cuda",dtype=torch.float16):
            logits=model(x)
            loss=F.cross_entropy(logits.reshape(-1,VOCAB_SIZE),y.reshape(-1))

        n=y.numel()
        total_loss+=loss.detach()*n
        total_count+=n

    dist.all_reduce(total_loss,op=dist.ReduceOp.SUM)
    dist.all_reduce(total_count,op=dist.ReduceOp.SUM)

    model.train()
    return (total_loss/total_count).item()

# =========================
# CHECKPOINT
# =========================
def get_raw_model(model):
    m=model.module
    if hasattr(m,"_orig_mod"):
        m=m._orig_mod
    return m

def save_checkpoint(model,optimizer,scaler,step,tokens_seen,best_val_loss,path):
    raw_model=get_raw_model(model)

    state={
        "model":raw_model.state_dict(),
        "optimizer":optimizer.state_dict(),
        "scaler":scaler.state_dict(),
        "step":step,
        "tokens_seen":tokens_seen,
        "best_val_loss":best_val_loss,
        "config":{
            "vocab_size":VOCAB_SIZE,
            "block_size":BLOCK_SIZE,
            "d_model":D_MODEL,
            "n_layers":N_LAYERS,
            "n_heads":N_HEADS,
            "ffn_dim":FFN_DIM,
            "dropout":DROPOUT
        }
    }

    torch.save(state,path)

# =========================
# MAIN
# =========================
def main():
    rank,local_rank,world_size=setup()
    device=torch.device(f"cuda:{local_rank}")

    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32=True

    tokens_per_step=BATCH_SIZE*world_size*(BLOCK_SIZE-1)
    total_steps=math.ceil(TARGET_TOKENS/tokens_per_step)
    warmup_steps=math.ceil(total_steps*WARMUP_FRAC)

    # -------------------------
    # Dataset
    # -------------------------
    train_ds=TokenBlockDataset(TRAIN_BIN,BLOCK_SIZE)
    valid_ds=TokenBlockDataset(VALID_BIN,BLOCK_SIZE)

    train_sampler=DistributedSampler(
        train_ds,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        drop_last=True
    )

    valid_sampler=DistributedSampler(
        valid_ds,
        num_replicas=world_size,
        rank=rank,
        shuffle=False,
        drop_last=True
    )

    train_loader=DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )

    valid_loader=DataLoader(
        valid_ds,
        batch_size=BATCH_SIZE,
        sampler=valid_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )

    # -------------------------
    # Model
    # -------------------------
    model=GPTModel(
        VOCAB_SIZE,
        D_MODEL,
        N_LAYERS,
        N_HEADS,
        FFN_DIM,
        DROPOUT
    ).to(device)

    params=sum(p.numel() for p in model.parameters())

    if rank==0:
        print("="*60)
        print("M1 LEGAL LM PRETRAINING")
        print("="*60)
        print(f"Parameters: {params/1e6:.4f} M")
        print(f"GPUs: {world_size}")
        print(f"Batch/GPU: {BATCH_SIZE}")
        print(f"Global batch: {BATCH_SIZE*world_size}")
        print(f"Tokens/step: {tokens_per_step}")
        print(f"Target tokens: {TARGET_TOKENS:,}")
        print(f"Total steps: {total_steps:,}")
        print(f"Warmup steps: {warmup_steps:,}")
        print("="*60)
        print("Starting torch.compile. The first step may take a few minutes...")
        print()

    model=torch.compile(model)

    model=DDP(
        model,
        device_ids=[local_rank],
        output_device=local_rank
    )

    optimizer=torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        fused=True
    )

    scaler=GradScaler("cuda")

    # -------------------------
    # Resume
    # -------------------------
    latest_path=f"{CKPT_DIR}/latest.pt"
    step=0
    tokens_seen=0
    best_val_loss=float("inf")

    if os.path.exists(latest_path):
        checkpoint=torch.load(
            latest_path,
            map_location=device,
            weights_only=False
        )

        get_raw_model(model).load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scaler.load_state_dict(checkpoint["scaler"])

        step=checkpoint["step"]
        tokens_seen=checkpoint["tokens_seen"]
        best_val_loss=checkpoint.get("best_val_loss",float("inf"))

        if rank==0:
            print(f"Resumed from step {step:,}")
            print(f"Tokens already seen: {tokens_seen:,}")
            print()

    # -------------------------
    # Progress bar
    # -------------------------
    pbar=None

    if rank==0:
        pbar=tqdm(
            total=total_steps,
            initial=step,
            desc="M1 training",
            unit="step",
            dynamic_ncols=True,
            smoothing=0.05
        )

    # -------------------------
    # Resume data position
    # -------------------------
    steps_per_epoch=len(train_loader)
    start_epoch=step//steps_per_epoch
    skip_batches=step%steps_per_epoch

    current_epoch=start_epoch

    train_sampler.set_epoch(current_epoch)
    train_iter=iter(train_loader)

    if skip_batches>0:
        if rank==0:
            print(f"Skipping {skip_batches:,} already-trained batches...")

        for _ in range(skip_batches):
            next(train_iter)

    # -------------------------
    # Training
    # -------------------------
    start_time=time.time()
    last_log_time=start_time
    last_log_tokens=tokens_seen

    model.train()

    while step<total_steps:

        try:
            x,y=next(train_iter)

        except StopIteration:
            current_epoch+=1
            train_sampler.set_epoch(current_epoch)
            train_iter=iter(train_loader)
            x,y=next(train_iter)

        x=x.to(device,non_blocking=True)
        y=y.to(device,non_blocking=True)

        lr=get_lr(step,total_steps,warmup_steps)

        for group in optimizer.param_groups:
            group["lr"]=lr

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda",dtype=torch.float16):
            logits=model(x)
            loss=F.cross_entropy(
                logits.reshape(-1,VOCAB_SIZE),
                y.reshape(-1)
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP
        )

        scaler.step(optimizer)
        scaler.update()

        step+=1
        tokens_seen+=tokens_per_step

        # -------------------------
        # Progress bar
        # -------------------------
        if rank==0:
            pbar.update(1)

            if step%25==0 or step==1:
                elapsed=time.time()-start_time
                tok_per_sec=tokens_seen/max(elapsed,1e-6)
                remaining=max(0,TARGET_TOKENS-tokens_seen)
                eta=remaining/max(tok_per_sec,1e-6)

                pbar.set_postfix(
                    loss=f"{loss.item():.3f}",
                    lr=f"{lr:.2e}",
                    tok=f"{tokens_seen/1e6:.1f}M",
                    speed=f"{tok_per_sec/1000:.1f}k/s",
                    eta=time.strftime("%H:%M:%S",time.gmtime(eta))
                )

        # -------------------------
        # Validation
        # -------------------------
        if step%VAL_EVERY==0 or step==total_steps:

            dist.barrier()

            val_loss=validate(
                model,
                valid_loader,
                device,
                VAL_BATCHES
            )

            if rank==0:
                pbar.write(
                    f"[VALID] step {step:,}/{total_steps:,} | "
                    f"tokens {tokens_seen/1e6:.1f}M | "
                    f"val_loss {val_loss:.4f}"
                )

                if val_loss<best_val_loss:
                    best_val_loss=val_loss

                    save_checkpoint(
                        model,
                        optimizer,
                        scaler,
                        step,
                        tokens_seen,
                        best_val_loss,
                        f"{CKPT_DIR}/best.pt"
                    )

                    pbar.write(
                        f"[BEST] New best validation loss: {best_val_loss:.4f}"
                    )

        # -------------------------
        # Checkpoint
        # -------------------------
        if step%SAVE_EVERY==0 or step==total_steps:

            dist.barrier()

            if rank==0:
                save_checkpoint(
                    model,
                    optimizer,
                    scaler,
                    step,
                    tokens_seen,
                    best_val_loss,
                    latest_path
                )

                pbar.write(
                    f"[SAVE] latest checkpoint saved at step {step:,}"
                )

    # -------------------------
    # Final checkpoint
    # -------------------------
    dist.barrier()

    if rank==0:
        save_checkpoint(
            model,
            optimizer,
            scaler,
            step,
            tokens_seen,
            best_val_loss,
            f"{CKPT_DIR}/final.pt"
        )

        elapsed=time.time()-start_time

        pbar.close()

        print()
        print("="*60)
        print("TRAINING COMPLETE")
        print("="*60)
        print(f"Steps: {step:,}")
        print(f"Tokens: {tokens_seen:,}")
        print(f"Time: {elapsed/3600:.2f} hours")
        print(f"Average speed: {tokens_seen/max(elapsed,1e-6)/1000:.2f}k tokens/sec")
        print(f"Best val loss: {best_val_loss:.4f}")
        print(f"Checkpoint: {CKPT_DIR}/final.pt")
        print("="*60)

    cleanup()

if __name__=="__main__":
    main()

Overwriting /kaggle/working/train_m1.py


In [146]:
train_loader=DataLoader(
    train_dataset,
    batch_size=6,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

x,y=next(iter(train_loader))

print("Batch x:",x.shape,x.dtype)
print("Batch y:",y.shape,y.dtype)
print("GPU:",torch.cuda.get_device_name(0))

Batch x: torch.Size([6, 2047]) torch.int64
Batch y: torch.Size([6, 2047]) torch.int64
GPU: Tesla T4


In [11]:
from train_data import TokenBlockDataset

train_ds=TokenBlockDataset(
    "/kaggle/working/legal_tokens/train_tokens.bin",
    2048
)

valid_ds=TokenBlockDataset(
    "/kaggle/working/legal_tokens/valid_tokens.bin",
    2048
)

print("Train blocks:",len(train_ds))
print("Valid blocks:",len(valid_ds))

x,y=train_ds[0]
print("x:",x.shape,x.dtype)
print("y:",y.shape,y.dtype)

Train blocks: 2799571
Valid blocks: 312739
x: torch.Size([2047]) torch.int64
y: torch.Size([2047]) torch.int64


In [12]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_m1.py

W1005 06:17:58.758000 120 torch/distributed/run.py:852] 
W1005 06:17:58.758000 120 torch/distributed/run.py:852] *****************************************
W1005 06:17:58.758000 120 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1005 06:17:58.758000 120 torch/distributed/run.py:852] *****************************************
[W1005 06:17:59.375988311 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1005 06:18:01.204691531 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1005 06:18:01.456990092 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
M1 LEGAL LM PRETRAINING
Parameters: 109.6128 M
GPUs: 2
Batch/GPU: 5
Global batch: 10
Tokens/step: 20470
Target tokens: 500,000,000
Total steps: 24,4

In [54]:
import sys
import torch
import pickle

sys.path.append("/kaggle/working")
from legal_m1 import GPTModel

DEVICE="cuda" if torch.cuda.is_available() else "cpu"

with open("/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl","rb") as f:
    bpe=pickle.load(f)

CHECKPOINT="/kaggle/working/m1_checkpoints/best.pt"
ckpt=torch.load(CHECKPOINT,map_location=DEVICE)

config=ckpt["config"]

model=GPTModel(
    vocab_size=config["vocab_size"],
    d_model=config["d_model"],
    n_layers=config["n_layers"],
    n_heads=config["n_heads"],
    ffn_dim=config["ffn_dim"],
    dropout=config["dropout"]
).to(DEVICE)

model.load_state_dict(ckpt["model"])
model.eval()

print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
print("Device:",DEVICE)

Parameters: 109.61M
Device: cuda


In [55]:
def decode_tokens(tokens):
    data=bytearray()

    for token in tokens:
        token=int(token)

        if token==bpe.eos_id:
            break

        data.extend(bpe._token_bytes[token-1])

    return data.decode("utf-8",errors="ignore")

In [56]:
from huggingface_hub import hf_hub_download
import json

test_file_path=hf_hub_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    filename="Phase_1_Pretraining/Test/test.json",
    repo_type="dataset"
)

with open(test_file_path,"r",encoding="utf-8") as f:
    test_data=json.load(f)

print("Number of test examples:",len(test_data))

Number of test examples: 1000


In [57]:
with open("/kaggle/working/legal_m1.py","a") as f:
    f.write(r'''

# ============================================================
# KV CACHE INFERENCE
# ============================================================

def _rope_cached(self,q,k,start_pos=0):
    seq_len=q.size(-2)

    cos=self.cos[:,:,start_pos:start_pos+seq_len,:]
    sin=self.sin[:,:,start_pos:start_pos+seq_len,:]

    q1=q[...,::2]
    q2=q[...,1::2]
    k1=k[...,::2]
    k2=k[...,1::2]

    q_rot=torch.stack(
        (q1*cos-q2*sin,q1*sin+q2*cos),
        dim=-1
    ).flatten(-2)

    k_rot=torch.stack(
        (k1*cos-k2*sin,k1*sin+k2*cos),
        dim=-1
    ).flatten(-2)

    return q_rot,k_rot


def _attn_forward_cached(self,x,past_kv=None,start_pos=0):
    B,T,C=x.shape

    qkv=self.qkv(x)
    q,k,v=qkv.chunk(3,dim=-1)

    q=q.view(B,T,self.n_heads,self.head_dim).transpose(1,2)
    k=k.view(B,T,self.n_heads,self.head_dim).transpose(1,2)
    v=v.view(B,T,self.n_heads,self.head_dim).transpose(1,2)

    q,k=_rope_cached(
        self.rope,
        q,
        k,
        start_pos
    )

    if past_kv is not None:
        past_k,past_v=past_kv
        k=torch.cat((past_k,k),dim=2)
        v=torch.cat((past_v,v),dim=2)

        causal=False
    else:
        causal=True

    out=F.scaled_dot_product_attention(
        q,
        k,
        v,
        dropout_p=0.0,
        is_causal=causal
    )

    new_kv=(k,v)

    out=out.transpose(1,2).contiguous()
    out=out.view(B,T,C)

    return self.out_proj(out),new_kv


def _block_forward_cached(self,x,past_kv=None,start_pos=0):
    attn_out,new_kv=_attn_forward_cached(
        self.attn,
        self.norm1(x),
        past_kv,
        start_pos
    )

    x=x+attn_out
    x=x+self.ffn(self.norm2(x))

    return x,new_kv


@torch.no_grad()
def _gpt_forward_cached(self,x,past_kvs=None,start_pos=0):
    x=self.token_embedding(x)

    if past_kvs is None:
        past_kvs=[None]*len(self.blocks)

    new_kvs=[]

    for i,block in enumerate(self.blocks):
        x,new_kv=_block_forward_cached(
            block,
            x,
            past_kvs[i],
            start_pos
        )

        new_kvs.append(new_kv)

    x=self.norm(x)
    logits=self.lm_head(x)

    return logits,new_kvs


CausalSelfAttention.forward_cached=_attn_forward_cached
TransformerBlock.forward_cached=_block_forward_cached
GPTModel.forward_cached=_gpt_forward_cached
''')

print("KV-cache code added.")

KV-cache code added.


In [69]:
model=GPTModel(
    VOCAB_SIZE,
    D_MODEL,
    N_LAYERS,
    N_HEADS,
    FFN_DIM,
    DROPOUT
).to(DEVICE)

checkpoint=torch.load(
    "/kaggle/working/m1_checkpoints/best.pt",
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model"])
model.eval()

print("Model loaded")
print("Has KV cache:",hasattr(model,"forward_cached"))

Model loaded
Has KV cache: True


In [70]:
model=GPTModel(
    VOCAB_SIZE,
    D_MODEL,
    N_LAYERS,
    N_HEADS,
    FFN_DIM,
    DROPOUT
).to(DEVICE)

checkpoint=torch.load(
    "/kaggle/working/m1_750_checkpoints/latest.pt",
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model"])
model.eval()

print("Model loaded")
print("Has KV cache:",hasattr(model,"forward_cached"))

Model loaded
Has KV cache: True


In [71]:
with open("/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl","rb") as f:
    bpe=pickle.load(f)

def decode_tokens(tokens):
    data=bytearray()

    for token in tokens:
        token=int(token)

        if token==bpe.eos_id:
            break

        data.extend(bpe._token_bytes[token-1])

    return data.decode("utf-8",errors="ignore")

In [61]:
def generate_128_words(prompt):
    MAX_GEN_TOKENS=450
    TEMPERATURE=0.7
    TOP_P=0.95
    REP_PENALTY=1.15
    REP_WINDOW=50

    prompt_tokens=bpe.encode_text(
        prompt,
        add_eos=False
    )

    prompt_tokens=prompt_tokens[-(2048-MAX_GEN_TOKENS):]

    x=torch.tensor(
        prompt_tokens,
        dtype=torch.long,
        device=DEVICE
    ).unsqueeze(0)

    generated=[]

    with torch.inference_mode():

        logits,past_kvs=model.forward_cached(
            x,
            None,
            0
        )

        for _ in range(MAX_GEN_TOKENS):

            logits=logits[:,-1,:]

            for token_id in set(generated[-REP_WINDOW:]):
                logits[0,token_id]/=REP_PENALTY

            logits=logits/TEMPERATURE

            probs=torch.softmax(logits,dim=-1)

            sorted_probs,sorted_indices=torch.sort(
                probs,
                dim=-1,
                descending=True
            )

            cumulative=torch.cumsum(
                sorted_probs,
                dim=-1
            )

            remove=cumulative>TOP_P
            remove[...,1:]=remove[...,:-1].clone()
            remove[...,0]=False

            sorted_probs[remove]=0

            sorted_probs=sorted_probs/sorted_probs.sum(
                dim=-1,
                keepdim=True
            )

            sample_idx=torch.multinomial(
                sorted_probs,
                1
            )

            next_token=sorted_indices.gather(
                1,
                sample_idx
            )

            token_id=int(next_token.item())

            if token_id!=bpe.eos_id:
                generated.append(token_id)

            current_pos=len(prompt_tokens)+len(generated)-1

            logits,past_kvs=model.forward_cached(
                next_token,
                past_kvs,
                current_pos
            )

    words=decode_tokens(generated).split()

    return " ".join(words[:128])

In [88]:
predictions={}

items=list(test_data.items())
total_examples=len(items)

start=time.time()

for i,(example_id,record) in enumerate(items):
    prediction=generate_128_words(record["text"])
    predictions[str(example_id)]={"text":prediction}

    if (i+1)%10==0 or i==0:
        elapsed=time.time()-start
        speed=(i+1)/max(elapsed,1e-6)
        eta=(total_examples-i-1)/max(speed,1e-6)

        print(
            f"{i+1}/{total_examples} "
            f"({100*(i+1)/total_examples:.1f}%) "
            f"| {speed:.2f} ex/s "
            f"| ETA {eta/60:.1f} min"
        )

1/1000 (0.1%) | 0.20 ex/s | ETA 83.5 min
10/1000 (1.0%) | 0.20 ex/s | ETA 82.4 min
20/1000 (2.0%) | 0.20 ex/s | ETA 81.5 min
30/1000 (3.0%) | 0.20 ex/s | ETA 80.7 min
40/1000 (4.0%) | 0.20 ex/s | ETA 79.9 min
50/1000 (5.0%) | 0.20 ex/s | ETA 79.2 min
60/1000 (6.0%) | 0.20 ex/s | ETA 78.4 min
70/1000 (7.0%) | 0.20 ex/s | ETA 77.6 min
80/1000 (8.0%) | 0.20 ex/s | ETA 76.7 min
90/1000 (9.0%) | 0.20 ex/s | ETA 75.9 min
100/1000 (10.0%) | 0.20 ex/s | ETA 75.1 min
110/1000 (11.0%) | 0.20 ex/s | ETA 74.2 min
120/1000 (12.0%) | 0.20 ex/s | ETA 73.4 min
130/1000 (13.0%) | 0.20 ex/s | ETA 72.6 min
140/1000 (14.0%) | 0.20 ex/s | ETA 71.7 min
150/1000 (15.0%) | 0.20 ex/s | ETA 70.9 min
160/1000 (16.0%) | 0.20 ex/s | ETA 70.0 min
170/1000 (17.0%) | 0.20 ex/s | ETA 69.2 min
180/1000 (18.0%) | 0.20 ex/s | ETA 68.4 min
190/1000 (19.0%) | 0.20 ex/s | ETA 67.6 min
200/1000 (20.0%) | 0.20 ex/s | ETA 66.8 min
210/1000 (21.0%) | 0.20 ex/s | ETA 65.9 min
220/1000 (22.0%) | 0.20 ex/s | ETA 65.1 min
230/1000 

In [89]:
word_counts=[len(x["text"].split()) for x in predictions.values()]

print("Total:",len(word_counts))
print("Min:",min(word_counts))
print("Max:",max(word_counts))
print("Exactly 128:",sum(x==128 for x in word_counts))
print("Not 128:",sum(x!=128 for x in word_counts))

Total: 1000
Min: 55
Max: 128
Exactly 128: 999
Not 128: 1


In [49]:
def generate_128_words_retry(prompt):
    MAX_GEN_TOKENS=600
    TEMPERATURE=0.7
    TOP_P=0.95
    REP_PENALTY=1.15
    REP_WINDOW=50

    prompt_tokens=bpe.encode_text(prompt,add_eos=False)
    prompt_tokens=prompt_tokens[-(2048-MAX_GEN_TOKENS):]

    x=torch.tensor(
        prompt_tokens,
        dtype=torch.long,
        device=DEVICE
    ).unsqueeze(0)

    generated=[]

    with torch.inference_mode():
        logits,past_kvs=model.forward_cached(x,None,0)

        for _ in range(MAX_GEN_TOKENS):
            logits=logits[:,-1,:]

            for token_id in set(generated[-REP_WINDOW:]):
                logits[0,token_id]/=REP_PENALTY

            logits=logits/TEMPERATURE

            probs=torch.softmax(logits,dim=-1)

            sorted_probs,sorted_indices=torch.sort(
                probs,dim=-1,descending=True
            )

            cumulative=torch.cumsum(sorted_probs,dim=-1)

            remove=cumulative>TOP_P
            remove[...,1:]=remove[...,:-1].clone()
            remove[...,0]=False

            sorted_probs[remove]=0
            sorted_probs=sorted_probs/sorted_probs.sum(
                dim=-1,keepdim=True
            )

            sample_idx=torch.multinomial(sorted_probs,1)
            next_token=sorted_indices.gather(1,sample_idx)

            token_id=int(next_token.item())

            # Ignore EOS so we can reach 128 words
            if token_id!=bpe.eos_id:
                generated.append(token_id)

            current_pos=len(prompt_tokens)+len(generated)-1

            logits,past_kvs=model.forward_cached(
                next_token,
                past_kvs,
                current_pos
            )

            words=decode_tokens(generated).split()

            if len(words)>=128:
                return " ".join(words[:128])

    words=decode_tokens(generated).split()
    return " ".join(words[:128])

In [95]:
word_counts=[len(x["text"].split()) for x in predictions.values()]

print("Total:",len(word_counts))
print("Min:",min(word_counts))
print("Max:",max(word_counts))
print("Exactly 128:",sum(x==128 for x in word_counts))
print("Not 128:",sum(x!=128 for x in word_counts))

Total: 1000
Min: 128
Max: 128
Exactly 128: 1000
Not 128: 0


In [96]:
import json

PREDICTION_PATH="/kaggle/working/prediction.json"

with open(PREDICTION_PATH,"w",encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:",PREDICTION_PATH)

Saved: /kaggle/working/prediction.json


In [97]:
import zipfile

PREDICTION_PATH="/kaggle/working/prediction.json"
with open(PREDICTION_PATH,"w",encoding="utf-8") as f:
    json.dump(predictions,f,ensure_ascii=False,indent=2)

print("Saved:",PREDICTION_PATH)

SUBMISSION_ZIP="/kaggle/working/submission.zip"
with zipfile.ZipFile(SUBMISSION_ZIP,"w",zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH,arcname="prediction.json")

with zipfile.ZipFile(SUBMISSION_ZIP,"r") as z:
    print("Contents:",z.namelist())

print("Upload this:",SUBMISSION_ZIP)

Saved: /kaggle/working/prediction.json
Contents: ['prediction.json']
Upload this: /kaggle/working/submission.zip


In [13]:
%%writefile /kaggle/working/train_m1_750.py

import os,math,time
import torch
import torch.nn.functional as F
import torch.distributed as dist
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.amp import autocast,GradScaler
from tqdm import tqdm

from legal_m1 import GPTModel
from train_data import TokenBlockDataset

# =========================
# CONFIG
# =========================
DATA_DIR="/kaggle/working/legal_tokens"

# Original 500M checkpoint
RESUME_PATH="/kaggle/working/m1_checkpoints/best.pt"

# New checkpoints
CKPT_DIR="/kaggle/working/m1_750_checkpoints"
os.makedirs(CKPT_DIR,exist_ok=True)

VOCAB_SIZE=32000
BLOCK_SIZE=2048
BATCH_SIZE=5
NUM_WORKERS=2

D_MODEL=768
N_LAYERS=12
N_HEADS=12
FFN_DIM=3072
DROPOUT=0.2

START_TOKENS=500_000_220
TARGET_TOKENS=750_000_000
CONTINUE_TOKENS=TARGET_TOKENS-START_TOKENS

START_LR=3e-5
MIN_LR=3e-6

WEIGHT_DECAY=0.1
GRAD_CLIP=1.0

VAL_EVERY=2500
VAL_BATCHES=100
SAVE_EVERY=2500

# First run this as True.
SMOKE_TEST=False
SMOKE_STEPS=2

TRAIN_BIN=f"{DATA_DIR}/train_tokens.bin"
VALID_BIN=f"{DATA_DIR}/valid_tokens.bin"

# =========================
# DDP
# =========================
def setup():
    dist.init_process_group("nccl")
    rank=int(os.environ["RANK"])
    local_rank=int(os.environ["LOCAL_RANK"])
    world_size=int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(local_rank)
    return rank,local_rank,world_size

def cleanup():
    dist.destroy_process_group()

# =========================
# LR SCHEDULE
# =========================
def get_lr(tokens_seen):
    progress=(tokens_seen-START_TOKENS)/CONTINUE_TOKENS
    progress=max(0.0,min(1.0,progress))

    return MIN_LR+0.5*(START_LR-MIN_LR)*(1+math.cos(math.pi*progress))

# =========================
# VALIDATION
# =========================
@torch.no_grad()
def validate(model,loader,device,val_batches):
    model.eval()

    total_loss=torch.tensor(0.0,device=device)
    total_count=torch.tensor(0.0,device=device)

    for i,(x,y) in enumerate(loader):
        if i>=val_batches:
            break

        x=x.to(device,non_blocking=True)
        y=y.to(device,non_blocking=True)

        with autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits=model(x)
            loss=F.cross_entropy(
                logits.reshape(-1,VOCAB_SIZE),
                y.reshape(-1)
            )

        n=y.numel()

        total_loss+=loss.detach()*n
        total_count+=n

    dist.all_reduce(
        total_loss,
        op=dist.ReduceOp.SUM
    )

    dist.all_reduce(
        total_count,
        op=dist.ReduceOp.SUM
    )

    model.train()

    return (total_loss/total_count).item()

# =========================
# CHECKPOINT
# =========================
def get_raw_model(model):
    m=model.module

    if hasattr(m,"_orig_mod"):
        m=m._orig_mod

    return m

def save_checkpoint(
    model,
    optimizer,
    scaler,
    step,
    tokens_seen,
    best_val_loss,
    path
):
    raw_model=get_raw_model(model)

    state={
        "model":raw_model.state_dict(),
        "optimizer":optimizer.state_dict(),
        "scaler":scaler.state_dict(),
        "step":step,
        "tokens_seen":tokens_seen,
        "best_val_loss":best_val_loss,
        "config":{
            "vocab_size":VOCAB_SIZE,
            "block_size":BLOCK_SIZE,
            "d_model":D_MODEL,
            "n_layers":N_LAYERS,
            "n_heads":N_HEADS,
            "ffn_dim":FFN_DIM,
            "dropout":DROPOUT,
            "start_tokens":START_TOKENS,
            "target_tokens":TARGET_TOKENS,
            "start_lr":START_LR,
            "min_lr":MIN_LR
        }
    }

    torch.save(state,path)

# =========================
# MAIN
# =========================
def main():

    rank,local_rank,world_size=setup()

    device=torch.device(
        f"cuda:{local_rank}"
    )

    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32=True

    tokens_per_step=(
        BATCH_SIZE*
        world_size*
        (BLOCK_SIZE-1)
    )

    continue_steps=math.ceil(
        CONTINUE_TOKENS/tokens_per_step
    )

    if SMOKE_TEST:
        run_steps=SMOKE_STEPS
    else:
        run_steps=continue_steps

    if rank==0:
        print("="*60)
        print("M1 CONTINUATION: 500M -> 750M")
        print("="*60)
        print(f"GPUs: {world_size}")
        print(f"Batch/GPU: {BATCH_SIZE}")
        print(f"Global batch: {BATCH_SIZE*world_size}")
        print(f"Tokens/step: {tokens_per_step}")
        print(f"Starting tokens: {START_TOKENS:,}")
        print(f"Target tokens: {TARGET_TOKENS:,}")
        print(f"Continuation tokens: {CONTINUE_TOKENS:,}")
        print(f"Continuation steps: {continue_steps:,}")
        print(f"Start LR: {START_LR:.2e}")
        print(f"Min LR: {MIN_LR:.2e}")
        print(f"Smoke test: {SMOKE_TEST}")
        print("="*60)

    # =========================
    # Dataset
    # =========================
    train_ds=TokenBlockDataset(
        TRAIN_BIN,
        BLOCK_SIZE
    )

    valid_ds=TokenBlockDataset(
        VALID_BIN,
        BLOCK_SIZE
    )

    train_sampler=DistributedSampler(
        train_ds,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        drop_last=True
    )

    valid_sampler=DistributedSampler(
        valid_ds,
        num_replicas=world_size,
        rank=rank,
        shuffle=False,
        drop_last=True
    )

    train_loader=DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )

    valid_loader=DataLoader(
        valid_ds,
        batch_size=BATCH_SIZE,
        sampler=valid_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )

    # =========================
    # Model
    # =========================
    model=GPTModel(
        VOCAB_SIZE,
        D_MODEL,
        N_LAYERS,
        N_HEADS,
        FFN_DIM,
        DROPOUT
    ).to(device)

    params=sum(
        p.numel()
        for p in model.parameters()
    )

    if rank==0:
        print(f"Parameters: {params/1e6:.4f} M")
        print("Loading 500M checkpoint...")

    model=torch.compile(model)

    model=DDP(
        model,
        device_ids=[local_rank],
        output_device=local_rank
    )

    optimizer=torch.optim.AdamW(
        model.parameters(),
        lr=START_LR,
        weight_decay=WEIGHT_DECAY,
        fused=True
    )

    scaler=GradScaler("cuda")

    # =========================
    # Resume
    # =========================
    checkpoint=torch.load(
        RESUME_PATH,
        map_location=device,
        weights_only=False
    )

    get_raw_model(model).load_state_dict(
        checkpoint["model"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer"]
    )

    scaler.load_state_dict(
        checkpoint["scaler"]
    )

    step=checkpoint["step"]
    tokens_seen=checkpoint["tokens_seen"]

    best_val_loss=checkpoint.get(
        "best_val_loss",
        float("inf")
    )

    if rank==0:
        print("="*60)
        print("CHECKPOINT LOADED")
        print(f"Checkpoint: {RESUME_PATH}")
        print(f"Step: {step:,}")
        print(f"Tokens: {tokens_seen:,}")
        print(f"Best val loss: {best_val_loss:.6f}")
        print(f"Current continuation LR: {get_lr(tokens_seen):.2e}")
        print("="*60)

    if tokens_seen!=START_TOKENS:
        if rank==0:
            print(
                f"ERROR: checkpoint has {tokens_seen:,} tokens "
                f"but expected {START_TOKENS:,}"
            )

        cleanup()
        return

    # =========================
    # Resume data position
    # =========================
    steps_per_epoch=len(train_loader)

    start_epoch=step//steps_per_epoch
    skip_batches=step%steps_per_epoch

    current_epoch=start_epoch

    train_sampler.set_epoch(
        current_epoch
    )

    train_iter=iter(train_loader)

    if rank==0:
        print(
            f"Resuming data at epoch {current_epoch:,}"
        )

    if skip_batches>0:

        if rank==0:
            print(
                f"Skipping {skip_batches:,} batches..."
            )

        for _ in range(skip_batches):
            next(train_iter)

    # =========================
    # Progress
    # =========================
    pbar=None

    if rank==0:
        pbar=tqdm(
            total=run_steps,
            desc="M1 500M->750M",
            unit="step",
            dynamic_ncols=True,
            smoothing=0.05
        )

    model.train()

    start_time=time.time()

    local_step=0

    # =========================
    # Training
    # =========================
    while local_step<run_steps:

        try:
            x,y=next(train_iter)

        except StopIteration:

            current_epoch+=1

            train_sampler.set_epoch(
                current_epoch
            )

            train_iter=iter(train_loader)

            x,y=next(train_iter)

        x=x.to(
            device,
            non_blocking=True
        )

        y=y.to(
            device,
            non_blocking=True
        )

        lr=get_lr(tokens_seen)

        for group in optimizer.param_groups:
            group["lr"]=lr

        optimizer.zero_grad(
            set_to_none=True
        )

        with autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits=model(x)

            loss=F.cross_entropy(
                logits.reshape(-1,VOCAB_SIZE),
                y.reshape(-1)
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP
        )

        scaler.step(optimizer)
        scaler.update()

        step+=1
        local_step+=1

        tokens_seen+=tokens_per_step

        if rank==0:

            pbar.update(1)

            pbar.set_postfix(
                loss=f"{loss.item():.3f}",
                lr=f"{lr:.2e}",
                tok=f"{tokens_seen/1e6:.1f}M"
            )

    # =========================
    # Final smoke checkpoint
    # =========================
    if rank==0:

        save_checkpoint(
            model,
            optimizer,
            scaler,
            step,
            tokens_seen,
            best_val_loss,
            f"{CKPT_DIR}/smoke.pt"
        )

        pbar.close()

        print()
        print("="*60)
        print("CONTINUATION TEST COMPLETE")
        print("="*60)
        print(f"Step: {step:,}")
        print(f"Tokens: {tokens_seen:,}")
        print(f"Next LR: {get_lr(tokens_seen):.2e}")
        print("="*60)

    dist.barrier()

    cleanup()


if __name__=="__main__":
    main()

Writing /kaggle/working/train_m1_750.py


In [22]:
path="/kaggle/working/train_m1_750.py"

with open(path,"r") as f:
    s=f.read()

s=s.replace("SMOKE_TEST=True","SMOKE_TEST=False")

with open(path,"w") as f:
    f.write(s)

print("SMOKE_TEST set to False")

SMOKE_TEST set to False


In [24]:
!torchrun --nproc_per_node=2 /kaggle/working/train_m1_750.py

W1006 12:36:02.582000 579 torch/distributed/run.py:852] 
W1006 12:36:02.582000 579 torch/distributed/run.py:852] *****************************************
W1006 12:36:02.582000 579 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1006 12:36:02.582000 579 torch/distributed/run.py:852] *****************************************
M1 CONTINUATION: 500M -> 750M
GPUs: 2
Batch/GPU: 5
Global batch: 10
Tokens/step: 20470
Starting tokens: 500,000,220
Target tokens: 750,000,000
Continuation tokens: 249,999,780
Continuation steps: 12,213
Start LR: 3.00e-05
Min LR: 3.00e-06
Smoke test: False
Parameters: 109.6128 M
Loading 500M checkpoint...
CHECKPOINT LOADED
Checkpoint: /kaggle/working/m1_checkpoints/best.pt
Step: 24,426
Tokens: 500,000,220
Best val loss: 1.382489
Current continuation LR: 3.00e-05
Resuming data 

In [28]:
%%writefile /kaggle/working/validate_m1_750.py
import torch,torch.distributed as dist
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.amp import autocast
from legal_m1 import GPTModel
from train_data import TokenBlockDataset

DATA_DIR="/kaggle/working/legal_tokens"
CKPT_PATH="/kaggle/working/m1_750_checkpoints/latest.pt"
VOCAB_SIZE=32000
BLOCK_SIZE=2048
BATCH_SIZE=5
NUM_WORKERS=2
D_MODEL=768
N_LAYERS=12
N_HEADS=12
FFN_DIM=3072
DROPOUT=0.2
VAL_BATCHES=100

dist.init_process_group("nccl")
rank=int(torch.distributed.get_rank())
local_rank=int(torch.distributed.get_rank())
torch.cuda.set_device(local_rank)
device=torch.device(f"cuda:{local_rank}")

valid_ds=TokenBlockDataset(f"{DATA_DIR}/valid_tokens.bin",BLOCK_SIZE)
valid_sampler=DistributedSampler(
    valid_ds,
    num_replicas=dist.get_world_size(),
    rank=rank,
    shuffle=False,
    drop_last=True
)
valid_loader=DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    sampler=valid_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

model=GPTModel(
    VOCAB_SIZE,
    D_MODEL,
    N_LAYERS,
    N_HEADS,
    FFN_DIM,
    DROPOUT
).to(device)

checkpoint=torch.load(
    CKPT_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model"])
model.eval()

total_loss=torch.tensor(0.0,device=device)
total_count=torch.tensor(0.0,device=device)

with torch.no_grad():
    for i,(x,y) in enumerate(valid_loader):
        if i>=VAL_BATCHES:
            break

        x=x.to(device,non_blocking=True)
        y=y.to(device,non_blocking=True)

        with autocast(device_type="cuda",dtype=torch.float16):
            logits=model(x)
            loss=F.cross_entropy(
                logits.reshape(-1,VOCAB_SIZE),
                y.reshape(-1)
            )

        n=y.numel()
        total_loss+=loss.detach()*n
        total_count+=n

dist.all_reduce(total_loss,op=dist.ReduceOp.SUM)
dist.all_reduce(total_count,op=dist.ReduceOp.SUM)

val_loss=(total_loss/total_count).item()

if rank==0:
    print("="*60)
    print("M1 750M FINAL VALIDATION")
    print("="*60)
    print("Checkpoint:",CKPT_PATH)
    print("Step:",checkpoint["step"])
    print("Tokens:",checkpoint["tokens_seen"])
    print("Stored best val loss:",checkpoint["best_val_loss"])
    print("Final 750M validation loss:",val_loss)
    print("PPL:",torch.exp(torch.tensor(val_loss)).item())
    print("="*60)

dist.barrier()
dist.destroy_process_group()

Writing /kaggle/working/validate_m1_750.py


In [29]:
!torchrun --nproc_per_node=2 /kaggle/working/validate_m1_750.py

W1006 15:48:21.168000 824 torch/distributed/run.py:852] 
W1006 15:48:21.168000 824 torch/distributed/run.py:852] *****************************************
W1006 15:48:21.168000 824 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1006 15:48:21.168000 824 torch/distributed/run.py:852] *****************************************
M1 750M FINAL VALIDATION
Checkpoint: /kaggle/working/m1_750_checkpoints/latest.pt
Step: 36639
Tokens: 750000330
Stored best val loss: 1.3824894428253174
Final 750M validation loss: 1.3552498817443848
PPL: 3.877729892730713
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


In [68]:
checkpoint=torch.load(
    "/kaggle/working/m1_750_checkpoints/latest.pt",
    map_location="cuda",
    weights_only=False
)

model=GPTModel(
    VOCAB_SIZE,
    D_MODEL,
    N_LAYERS,
    N_HEADS,
    FFN_DIM,
    DROPOUT
).cuda()

model.load_state_dict(checkpoint["model"])
model.eval()

print("M1 750M loaded")
print("Step:",checkpoint["step"])
print("Tokens:",checkpoint["tokens_seen"])

M1 750M loaded
Step: 36639
Tokens: 750000330


In [46]:
from huggingface_hub import hf_hub_download
import json,pickle

test_file_path=hf_hub_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    filename="Phase_1_Pretraining/Test/test.json",
    repo_type="dataset"
)

with open(test_file_path,"r",encoding="utf-8") as f:
    test_data=json.load(f)

with open("/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl","rb") as f:
    bpe=pickle.load(f)

print("Test examples:",len(test_data))
print("First prompt words:",len(test_data["1"]["text"].split()))

test.json: 0.00B [00:00, ?B/s]

Test examples: 1000
First prompt words: 512


In [8]:
def decode_tokens(tokens):
    data=bytearray()
    for token in tokens:
        token=int(token)
        if token==bpe.eos_id:
            break
        data.extend(bpe._token_bytes[token-1])
    return data.decode("utf-8",errors="ignore")

print("Decoder ready")

Decoder ready


In [48]:
prompt=test_data["1"]["text"]
prompt_tokens=bpe.encode_text(prompt)

print("Prompt words:",len(prompt.split()))
print("Prompt BPE tokens:",len(prompt_tokens))

Prompt words: 512
Prompt BPE tokens: 1075


In [62]:
prediction=generate_128_words(test_data["1"]["text"])

print("Generated words:",len(prediction.split()))
print("\n",prediction)

Generated words: 128

 not the case of the petitioner that in any event the notification was issued after the land acquisition proceedings had lapsed. It is not the case of the respondent No.1 that the petitioner has not approached the Court within reasonable time." 10. In Bhagwan Singh Vs. State of Rajasthan & Ors. AIR 1996 SC 297, this Court held as under:- "28. There are two types of cases in which the present case is one and the same. In all these cases, the land acquisition proceedings were initiated after the death of a person who had died in the year 1964. The claim that there was delay in filing the writ petition is also not supported by any material and even if it is assumed, a cause of action


In [72]:
checkpoint=torch.load(
    "/kaggle/working/m1_750_checkpoints/latest.pt",
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model"])

<All keys matched successfully>

In [74]:
saved_state=checkpoint["model"]
current_state=model.state_dict()

same=True

for name in saved_state:
    if not torch.equal(current_state[name].cpu(),saved_state[name].cpu()):
        same=False
        print("Different parameter:",name)
        breakprint("Model currently in memory:")
print("Parameters:",sum(p.numel() for p in model.parameters()))
print("750M checkpoint step:",checkpoint["step"])
print("750M checkpoint tokens:",checkpoint["tokens_seen"])

same=True
for name,param in model.state_dict().items():
    if not torch.equal(param.cpu(),checkpoint["model"][name].cpu()):
        same=False
        break

print("Current model == loaded checkpoint:",same)

print("Current model == checkpoint weights:",same)

Parameters: 109612800
750M checkpoint step: 36639
750M checkpoint tokens: 750000330
Current model == loaded checkpoint: True
Current model == checkpoint weights: True


In [75]:
from tqdm.auto import tqdm
import time

predictions={}
start=time.time()

for example_id,record in tqdm(
    test_data.items(),
    total=len(test_data),
    desc="Generating M1 750M predictions"
):
    predictions[str(example_id)]={"text":generate_128_words(record["text"])}

elapsed=time.time()-start

print("\nGeneration complete")
print("Predictions:",len(predictions))
print("Time:",round(elapsed/60,2),"minutes")
print("Average:",round(elapsed/len(predictions),2),"sec/example")

Generating M1 750M predictions:   0%|          | 0/1000 [00:00<?, ?it/s]


Generation complete
Predictions: 1000
Time: 77.52 minutes
Average: 4.65 sec/example


In [79]:
print(predictions["1"])

{'text': 'well settled that a writ petition is not maintainable when the same cause of action has arisen. In [Union of India & Ors. Vs. R. K. Sharma] AIR 2005 SC 907; and State of M.P. Vs. N.K. Sharma, AIR 2003 SC 1838 the Supreme Court held as under:- "Delay in approaching the High Courts is a significant circumstance. Delay in raising objections or filing of writ petitions causes loss to public interest. Delay has always been explained by the Court and a delay is not condoned on such grounds. There are no merits in these appeals which are accordingly dismissed with costs." 11. In [U.P. Power Corporation Ltd. & Anr. Vs. Ram Chandra Agarwal] AIR 2004 SC 3084; this Court observed as under:- "It is well settled that'}


In [80]:
bad=[]

for idx,data in predictions.items():
    wc=len(data["text"].split())
    if wc!=128:
        bad.append((idx,wc))

print("Total predictions:",len(predictions))
print("Exactly 128 words:",len(predictions)-len(bad))
print("Not 128 words:",len(bad))

if bad:
    print("\nBad predictions:")
    for idx,wc in bad[:20]:
        print(idx,wc)
else:
    print("\nAll predictions have exactly 128 words.")

Total predictions: 1000
Exactly 128 words: 1000
Not 128 words: 0

All predictions have exactly 128 words.


In [81]:
import json

PREDICTION_PATH="/kaggle/working/prediction.json"

with open(PREDICTION_PATH,"w",encoding="utf-8") as f:
    json.dump(predictions,f,ensure_ascii=False,indent=2)

print("Saved:",PREDICTION_PATH)

Saved: /kaggle/working/prediction.json


In [82]:
import zipfile

PREDICTION_PATH="/kaggle/working/prediction.json"
SUBMISSION_ZIP="/kaggle/working/submission_750.zip"

with zipfile.ZipFile(SUBMISSION_ZIP,"w",zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH,arcname="prediction.json")

with zipfile.ZipFile(SUBMISSION_ZIP,"r") as z:
    print("Contents:",z.namelist())

print("Upload this:",SUBMISSION_ZIP)

Contents: ['prediction.json']
Upload this: /kaggle/working/submission_750.zip


Phase 2

In [2]:
import os
import json
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILES = {
    "fold1_text": "Phase_2_LNER/Train/fold1-text.json",
    "fold1_gold": "Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text": "Phase_2_LNER/Train/fold2-text.json",
    "fold2_gold": "Phase_2_LNER/Train/fold2-gold.json",
    "labels": "Phase_2_LNER/Train/labels.txt"
}

TEST_FILES = {
    "fold3_text": "Phase_2_LNER/Test/fold3-text.json"
}

source_paths = {}

for source_name, file_name in {**TRAIN_FILES, **TEST_FILES}.items():
    print("Downloading:", source_name)

    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=file_name,
        repo_type="dataset",
        token=HF_TOKEN
    )

    source_paths[source_name] = path

    print("Path:", path)
    print()

print("All Phase 2 files downloaded/accessed successfully.")

Downloading: fold1_text


fold1-text.json: 0.00B [00:00, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-text.json

Downloading: fold1_gold


fold1-gold.json: 0.00B [00:00, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-gold.json

Downloading: fold2_text


fold2-text.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/labels.txt

Downloading: fold3_text


fold3-text.json: 0.00B [00:00, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Test/fold3-text.json

All Phase 2 files downloaded/accessed successfully.


In [3]:
with open(source_paths["fold1_text"], "r", encoding="utf-8") as f:
    fold1_text = json.load(f)

with open(source_paths["fold1_gold"], "r", encoding="utf-8") as f:
    fold1_gold = json.load(f)

with open(source_paths["fold2_text"], "r", encoding="utf-8") as f:
    fold2_text = json.load(f)

with open(source_paths["fold2_gold"], "r", encoding="utf-8") as f:
    fold2_gold = json.load(f)

with open(source_paths["fold3_text"], "r", encoding="utf-8") as f:
    fold3_text = json.load(f)

with open(source_paths["labels"], "r", encoding="utf-8") as f:
    labels = [line.strip() for line in f if line.strip()]

print("Fold 1 documents:", len(fold1_text))
print("Fold 2 documents:", len(fold2_text))
print("Test documents:", len(fold3_text))
print("Entity labels:", labels)

Fold 1 documents: 35
Fold 2 documents: 35
Test documents: 35
Entity labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']


In [4]:
from collections import Counter

print("Fold 1 text IDs:", list(fold1_text.keys())[:5])
print("Fold 1 gold IDs:", list(fold1_gold.keys())[:5])

# Inspect one document and its annotations
doc_id = list(fold1_text.keys())[0]

print("\nDocument ID:", doc_id)
print("\nDocument text (first 1000 characters):")
print(repr(fold1_text[doc_id][:1000]))

print("\nGold annotations:")
print(fold1_gold[doc_id][:10] if isinstance(fold1_gold[doc_id], list)
      else fold1_gold[doc_id])

# Count entities across both training folds
label_counts = Counter()
total_entities = 0
invalid_offsets = 0
missing_text_ids = 0

for texts, gold in [
    (fold1_text, fold1_gold),
    (fold2_text, fold2_gold)
]:
    for doc_id, annotations in gold.items():
        if doc_id not in texts:
            missing_text_ids += 1
            continue

        for entity in annotations:
            start, end, label = entity[:3]
            label_counts[label] += 1
            total_entities += 1

            if start < 0 or end > len(texts[doc_id]) or start >= end:
                invalid_offsets += 1
            elif " ".join(texts[doc_id][start:end].split()) == "":
                invalid_offsets += 1

print("\nTotal training entities:", total_entities)
print("Invalid offsets:", invalid_offsets)
print("Gold IDs without matching text:", missing_text_ids)

print("\nEntity counts:")
for label in labels:
    print(label, ":", label_counts[label])

Fold 1 text IDs: ['115651329', '37849282', '975074', '189525449', '736324']
Fold 1 gold IDs: ['115651329', '37849282', '975074', '189525449', '736324']

Document ID: 115651329

Document text (first 1000 characters):
'REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, whereby the High Court partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 2. Briefly stated, case of the prosecution is that on the fateful day i.e. 18.11.1994, at about 8.00 A.M. in the morning the complainant Jagdish (PW-5) along with his two sons namely Sukhbir and Mange Ram (PW-6) were busy in cutting pullas (reeds) from the dola of their field. At th

In [9]:
import os
import sys
import pickle
import numpy as np

# Paths to your Phase 1 tokenizer and implementation
TOKENIZER_PATH = "/kaggle/working/legal_bpe_32k.pkl"
TOKENIZER_CODE_PATH = "/kaggle/working/legal_bpe.py"

# Make sure the custom tokenizer module can be imported
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

# Import the original implementation before loading the pickle
import legal_bpe

with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

print("Tokenizer loaded successfully.")
print("Tokenizer class:", type(tokenizer).__name__)
print("Vocabulary size:", tokenizer.vocab_size)

# Inspect the encoding output format
sample = list(fold1_text.values())[0][:500]
encoded_sample = tokenizer.encode_text(sample)

print("\nEncoded sample type:", type(encoded_sample))
print("Encoded sample preview:", str(encoded_sample)[:300])

# Calculate token lengths using the original tokenizer
all_train_texts = list(fold1_text.values()) + list(fold2_text.values())
token_lengths = []

for text in all_train_texts:
    encoded = tokenizer.encode_text(text)
    token_lengths.append(len(encoded))

print("\nToken length statistics")
print("Documents:", len(token_lengths))
print("Minimum:", min(token_lengths))
print("Median:", int(np.median(token_lengths)))
print("95th percentile:", int(np.percentile(token_lengths, 95)))
print("Maximum:", max(token_lengths))

# Estimate chunks for an initial configuration
CHUNK_SIZE = 1536
OVERLAP = 256
STEP = CHUNK_SIZE - OVERLAP

total_chunks = 0

for length in token_lengths:
    if length <= CHUNK_SIZE:
        total_chunks += 1
    else:
        total_chunks += int(np.ceil((length - CHUNK_SIZE) / STEP)) + 1

print("\nEstimated chunks:", total_chunks)

Tokenizer loaded successfully.
Tokenizer class: ByteLevelBPE
Vocabulary size: 32000

Encoded sample type: <class 'list'>
Encoded sample preview: [22597, 33, 494, 33, 608, 33, 17302, 33, 1849, 33, 561, 33, 5510, 33, 3664, 33, 4638, 33, 7760, 33, 3664, 33, 2555, 33, 1007, 33, 5951, 4232, 33, 75, 4616, 33, 968, 33, 39, 33, 2271, 33, 453, 3101, 33, 910, 33, 1092, 33, 561, 33, 12890, 33, 453, 541, 33, 75, 33, 86, 33, 69, 33, 72, 33, 78, 33, 70, 3

Token length statistics
Documents: 70
Minimum: 3608
Median: 9428
95th percentile: 21101
Maximum: 41761

Estimated chunks: 635


In [29]:

doc_id = "1243363"
text = fold1_text.get(doc_id, fold2_text.get(doc_id))

start = 25083
end = 25099

token_ids, offsets = tokenize_with_offsets(text, tokenizer)

print("Gold entity:", repr(text[start:end]))
print("Context:", repr(text[start-15:end+20]))

for i, (s, e) in enumerate(offsets):
    if s < end + 5 and e > start - 5:
        print(i, (s, e), repr(text[s:e]), token_ids[i])


Gold entity: 'State Government'
Context: ' including the State Governments and the Universiti'
8446 (25078, 25079) ' ' 33
8447 (25079, 25082) 'the' 258
8448 (25082, 25083) ' ' 33
8449 (25083, 25088) 'State' 443
8450 (25088, 25089) ' ' 33
8451 (25089, 25100) 'Governments' 15163
8452 (25100, 25101) ' ' 33
8453 (25101, 25104) 'and' 276


In [28]:

from collections import Counter

audit = Counter()
examples = []

for texts, gold in [
    (fold1_text, fold1_gold),
    (fold2_text, fold2_gold)
]:
    for doc_id, annotations in gold.items():
        text = texts[doc_id]
        token_ids, offsets = tokenize_with_offsets(text, tokenizer)

        starts = set()
        ends = set()

        for s, e in offsets:
            starts.add(s)
            ends.add(e)

        for entity in annotations:
            start, end, label, *rest = entity
            gold_text = text[start:end]

            if rest and gold_text != rest[0]:
                audit["text_mismatch"] += 1

            start_ok = start in starts
            end_ok = end in ends

            if start_ok and end_ok:
                audit["both_aligned"] += 1
            elif start_ok and not end_ok:
                audit["end_only_misaligned"] += 1
            elif not start_ok and end_ok:
                audit["start_only_misaligned"] += 1
            else:
                audit["both_misaligned"] += 1

            if end not in ends:
                token = next(
                    ((s, e, text[s:e]) for s, e in offsets if s < end < e),
                    None
                )

                if token is not None and len(examples) < 20:
                    ts, te, token_text = token
                    examples.append({
                        "doc_id": doc_id,
                        "label": label,
                        "gold_text": gold_text,
                        "token_text": token_text,
                        "gold_end": end,
                        "token_end": te
                    })

print("Audit counts:", audit)
print("\nSample end-boundary mismatches:")
for example in examples[:10]:
    print(example)


Audit counts: Counter({'both_aligned': 5745, 'end_only_misaligned': 2565, 'both_misaligned': 42, 'start_only_misaligned': 41})

Sample end-boundary mismatches:
{'doc_id': '115651329', 'label': 'DATE', 'gold_text': '18.11.1994', 'token_text': '1994,', 'gold_end': 585, 'token_end': 586}
{'doc_id': '115651329', 'label': 'WIT', 'gold_text': 'Pardeep Kumar', 'token_text': 'Kumar,', 'gold_end': 1822, 'token_end': 1823}
{'doc_id': '115651329', 'label': 'WIT', 'gold_text': 'Mange Ram', 'token_text': 'Ram.', 'gold_end': 2323, 'token_end': 2324}
{'doc_id': '115651329', 'label': 'STAT', 'gold_text': 'IPC', 'token_text': 'IPC.', 'gold_end': 2494, 'token_end': 2495}
{'doc_id': '115651329', 'label': 'STAT', 'gold_text': 'IPC', 'token_text': 'IPC.', 'gold_end': 2688, 'token_end': 2689}
{'doc_id': '115651329', 'label': 'APP', 'gold_text': 'Jage Ram', 'token_text': 'Ram.', 'gold_end': 3591, 'token_end': 3592}
{'doc_id': '115651329', 'label': 'COURT', 'gold_text': 'High Court of Punjab and Haryana', 'to

In [30]:

import os
import sys
import torch

# Locate the directory containing legal_m1.py.
MODEL_DIR = "/kaggle/working"

if MODEL_DIR not in sys.path:
    sys.path.insert(0, MODEL_DIR)

from legal_m1 import GPTModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Prefer the 750M-token checkpoint.
CHECKPOINT_PATH = "/kaggle/working/m1_750_checkpoints/latest.pt"

# Fall back to the original 500M-token checkpoint if necessary.
if not os.path.exists(CHECKPOINT_PATH):
    CHECKPOINT_PATH = "/kaggle/working/m1_checkpoints/best.pt"

assert os.path.exists(CHECKPOINT_PATH), (
    "Checkpoint not found: " + CHECKPOINT_PATH
)

model = GPTModel(
    vocab_size=32000,
    d_model=768,
    n_layers=12,
    n_heads=12,
    ffn_dim=3072,
    dropout=0.2
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(checkpoint["model"], strict=True)
model = model.to(device)
model.eval()

print("Checkpoint:", CHECKPOINT_PATH)
print("Device:", device)
print("Tokens seen:", checkpoint.get("tokens_seen", "not recorded"))
print("Step:", checkpoint.get("step", "not recorded"))
print("Parameters:", sum(p.numel() for p in model.parameters()))


Checkpoint: /kaggle/working/m1_750_checkpoints/latest.pt
Device: cuda
Tokens seen: 750000330
Step: 36639
Parameters: 109612800


In [72]:

import types
import torch
import torch.nn as nn

def get_hidden_states(self, input_ids):
    x = self.token_embedding(input_ids)
    x = self.dropout(x)

    for block in self.blocks:
        x = block(x)

    x = self.norm(x)
    return x

model.get_hidden_states = types.MethodType(get_hidden_states, model)

# Smoke test with a short sequence.
test_ids = torch.randint(
    low=0,
    high=32000,
    size=(1, 32),
    device=device
)

with torch.no_grad():
    hidden = model.get_hidden_states(test_ids)

print("Input shape:", tuple(test_ids.shape))
print("Hidden-state shape:", tuple(hidden.shape))
print("Hidden states finite:", torch.isfinite(hidden).all().item())


Input shape: (1, 32)
Hidden-state shape: (1, 32, 768)
Hidden states finite: True


In [73]:

class CharacterBoundaryHead(nn.Module):
    def __init__(
        self,
        hidden_size=768,
        num_labels=12,
        char_embed_size=32,
        relative_embed_size=32,
        max_relative_position=256
    ):
        super().__init__()

        self.char_embedding = nn.Embedding(256, char_embed_size)
        self.relative_embedding = nn.Embedding(
            max_relative_position,
            relative_embed_size
        )

        self.max_relative_position = max_relative_position

        combined_size = (
            hidden_size + char_embed_size + relative_embed_size
        )

        self.start_layer = nn.Linear(combined_size, num_labels)
        self.end_layer = nn.Linear(combined_size, num_labels)

    def forward(self, hidden_states, text, offsets):
        device = hidden_states.device
        num_chars = len(text)
        hidden_size = hidden_states.size(-1)

        # Map each character to its BPE token.
        token_indices = torch.zeros(
            num_chars, dtype=torch.long, device=device
        )
        relative_positions = torch.zeros(
            num_chars, dtype=torch.long, device=device
        )

        for token_idx, (start, end) in enumerate(offsets):
            token_indices[start:end] = token_idx

            positions = torch.arange(
                end - start, dtype=torch.long, device=device
            )
            relative_positions[start:end] = positions.clamp(
                max=self.max_relative_position - 1
            )

        # Represent each character by its UTF-8 bytes.
        byte_lists = [list(char.encode("utf-8")) for char in text]

        max_bytes = max(len(x) for x in byte_lists)

        byte_ids = torch.zeros(num_chars, max_bytes, dtype=torch.long, device=device)
        byte_mask = torch.zeros(
            num_chars, max_bytes, dtype=torch.float32, device=device
        )

        for i, values in enumerate(byte_lists):
            byte_ids[i, :len(values)] = torch.tensor(
                values, dtype=torch.long, device=device
            )
            byte_mask[i, :len(values)] = 1.0

        byte_emb = self.char_embedding(byte_ids)
        char_emb = (
            (byte_emb * byte_mask.unsqueeze(-1)).sum(dim=1)
            / byte_mask.sum(dim=1, keepdim=True).clamp_min(1.0)
        )

        relative_emb = self.relative_embedding(relative_positions)

        token_hidden = hidden_states[0, token_indices, :]

        char_features = torch.cat(
            [token_hidden, char_emb, relative_emb], dim=-1
        ).unsqueeze(0)

        start_logits = self.start_layer(char_features)
        end_logits = self.end_layer(char_features)

        return start_logits, end_logits


In [74]:

NUM_LABELS = len(labels)
HIDDEN_SIZE = 768

boundary_head = CharacterBoundaryHead(
    hidden_size=HIDDEN_SIZE,
    num_labels=NUM_LABELS
).to(device)

print("New character-aware boundary head initialized.")


New character-aware boundary head initialized.


In [75]:
start_logits, end_logits = boundary_head(
    hidden_states,
    chunk_text,
    chunk_offsets
)

In [35]:
#character representation
def token_states_to_char_states(hidden_states, offsets, text_length):
    """
    hidden_states: (1, number_of_tokens, hidden_size)
    offsets: list of (start, end) character offsets
    Returns: (1, text_length, hidden_size)
    """
    char_states = hidden_states.new_zeros(
        1, text_length, hidden_states.size(-1)
    )

    counts = hidden_states.new_zeros(1, text_length, 1)

    for token_idx, (start, end) in enumerate(offsets):
        char_states[:, start:end, :] += hidden_states[:, token_idx:token_idx+1, :]
        counts[:, start:end, :] += 1

    char_states = char_states / counts.clamp_min(1)

    return char_states


In [36]:

doc_id = "115651329"
text = fold1_text[doc_id]

token_ids, offsets = tokenize_with_offsets(text, tokenizer)

n = 32
input_ids = torch.tensor(
    [token_ids[:n]],
    dtype=torch.long,
    device=device
)

with torch.no_grad():
    hidden_states = model.get_hidden_states(input_ids)

char_states = token_states_to_char_states(
    hidden_states,
    offsets[:n],
    len(text[:0]) + max(end for start, end in offsets[:n])
)

start_logits, end_logits = boundary_head(char_states)

print("Token states:", tuple(hidden_states.shape))
print("Character states:", tuple(char_states.shape))
print("Start logits:", tuple(start_logits.shape))
print("End logits:", tuple(end_logits.shape))
print("Finite:", torch.isfinite(char_states).all().item())


Token states: (1, 32, 768)
Character states: (1, 106, 768)
Start logits: (1, 106, 12)
End logits: (1, 106, 12)
Finite: True


In [37]:
#target
def make_boundary_targets(chunk_start, chunk_end, annotations, labels):
    """
    Returns start and end targets of shape:
        (chunk_length, number_of_labels)

    Only entities fully contained within the chunk are included.
    """
    chunk_length = chunk_end - chunk_start
    num_labels = len(labels)
    label_to_id = {label: i for i, label in enumerate(labels)}

    start_targets = torch.zeros(chunk_length, num_labels)
    end_targets = torch.zeros(chunk_length, num_labels)

    included = 0

    for entity in annotations:
        start, end, label = entity[:3]

        if start < chunk_start or end > chunk_end:
            continue

        if label not in label_to_id:
            continue

        start_idx = start - chunk_start
        end_idx = end - 1 - chunk_start

        start_targets[start_idx, label_to_id[label]] = 1.0
        end_targets[end_idx, label_to_id[label]] = 1.0
        included += 1

    return start_targets, end_targets, included


In [46]:

doc_id = "115651329"
text = fold1_text[doc_id]

chunk_start = 1750
chunk_end = 1870
chunk_text = text[chunk_start:chunk_end]

chunk_ids, chunk_offsets = tokenize_with_offsets(
    chunk_text, tokenizer
)

input_ids = torch.tensor(
    [chunk_ids],
    dtype=torch.long,
    device=device
)

model.eval()
boundary_head.train()

with torch.no_grad():
    hidden_states = model.get_hidden_states(input_ids)

start_logits, end_logits = boundary_head(
    hidden_states,
    chunk_text,
    chunk_offsets
)

start_targets, end_targets, included = make_boundary_targets(
    chunk_start,
    chunk_end,
    fold1_gold[doc_id],
    labels
)

# Add the batch dimension to the targets.
start_targets = start_targets.unsqueeze(0).to(device)
end_targets = end_targets.unsqueeze(0).to(device)

print("Hidden states:", tuple(hidden_states.shape))
print("Start logits:", tuple(start_logits.shape))
print("End logits:", tuple(end_logits.shape))
print("Start targets:", tuple(start_targets.shape))
print("End targets:", tuple(end_targets.shape))
print("Entities included:", included)


Hidden states: (1, 41, 768)
Start logits: (1, 120, 12)
End logits: (1, 120, 12)
Start targets: (1, 120, 12)
End targets: (1, 120, 12)
Entities included: 1


In [109]:
import torch.nn.functional as F

def boundary_loss(
    start_logits, end_logits,
    start_targets, end_targets,
    start_pos_weight, end_pos_weight,
    start_mask, end_mask
):
    start_targets = start_targets.to(start_logits.device, dtype=start_logits.dtype)
    end_targets = end_targets.to(end_logits.device, dtype=end_logits.dtype)
    start_mask = start_mask.to(start_logits.device, dtype=start_logits.dtype)
    end_mask = end_mask.to(end_logits.device, dtype=end_logits.dtype)

    if start_targets.dim() == 2:
        start_targets = start_targets.unsqueeze(0)
        end_targets = end_targets.unsqueeze(0)
        start_mask = start_mask.unsqueeze(0)
        end_mask = end_mask.unsqueeze(0)

    loss_start = F.binary_cross_entropy_with_logits(
        start_logits, start_targets,
        pos_weight=start_pos_weight, reduction="none"
    )
    loss_end = F.binary_cross_entropy_with_logits(
        end_logits, end_targets,
        pos_weight=end_pos_weight, reduction="none"
    )

    loss_start = (loss_start * start_mask).sum() / start_mask.sum().clamp_min(1)
    loss_end = (loss_end * end_mask).sum() / end_mask.sum().clamp_min(1)

    return loss_start + loss_end

In [77]:

test_loss = boundary_loss(
    start_logits,
    end_logits,
    start_targets,
    end_targets,
    start_pos_weight,
    end_pos_weight
)

print("Fixed-weight loss:", test_loss.item())
print("Finite:", torch.isfinite(test_loss).item())


Fixed-weight loss: 1.7851147651672363
Finite: True


In [43]:

# Get the BPE tokens and offsets for this chunk.
doc_id = "115651329"
text = fold1_text[doc_id]

chunk_start = 1750
chunk_end = 1870
chunk_text = text[chunk_start:chunk_end]

chunk_ids, chunk_offsets = tokenize_with_offsets(chunk_text, tokenizer)

input_ids = torch.tensor(
    [chunk_ids],
    dtype=torch.long,
    device=device
)

with torch.no_grad():
    hidden_states = model.get_hidden_states(input_ids)

char_states = token_states_to_char_states(
    hidden_states,
    chunk_offsets,
    len(chunk_text)
)

start_logits, end_logits = boundary_head(char_states)

start_targets, end_targets, included = make_boundary_targets(
    chunk_start,
    chunk_end,
    fold1_gold[doc_id],
    labels
)

optimizer = torch.optim.AdamW(
    boundary_head.parameters(),
    lr=1e-3
)

optimizer.zero_grad()

loss = boundary_loss(
    start_logits,
    end_logits,
    start_targets,
    end_targets
)

loss.backward()
optimizer.step()

print("Entities included:", included)
print("Loss:", loss.item())
print("Finite loss:", torch.isfinite(loss).item())
print("Start-head gradient:", boundary_head.start_layer.weight.grad.norm().item())
print("End-head gradient:", boundary_head.end_layer.weight.grad.norm().item())


Entities included: 1
Loss: 1.6096231937408447
Finite loss: True
Start-head gradient: 3.231882095336914
End-head gradient: 3.2859280109405518


In [54]:

optimizer = torch.optim.AdamW(
    boundary_head.parameters(),
    lr=1e-3
)

optimizer.zero_grad()

loss = boundary_loss(
    start_logits,
    end_logits,
    start_targets,
    end_targets
)

loss.backward()
optimizer.step()

print("Loss:", loss.item())
print("Finite loss:", torch.isfinite(loss).item())
print(
    "Start-head gradient:",
    boundary_head.start_layer.weight.grad.norm().item()
)
print(
    "End-head gradient:",
    boundary_head.end_layer.weight.grad.norm().item()
)


Loss: 1.8251299858093262
Finite loss: True
Start-head gradient: 4.116545677185059
End-head gradient: 3.7564547061920166


In [55]:

# Find the first token containing at least two characters.
found = False

for token_idx, (start, end) in enumerate(chunk_offsets):
    if end - start >= 2:
        char1 = start
        char2 = start + 1

        print("Token text:", repr(chunk_text[start:end]))
        print("Character 1:", repr(chunk_text[char1]))
        print("Character 2:", repr(chunk_text[char2]))

        print(
            "Start-logit difference:",
            (start_logits[0, char1] - start_logits[0, char2])
            .abs().max().item()
        )

        found = True
        break

if not found:
    print("No multi-character token found in this chunk.")


Token text: 'were'
Character 1: 'w'
Character 2: 'e'
Start-logit difference: 0.7013030052185059


Dataset = overlapping chunks


In [56]:

CHUNK_SIZE = 1536
CHUNK_OVERLAP = 256
CHUNK_STRIDE = CHUNK_SIZE - CHUNK_OVERLAP

def make_document_chunks(text, tokenizer,
                         chunk_size=CHUNK_SIZE,
                         overlap=CHUNK_OVERLAP):

    stride = chunk_size - overlap

    # Tokenize the document once, retaining character offsets.
    token_ids, token_offsets = tokenize_with_offsets(text, tokenizer)

    chunks = []
    start_token = 0

    while start_token < len(token_ids):
        end_token = min(start_token + chunk_size, len(token_ids))

        char_start = token_offsets[start_token][0]
        char_end = token_offsets[end_token - 1][1]

        chunks.append({
            "start_token": start_token,
            "end_token": end_token,
            "char_start": char_start,
            "char_end": char_end,
            "input_ids": token_ids[start_token:end_token],
            "text": text[char_start:char_end],
            "offsets": [
                (start - char_start, end - char_start)
                for start, end in token_offsets[start_token:end_token]
            ]
        })

        if end_token == len(token_ids):
            break

        start_token += stride

    return chunks


In [58]:

covered = 0
uncovered = []

for entity in annotations:
    start, end, label = entity[:3]

    matching_chunks = []

    for i, chunk in enumerate(chunks):
        if chunk["char_start"] <= start and end <= chunk["char_end"]:
            matching_chunks.append(i)

    if len(matching_chunks) > 0:
        covered += 1
    else:
        uncovered.append(entity)

print("Total entities:", len(annotations))
print("Entities covered by at least one chunk:", covered)
print("Uncovered entities:", len(uncovered))


Total entities: 104
Entities covered by at least one chunk: 104
Uncovered entities: 0


In [59]:

def assign_entities_to_chunks(chunks, annotations):
    assigned = [[] for _ in chunks]
    uncovered = []

    for entity in annotations:
        start, end, label = entity[:3]
        entity_mid = (start + end) / 2.0

        candidates = []

        for i, chunk in enumerate(chunks):
            if chunk["char_start"] <= start and end <= chunk["char_end"]:
                chunk_mid = (
                    chunk["char_start"] + chunk["char_end"]
                ) / 2.0

                distance = abs(entity_mid - chunk_mid)
                candidates.append((distance, i))

        if len(candidates) == 0:
            uncovered.append(entity)
            continue

        candidates.sort()
        best_chunk = candidates[0][1]
        assigned[best_chunk].append(entity)

    return assigned, uncovered


assigned_entities, uncovered = assign_entities_to_chunks(
    chunks, annotations
)

print("Assigned entities:", sum(len(x) for x in assigned_entities))
print("Uncovered entities:", len(uncovered))

for i, entity_list in enumerate(assigned_entities):
    print("Chunk", i, "assigned entities:", len(entity_list))


Assigned entities: 104
Uncovered entities: 0
Chunk 0 assigned entities: 32
Chunk 1 assigned entities: 22
Chunk 2 assigned entities: 12
Chunk 3 assigned entities: 13
Chunk 4 assigned entities: 25


In [60]:

def make_chunk_targets(chunk, assigned_entities, labels):
    chunk_length = len(chunk["text"])
    label_to_id = {label: i for i, label in enumerate(labels)}

    start_targets = torch.zeros(chunk_length, len(labels))
    end_targets = torch.zeros(chunk_length, len(labels))

    for entity in assigned_entities:
        start, end, label = entity[:3]

        # Convert document offsets to chunk-local offsets.
        local_start = start - chunk["char_start"]
        local_end = end - chunk["char_start"]

        if local_start < 0 or local_end > chunk_length:
            raise ValueError("Entity falls outside its assigned chunk.")

        if local_start >= local_end:
            raise ValueError("Invalid entity span.")

        if label not in label_to_id:
            raise ValueError("Unknown label: " + str(label))

        label_id = label_to_id[label]

        start_targets[local_start, label_id] = 1.0
        end_targets[local_end - 1, label_id] = 1.0

    return start_targets, end_targets


# Verify target generation on all five chunks.
all_chunk_targets = []

for i, chunk in enumerate(chunks):
    start_t, end_t = make_chunk_targets(
        chunk, assigned_entities[i], labels
    )

    assert start_t.shape == (len(chunk["text"]), len(labels))
    assert end_t.shape == (len(chunk["text"]), len(labels))

    assert int(start_t.sum().item()) == len(assigned_entities[i])
    assert int(end_t.sum().item()) == len(assigned_entities[i])

    all_chunk_targets.append((start_t, end_t))

    print(
        "Chunk", i,
        "| entities:", len(assigned_entities[i]),
        "| positive starts:", int(start_t.sum().item()),
        "| positive ends:", int(end_t.sum().item())
    )

print("All chunk targets verified.")


Chunk 0 | entities: 32 | positive starts: 32 | positive ends: 32
Chunk 1 | entities: 22 | positive starts: 22 | positive ends: 22
Chunk 2 | entities: 12 | positive starts: 12 | positive ends: 12
Chunk 3 | entities: 13 | positive starts: 13 | positive ends: 13
Chunk 4 | entities: 25 | positive starts: 25 | positive ends: 25
All chunk targets verified.


In [62]:

def build_chunk_dataset(text_dict, gold_dict, tokenizer):
    dataset = []
    total_entities = 0
    total_uncovered = 0

    for doc_id in text_dict:
        text = text_dict[doc_id]
        annotations = gold_dict[doc_id]

        doc_chunks = make_document_chunks(text, tokenizer)

        assigned, uncovered = assign_entities_to_chunks(
            doc_chunks, annotations
        )

        total_entities += len(annotations)
        total_uncovered += len(uncovered)

        for i, chunk in enumerate(doc_chunks):
            start_targets, end_targets = make_chunk_targets(
                chunk, assigned[i], labels
            )

            dataset.append({
                "doc_id": doc_id,
                "chunk_idx": i,
                "input_ids": chunk["input_ids"],
                "text": chunk["text"],
                "offsets": chunk["offsets"],
                "char_start": chunk["char_start"],
                "char_end": chunk["char_end"],
                "entities": assigned[i],
                "start_targets": start_targets,
                "end_targets": end_targets
            })

    print("Documents:", len(text_dict))
    print("Chunks:", len(dataset))
    print("Entities:", total_entities)
    print("Uncovered entities:", total_uncovered)

    return dataset


fold1_dataset = build_chunk_dataset(
    fold1_text, fold1_gold, tokenizer
)

fold2_dataset = build_chunk_dataset(
    fold2_text, fold2_gold, tokenizer
)

train_documents = set(fold1_text.keys()) | set(fold2_text.keys())

print("Unique training documents:", len(train_documents))


Documents: 35
Chunks: 315
Entities: 4120
Uncovered entities: 0
Documents: 35
Chunks: 320
Entities: 4273
Uncovered entities: 0
Unique training documents: 70


Validation split

In [64]:

import random

all_doc_ids = list(train_documents)

random.seed(42)
random.shuffle(all_doc_ids)

val_doc_ids = set(all_doc_ids[:7])
train_doc_ids = set(all_doc_ids[7:])

train_dataset = [
    item for item in fold1_dataset + fold2_dataset
    if item["doc_id"] in train_doc_ids
]

val_dataset = [
    item for item in fold1_dataset + fold2_dataset
    if item["doc_id"] in val_doc_ids
]

print("Training documents:", len(train_doc_ids))
print("Validation documents:", len(val_doc_ids))
print("Training chunks:", len(train_dataset))
print("Validation chunks:", len(val_dataset))
print(
    "Document overlap:",
    len(train_doc_ids.intersection(val_doc_ids))
)


Training documents: 63
Validation documents: 7
Training chunks: 586
Validation chunks: 49
Document overlap: 0


In [65]:

from collections import Counter

def count_entities_by_label(dataset):
    counts = Counter()

    for item in dataset:
        for entity in item["entities"]:
            counts[entity[2]] += 1

    return counts

train_counts = count_entities_by_label(train_dataset)
val_counts = count_entities_by_label(val_dataset)

print("Label             Train   Validation")

for label in labels:
    print(
        "{:<16} {:>6} {:>12}".format(
            label,
            train_counts[label],
            val_counts[label]
        )
    )


Label             Train   Validation
APP                 332           75
RESP                365           16
A.COUNSEL           182           13
R.COUNSEL           172           12
JUDGE               214           31
WIT                 177           10
AUTH                973           30
COURT               888          125
STAT               1144          116
PREC               1126           57
DATE               1462          113
CASENO              734           26


In [78]:

def calculate_positive_weights(dataset, labels):
    num_labels = len(labels)

    positive_starts = torch.zeros(num_labels)
    positive_ends = torch.zeros(num_labels)
    total_positions = 0

    for item in dataset:
        positive_starts += item["start_targets"].sum(dim=0)
        positive_ends += item["end_targets"].sum(dim=0)
        total_positions += item["start_targets"].shape[0]

    negative_starts = total_positions - positive_starts
    negative_ends = total_positions - positive_ends

    start_weights = (
        negative_starts / positive_starts.clamp_min(1)
    ).clamp(min=1.0, max=20.0)

    end_weights = (
        negative_ends / positive_ends.clamp_min(1)
    ).clamp(min=1.0, max=20.0)

    return start_weights.to(device), end_weights.to(device)


start_pos_weight, end_pos_weight = calculate_positive_weights(
    train_dataset, labels
)

print("Label             Start weight   End weight")
for i, label in enumerate(labels):
    print(
        "{:<16} {:>12.2f} {:>12.2f}".format(
            label,
            start_pos_weight[i].item(),
            end_pos_weight[i].item()
        )
    )


Label             Start weight   End weight
APP                     20.00        20.00
RESP                    20.00        20.00
A.COUNSEL               20.00        20.00
R.COUNSEL               20.00        20.00
JUDGE                   20.00        20.00
WIT                     20.00        20.00
AUTH                    20.00        20.00
COURT                   20.00        20.00
STAT                    20.00        20.00
PREC                    20.00        20.00
DATE                    20.00        20.00
CASENO                  20.00        20.00


In [79]:

doc_id = "115651329"
text = fold1_text[doc_id]

chunk_start = 1750
chunk_end = 1870
chunk_text = text[chunk_start:chunk_end]

chunk_ids, chunk_offsets = tokenize_with_offsets(
    chunk_text, tokenizer
)

input_ids = torch.tensor(
    [chunk_ids], dtype=torch.long, device=device
)

model.eval()
boundary_head.train()

with torch.no_grad():
    hidden_states = model.get_hidden_states(input_ids)

start_logits, end_logits = boundary_head(
    hidden_states, chunk_text, chunk_offsets
)

start_targets, end_targets, included = make_boundary_targets(
    chunk_start, chunk_end, fold1_gold[doc_id], labels
)

start_targets = start_targets.unsqueeze(0).to(device)
end_targets = end_targets.unsqueeze(0).to(device)

print("Logits:", start_logits.shape, end_logits.shape)
print("Targets:", start_targets.shape, end_targets.shape)
print("Entities included:", included)


Logits: torch.Size([1, 120, 12]) torch.Size([1, 120, 12])
Targets: torch.Size([1, 120, 12]) torch.Size([1, 120, 12])
Entities included: 1


In [89]:

def decode_entities(start_logits, end_logits, text_length, labels,
                    start_threshold=0.5, end_threshold=0.5,
                    max_span_length=200, top_k=100):

    start_probs = torch.sigmoid(start_logits[0]).detach().cpu()
    end_probs = torch.sigmoid(end_logits[0]).detach().cpu()
    predictions = []

    for label_id, label in enumerate(labels):
        start_scores = start_probs[:, label_id]
        end_scores = end_probs[:, label_id]

        start_positions = torch.where(start_scores >= start_threshold)[0]
        end_positions = torch.where(end_scores >= end_threshold)[0]

        if len(start_positions) > top_k:
            indices = torch.topk(start_scores[start_positions], top_k).indices
            start_positions = start_positions[indices]

        if len(end_positions) > top_k:
            indices = torch.topk(end_scores[end_positions], top_k).indices
            end_positions = end_positions[indices]

        start_positions = sorted(start_positions.tolist())
        end_positions = sorted(end_positions.tolist())
        used_ends = set()

        for start in start_positions:
            candidates = [
                end for end in end_positions
                if start <= end < min(text_length, start + max_span_length)
                and end not in used_ends
            ]

            if not candidates:
                continue

            end = min(candidates)
            used_ends.add(end)
            predictions.append((start, end + 1, label))

    return predictions


In [90]:

def calculate_span_metrics(gold_entities, predicted_entities, labels):
    results = {}
    f1_values = []

    for label in labels:
        gold_set = set()
        pred_set = set()

        for entity in gold_entities:
            start, end, entity_label = entity[:3]
            if entity_label == label:
                gold_set.add((start, end, entity_label))

        for entity in predicted_entities:
            start, end, entity_label = entity[:3]
            if entity_label == label:
                pred_set.add((start, end, entity_label))

        true_positive = len(gold_set.intersection(pred_set))
        false_positive = len(pred_set - gold_set)
        false_negative = len(gold_set - pred_set)

        precision = (
            true_positive / (true_positive + false_positive)
            if true_positive + false_positive > 0 else 0.0
        )

        recall = (
            true_positive / (true_positive + false_negative)
            if true_positive + false_negative > 0 else 0.0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0 else 0.0
        )

        results[label] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "tp": true_positive,
            "fp": false_positive,
            "fn": false_negative
        }

        f1_values.append(f1)

    results["macro_f1"] = sum(f1_values) / len(f1_values)

    return results


In [91]:

test_gold = [
    (10, 15, "APP"),
    (30, 42, "COURT")
]

test_predictions = [
    (10, 15, "APP"),
    (30, 42, "COURT")
]

test_metrics = calculate_span_metrics(
    test_gold, test_predictions, labels
)

print("Perfect-prediction macro F1:", test_metrics["macro_f1"])

for label in ["APP", "COURT"]:
    print(label, test_metrics[label])


Perfect-prediction macro F1: 0.16666666666666666
APP {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'tp': 1, 'fp': 0, 'fn': 0}
COURT {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'tp': 1, 'fp': 0, 'fn': 0}


Validation

In [92]:

from tqdm.auto import tqdm
from collections import defaultdict
import torch

def evaluate_documents(dataset, text_dict, gold_dict, model, boundary_head, labels, device):
    model.eval()
    boundary_head.eval()
    predictions = defaultdict(set)

    with torch.no_grad():
        for item in tqdm(dataset, desc="Evaluating", unit="chunk"):
            ids = torch.tensor([item["input_ids"]], dtype=torch.long, device=device)
            hidden = model.get_hidden_states(ids)
            start_logits, end_logits = boundary_head(hidden, item["text"], item["offsets"])

            spans = decode_entities(start_logits, end_logits, len(item["text"]), labels)
            for start, end, label in spans:
                gs = item["char_start"] + start
                ge = item["char_start"] + end
                if 0 <= gs < ge <= len(text_dict[item["doc_id"]]):
                    predictions[item["doc_id"]].add((gs, ge, label))

    gold, pred = [], []
    for doc_id in text_dict:
        gold.extend((e[0], e[1], e[2]) for e in gold_dict[doc_id])
        pred.extend(predictions[doc_id])

    return calculate_span_metrics(gold, pred, labels), predictions


In [93]:
val_metrics, val_predictions = evaluate_documents(
    val_dataset,
    val_text_dict,
    val_gold_dict,
    model,
    boundary_head,
    labels,
    device
)

print("Baseline macro F1:", val_metrics["macro_f1"])

Evaluating:   0%|          | 0/49 [00:00<?, ?chunk/s]

Baseline macro F1: 0.0


In [88]:

item = val_dataset[0]
ids = torch.tensor([item["input_ids"]], dtype=torch.long, device=device)

model.eval()
boundary_head.eval()

with torch.no_grad():
    hidden = model.get_hidden_states(ids)
    start_logits, end_logits = boundary_head(hidden, item["text"], item["offsets"])
    start_probs = torch.sigmoid(start_logits)
    end_probs = torch.sigmoid(end_logits)

for name, probs in [("Start", start_probs), ("End", end_probs)]:
    print(name)
    for threshold in [0.5, 0.7, 0.9, 0.99, 0.999]:
        count = (probs >= threshold).sum().item()
        print("  threshold", threshold, ":", count, "positive predictions")


Start
  threshold 0.5 : 22173 positive predictions
  threshold 0.7 : 5046 positive predictions
  threshold 0.9 : 80 positive predictions
  threshold 0.99 : 0 positive predictions
  threshold 0.999 : 0 positive predictions
End
  threshold 0.5 : 19898 positive predictions
  threshold 0.7 : 6526 positive predictions
  threshold 0.9 : 518 positive predictions
  threshold 0.99 : 0 positive predictions
  threshold 0.999 : 0 positive predictions


In [94]:

from tqdm.auto import tqdm
import os
import torch

model.eval()
for param in model.parameters():
    param.requires_grad = False

boundary_head.train()

optimizer = torch.optim.AdamW(
    boundary_head.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

checkpoint_dir = "/kaggle/working/lner_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

print("Encoder frozen:", not any(p.requires_grad for p in model.parameters()))
print("Trainable head parameters:",
      sum(p.numel() for p in boundary_head.parameters() if p.requires_grad))
print("Training chunks:", len(train_dataset))


Encoder frozen: True
Trainable head parameters: 36376
Training chunks: 586


In [131]:
def train_one_epoch(dataset, model, boundary_head, optimizer,
                    start_pos_weight, end_pos_weight, device):

    model.train()
    boundary_head.train()
    total_loss = 0.0

    progress = tqdm(dataset, desc="Training", unit="chunk")

    for item in progress:
        ids = torch.tensor(
            [item["input_ids"]], dtype=torch.long, device=device
        )

        optimizer.zero_grad(set_to_none=True)

        hidden = model.get_hidden_states(ids)

        start_logits, end_logits = boundary_head(
            hidden, item["text"], item["offsets"]
        )

        loss = boundary_loss(
            start_logits, end_logits,
            item["start_targets"], item["end_targets"],
            start_pos_weight, end_pos_weight,
            item["start_mask"], item["end_mask"]
        )

        if not torch.isfinite(loss):
            raise ValueError("Non-finite loss encountered.")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad] +
            list(boundary_head.parameters()),
            1.0
        )
        optimizer.step()

        total_loss += loss.item()
        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            avg=f"{total_loss / (progress.n + 1):.4f}"
        )

    return total_loss / len(dataset)

In [112]:
metrics, val_predictions = evaluate_documents(
    val_dataset,
    val_text_dict,
    val_gold_dict,
    model,
    boundary_head,
    labels,
    device
)

print("\nValidation Macro F1:", round(metrics["macro_f1"], 4))
print("\n{:<12} {:>8} {:>8} {:>8} {:>6}".format(
    "Label", "Prec.", "Recall", "F1", "TP"
))

for label in labels:
    m = metrics[label]
    print("{:<12} {:>8.3f} {:>8.3f} {:>8.3f} {:>6}".format(
        label, m["precision"], m["recall"], m["f1"], m["tp"]
    ))

Evaluating:   0%|          | 0/49 [00:00<?, ?chunk/s]


Validation Macro F1: 0.1915

Label           Prec.   Recall       F1     TP
APP             0.067    0.067    0.067      5
RESP            0.025    0.062    0.036      1
A.COUNSEL       0.042    0.077    0.054      1
R.COUNSEL       0.025    0.083    0.038      1
JUDGE           0.065    0.129    0.086      4
WIT             0.091    0.100    0.095      1
AUTH            0.250    0.067    0.105      2
COURT           0.545    0.336    0.416     42
STAT            0.412    0.483    0.444     56
PREC            0.200    0.158    0.176      9
DATE            0.694    0.761    0.726     86
CASENO          0.035    0.115    0.054      3


In [105]:
train_text_dict = {}
train_gold_dict = {}
val_text_dict = {}
val_gold_dict = {}

for doc_id in train_doc_ids:
    if doc_id in fold1_text:
        train_text_dict[doc_id] = fold1_text[doc_id]
        train_gold_dict[doc_id] = fold1_gold[doc_id]
    elif doc_id in fold2_text:
        train_text_dict[doc_id] = fold2_text[doc_id]
        train_gold_dict[doc_id] = fold2_gold[doc_id]

for doc_id in val_doc_ids:
    if doc_id in fold1_text:
        val_text_dict[doc_id] = fold1_text[doc_id]
        val_gold_dict[doc_id] = fold1_gold[doc_id]
    elif doc_id in fold2_text:
        val_text_dict[doc_id] = fold2_text[doc_id]
        val_gold_dict[doc_id] = fold2_gold[doc_id]

print("Training documents:", len(train_text_dict))
print("Validation documents:", len(val_text_dict))

Training documents: 63
Validation documents: 7


In [103]:
import torch
import torch.nn.functional as F

def rebuild_chunk_targets(dataset, text_dict, gold_dict, labels):
    label_to_id = {label: i for i, label in enumerate(labels)}
    by_doc = {}

    for item in dataset:
        doc_id = item["doc_id"]
        if doc_id not in by_doc:
            by_doc[doc_id] = []
        by_doc[doc_id].append(item)

    for doc_id, chunks in by_doc.items():
        annotations = gold_dict[doc_id]

        # Keep each entity assigned to its existing chunk.
        assigned = {}
        for chunk in chunks:
            for entity in chunk["entities"]:
                start, end, label = entity[:3]
                assigned[(start, end, label)] = chunk["chunk_idx"]

        for chunk in chunks:
            length = len(chunk["text"])
            start_targets = torch.zeros(length, len(labels))
            end_targets = torch.zeros(length, len(labels))
            start_mask = torch.ones(length, len(labels))
            end_mask = torch.ones(length, len(labels))

            cs, ce = chunk["char_start"], chunk["char_end"]
            chunk_idx = chunk["chunk_idx"]

            for entity in annotations:
                start, end, label = entity[:3]
                label_id = label_to_id[label]
                owner = assigned.get((start, end, label))

                # Mask duplicate start-boundary supervision.
                if cs <= start < ce and owner != chunk_idx:
                    start_mask[start - cs, label_id] = 0.

                # Mask duplicate end-boundary supervision.
                if cs < end <= ce and owner != chunk_idx:
                    end_mask[end - 1 - cs, label_id] = 0.

            # Restore positive targets only in the owning chunk.
            for entity in chunk["entities"]:
                start, end, label = entity[:3]
                label_id = label_to_id[label]
                local_start = start - cs
                local_end = end - 1 - cs

                if 0 <= local_start < length:
                    start_targets[local_start, label_id] = 1.
                if 0 <= local_end < length:
                    end_targets[local_end, label_id] = 1.

            chunk["start_targets"] = start_targets
            chunk["end_targets"] = end_targets
            chunk["start_mask"] = start_mask
            chunk["end_mask"] = end_mask

    return dataset

In [106]:
train_dataset = rebuild_chunk_targets(
    train_dataset, train_text_dict, train_gold_dict, labels
)

val_dataset = rebuild_chunk_targets(
    val_dataset, val_text_dict, val_gold_dict, labels
)

print("Train chunks:", len(train_dataset))
print("Validation chunks:", len(val_dataset))
print("Mask shapes:", train_dataset[0]["start_mask"].shape,
      train_dataset[0]["end_mask"].shape)

Train chunks: 586
Validation chunks: 49
Mask shapes: torch.Size([4344, 12]) torch.Size([4344, 12])


In [113]:
import os
import torch

best_path = "/kaggle/working/lner_checkpoints/best_epoch2.pt"

torch.save({
    "epoch": 2,
    "boundary_head": boundary_head.state_dict(),
    "optimizer": optimizer.state_dict(),
    "train_loss": 0.011299459122533754,
    "val_macro_f1": metrics["macro_f1"],
    "labels": labels
}, best_path)

print("Best checkpoint saved:", best_path)

Best checkpoint saved: /kaggle/working/lner_checkpoints/best_epoch2.pt


In [115]:
from tqdm.auto import tqdm

original_decoder = decode_entities
results = []
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

for threshold in thresholds:
    def decode_with_threshold(
        start_logits, end_logits, text_length, labels,
        start_threshold=0.5, end_threshold=0.5,
        max_span_length=200, top_k=100
    ):
        return original_decoder(
            start_logits, end_logits, text_length, labels,
            start_threshold=threshold,
            end_threshold=threshold,
            max_span_length=max_span_length,
            top_k=top_k
        )

    decode_entities = decode_with_threshold
    result, _ = evaluate_documents(
        val_dataset, val_text_dict, val_gold_dict,
        model, boundary_head, labels, device
    )
    results.append((threshold, result["macro_f1"]))
    print(f"Threshold {threshold:.1f}: macro F1 = {result['macro_f1']:.4f}")

decode_entities = original_decoder

print("\nBest threshold:", max(results, key=lambda x: x[1]))

Evaluating:   0%|          | 0/49 [00:00<?, ?chunk/s]

Threshold 0.3: macro F1 = 0.1138


Evaluating:   0%|          | 0/49 [00:00<?, ?chunk/s]

Threshold 0.4: macro F1 = 0.1553


Evaluating:   0%|          | 0/49 [00:00<?, ?chunk/s]

Threshold 0.5: macro F1 = 0.1915


... (4 outputs left out)

Evaluating:   0%|          | 0/49 [00:00<?, ?chunk/s]

Threshold 0.8: macro F1 = 0.1891


Evaluating:   0%|          | 0/49 [00:00<?, ?chunk/s]

Threshold 0.9: macro F1 = 0.1547

Best threshold: (0.7, 0.22602020914479448)


In [116]:
def decode_entities(
    start_logits, end_logits, text_length, labels,
    start_threshold=0.7, end_threshold=0.7,
    max_span_length=200, top_k=100
):
    start_probs = torch.sigmoid(start_logits[0]).detach().cpu()
    end_probs = torch.sigmoid(end_logits[0]).detach().cpu()
    predictions = []

    for label_id, label in enumerate(labels):
        start_scores = start_probs[:, label_id]
        end_scores = end_probs[:, label_id]

        starts = torch.where(start_scores >= start_threshold)[0]
        ends = torch.where(end_scores >= end_threshold)[0]

        if len(starts) > top_k:
            starts = starts[torch.topk(start_scores[starts], top_k).indices]
        if len(ends) > top_k:
            ends = ends[torch.topk(end_scores[ends], top_k).indices]

        starts = sorted(starts.tolist())
        ends = sorted(ends.tolist())
        used_ends = set()

        for start in starts:
            candidates = [
                end for end in ends
                if start <= end < min(text_length, start + max_span_length)
                and end not in used_ends
            ]
            if candidates:
                end = min(candidates)
                used_ends.add(end)
                predictions.append((start, end + 1, label))

    return predictions

In [122]:
import os
import torch
from tqdm.auto import tqdm

checkpoint_dir = "/kaggle/working/lner_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# Keep the pretrained encoder frozen
model.eval()
for param in model.parameters():
    param.requires_grad = False

# Continue with the current trained head (do not reinitialize it)
boundary_head.train()

optimizer = torch.optim.AdamW(
    boundary_head.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

history = []

for epoch in range(1, 6):
    print(f"\n{'='*15} Epoch {epoch}/5 {'='*15}")

    train_loss = train_one_epoch(
        train_dataset,
        model,
        boundary_head,
        optimizer,
        start_pos_weight,
        end_pos_weight,
        device
    )

    checkpoint_path = os.path.join(
        checkpoint_dir, f"continued_epoch_{epoch}.pt"
    )

    torch.save({
        "epoch": epoch,
        "boundary_head": boundary_head.state_dict(),
        "optimizer": optimizer.state_dict(),
        "train_loss": train_loss,
        "labels": labels
    }, checkpoint_path)

    history.append({"epoch": epoch, "train_loss": train_loss})

    print(f"Training loss: {train_loss:.6f}")
    print(f"Saved: {checkpoint_path}")

# Evaluate once after all five epochs
boundary_head.eval()

metrics, val_predictions = evaluate_documents(
    val_dataset,
    val_text_dict,
    val_gold_dict,
    model,
    boundary_head,
    labels,
    device
)

print("\n========== Final Validation ==========")
print(f"Macro F1: {metrics['macro_f1']:.4f}")

print("\n{:<12} {:>8} {:>8} {:>8} {:>6}".format(
    "Label", "Prec.", "Recall", "F1", "TP"
))

for label in labels:
    m = metrics[label]
    print("{:<12} {:>8.3f} {:>8.3f} {:>8.3f} {:>6}".format(
        label, m["precision"], m["recall"], m["f1"], m["tp"]
    ))

torch.save({
    "epoch": 5,
    "boundary_head": boundary_head.state_dict(),
    "optimizer": optimizer.state_dict(),
    "history": history,
    "val_macro_f1": metrics["macro_f1"],
    "labels": labels
}, os.path.join(checkpoint_dir, "continued_final.pt"))

print("\nFinal checkpoint saved.")


=============== Epoch 1/5 ===============


Training:   0%|          | 0/586 [00:00<?, ?chunk/s]

Training loss: 0.008800
Saved: /kaggle/working/lner_checkpoints/continued_epoch_1.pt

=============== Epoch 2/5 ===============


Training:   0%|          | 0/586 [00:00<?, ?chunk/s]

Training loss: 0.007049
Saved: /kaggle/working/lner_checkpoints/continued_epoch_2.pt

=============== Epoch 3/5 ===============


Training:   0%|          | 0/586 [00:00<?, ?chunk/s]

... (3 outputs left out)

Training:   0%|          | 0/586 [00:00<?, ?chunk/s]

Training loss: 0.005573
Saved: /kaggle/working/lner_checkpoints/continued_epoch_5.pt


Evaluating:   0%|          | 0/49 [00:00<?, ?chunk/s]


========== Final Validation ==========
Macro F1: 0.2255

Label           Prec.   Recall       F1     TP
APP             0.071    0.040    0.051      3
RESP            0.053    0.062    0.057      1
A.COUNSEL       0.062    0.077    0.069      1
R.COUNSEL       0.000    0.000    0.000      0
JUDGE           0.138    0.129    0.133      4
WIT             0.167    0.100    0.125      1
AUTH            0.286    0.067    0.108      2
COURT           0.621    0.432    0.509     54
STAT            0.521    0.431    0.472     50
PREC            0.200    0.070    0.104      4
DATE            0.885    0.885    0.885    100
CASENO          0.149    0.269    0.192      7

Final checkpoint saved.


In [123]:
import json
import os
import torch
from tqdm.auto import tqdm

model.eval()
boundary_head.eval()

test_predictions = {}

with torch.no_grad():
    for doc_id, text in tqdm(fold3_text.items(), desc="Predicting test", unit="doc"):
        chunks = make_document_chunks(
            text, tokenizer,
            chunk_size=CHUNK_SIZE,
            overlap=CHUNK_OVERLAP
        )

        entities = set()

        for item in chunks:
            ids = torch.tensor(
                [item["input_ids"]],
                dtype=torch.long,
                device=device
            )

            hidden = model.get_hidden_states(ids)
            start_logits, end_logits = boundary_head(
                hidden, item["text"], item["offsets"]
            )

            spans = decode_entities(
                start_logits, end_logits,
                len(item["text"]), labels
            )

            for start, end, label in spans:
                global_start = item["char_start"] + start
                global_end = item["char_start"] + end

                if 0 <= global_start < global_end <= len(text):
                    entities.add((global_start, global_end, label))

        test_predictions[doc_id] = [
            [start, end, label]
            for start, end, label in sorted(entities)
        ]

output_path = "/kaggle/working/submission.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(test_predictions, f, ensure_ascii=False)

print("Test documents:", len(test_predictions))
print("Total predicted entities:", sum(len(v) for v in test_predictions.values()))
print("Saved submission:", output_path)
print("Example:", next(iter(test_predictions.items())))

Predicting test:   0%|          | 0/35 [00:00<?, ?doc/s]

Test documents: 35
Total predicted entities: 3482
Saved submission: /kaggle/working/submission.json
Example: ('1128835', [[18, 40, 'COURT'], [70, 98, 'CASENO'], [99, 173, 'APP'], [114, 162, 'CASENO'], [258, 286, 'CASENO'], [287, 391, 'APP'], [302, 350, 'CASENO'], [392, 423, 'APP'], [499, 531, 'CASENO'], [535, 563, 'CASENO'], [564, 638, 'APP'], [579, 619, 'CASENO'], [587, 627, 'CASENO'], [735, 746, 'JUDGE'], [887, 901, 'DATE'], [917, 927, 'COURT'], [951, 975, 'CASENO'], [957, 988, 'CASENO'], [1188, 1198, 'AUTH'], [1410, 1414, 'AUTH'], [1529, 1549, 'STAT'], [1598, 1796, 'DATE'], [2470, 2482, 'DATE'], [3860, 3874, 'DATE'], [3940, 3950, 'AUTH'], [4516, 4528, 'STAT'], [4706, 4742, 'CASENO'], [4750, 4760, 'COURT'], [4812, 4824, 'STAT'], [4901, 4911, 'COURT'], [5405, 5421, 'AUTH'], [5494, 5510, 'DATE'], [6443, 6460, 'DATE'], [7471, 7481, 'COURT'], [7679, 7715, 'CASENO'], [7686, 7716, 'CASENO'], [7730, 7752, 'APP'], [9319, 9449, 'DATE'], [9922, 9932, 'AUTH'], [11436, 11452, 'AUTH'], [12159, 12

In [119]:
import os
import zipfile

json_path = "/kaggle/working/submission.json"
zip_path = "/kaggle/working/submission.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname="prediction.json")

# Verify the archive structure
with zipfile.ZipFile(zip_path, "r") as z:
    print("ZIP contents:", z.namelist())
    assert z.namelist() == ["prediction.json"]

print("Submission ZIP created:", zip_path)
print("Size:", round(os.path.getsize(zip_path) / 1024, 2), "KB")

ZIP contents: ['prediction.json']
Submission ZIP created: /kaggle/working/submission.zip
Size: 20.0 KB


In [124]:
import os
import zipfile

json_path = "/kaggle/working/submission.json"
zip_path = "/kaggle/working/submission_2.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname="prediction.json")

# Verify the archive structure
with zipfile.ZipFile(zip_path, "r") as z:
    print("ZIP contents:", z.namelist())
    assert z.namelist() == ["prediction.json"]

print("Submission ZIP created:", zip_path)
print("Size:", round(os.path.getsize(zip_path) / 1024, 2), "KB")

ZIP contents: ['prediction.json']
Submission ZIP created: /kaggle/working/submission_2.zip
Size: 22.58 KB


In [125]:
def decode_entities(start_logits, end_logits, text_length, labels,
    start_threshold=0.7, end_threshold=0.7,max_span_length=200, top_k=100):
    
    start_probs = torch.sigmoid(start_logits[0]).detach().cpu()
    end_probs = torch.sigmoid(end_logits[0]).detach().cpu()
    predictions = []

    for label_id, label in enumerate(labels):
        start_scores = start_probs[:, label_id]
        end_scores = end_probs[:, label_id]

        starts = torch.where(start_scores >= start_threshold)[0]
        ends = torch.where(end_scores >= end_threshold)[0]

        if len(starts) > top_k:
            starts = starts[torch.topk(start_scores[starts], top_k).indices]
        if len(ends) > top_k:
            ends = ends[torch.topk(end_scores[ends], top_k).indices]

        candidates = []

        for start in starts.tolist():
            valid_ends = ends[
                (ends >= start) &
                (ends < min(text_length, start + max_span_length))
            ]

            for end in valid_ends.tolist():
                score = (
                    start_scores[start].item() *
                    end_scores[end].item()
                )
                candidates.append((score, start, end))

        # Consider the strongest start-end pairs first
        candidates.sort(reverse=True)
        used_starts = set()
        used_ends = set()

        for score, start, end in candidates:
            if start in used_starts or end in used_ends:
                continue

            predictions.append((start, end + 1, label))
            used_starts.add(start)
            used_ends.add(end)

    return predictions

In [155]:
metrics, val_predictions = evaluate_documents(
    val_dataset,
    val_text_dict,
    val_gold_dict,
    model,
    boundary_head,
    labels,
    device
)

print("New validation macro F1:", round(metrics["macro_f1"], 4))

Evaluating:   0%|          | 0/49 [00:00<?, ?chunk/s]

New validation macro F1: 0.3167


In [129]:
import os
import torch

checkpoint_dir = "/kaggle/working/lner_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

torch.save({
    "boundary_head": boundary_head.state_dict(),
    "labels": labels,
    "val_macro_f1": metrics["macro_f1"],
    "decoder": "score_based",
    "threshold": 0.7
}, os.path.join(checkpoint_dir, "best_f1_02916.pt"))

print("Saved best head. Validation F1:", metrics["macro_f1"])

Saved best head. Validation F1: 0.29163523999636304


In [128]:
import json
import zipfile
import os

json_path = "/kaggle/working/prediction.json"
zip_path = "/kaggle/working/submission.zip"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(test_predictions, f, ensure_ascii=False)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname="prediction.json")

with zipfile.ZipFile(zip_path, "r") as z:
    print("ZIP contents:", z.namelist())
    assert z.namelist() == ["prediction.json"]

print("Submission ready:", zip_path)

ZIP contents: ['prediction.json']
Submission ready: /kaggle/working/submission.zip


In [132]:
for param in model.parameters():
    param.requires_grad = False

for block in model.blocks[-2:]:
    for param in block.parameters():
        param.requires_grad = True

for param in model.norm.parameters():
    param.requires_grad = True

for param in boundary_head.parameters():
    param.requires_grad = True

model.eval()  # Set by the training function as needed

GPTModel(
  (token_embedding): Embedding(32000, 768)
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (norm1): RMSNorm()
      (attn): CausalSelfAttention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (rope): RotaryEmbedding()
      )
      (norm2): RMSNorm()
      (ffn): FeedForward(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (norm): RMSNorm()
  (lm_head): Linear(in_features=768, out_features=32000, bias=False)
  (dropout): Dropout(p=0.2, inplace=False)
)

In [134]:
optimizer = torch.optim.AdamW([
    {"params": model.blocks[-2:].parameters(), "lr": 1e-5},
    {"params": model.norm.parameters(), "lr": 1e-5},
    {"params": boundary_head.parameters(), "lr": 3e-4}
], weight_decay=1e-4)

In [154]:

import os
import torch
from tqdm.auto import tqdm

checkpoint_dir = "/kaggle/working/lner_finetune"
os.makedirs(checkpoint_dir, exist_ok=True)

# Unfreeze only the last two Transformer blocks and final norm
for param in model.parameters():
    param.requires_grad = False

for block in model.blocks[-2:]:
    for param in block.parameters():
        param.requires_grad = True

for param in model.norm.parameters():
    param.requires_grad = True

for param in boundary_head.parameters():
    param.requires_grad = True

print("Trainable encoder parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

# Smaller LR for pretrained encoder, larger LR for task head
optimizer = torch.optim.AdamW([
    {"params": model.blocks[-2:].parameters(), "lr": 1e-5},
    {"params": model.norm.parameters(), "lr": 1e-5},
    {"params": boundary_head.parameters(), "lr": 3e-4}
], weight_decay=1e-4)

best_f1 = checkpoint["val_macro_f1"]

for epoch in range(3):

    model.train()
    boundary_head.train()

    train_loss = train_one_epoch(
        train_dataset,
        model,
        boundary_head,
        optimizer,
        start_pos_weight,
        end_pos_weight,
        device
    )

    model.eval()
    boundary_head.eval()

    val_metrics, val_predictions = evaluate_documents(
        val_dataset,
        val_text_dict,
        val_gold_dict,
        model,
        boundary_head,
        labels,
        device
    )

    val_f1 = val_metrics["macro_f1"]
    print(f"Train loss: {train_loss:.5f} | Val macro F1: {val_f1:.4f}")

    # Save the latest completed epoch (overwrites previous latest.pt)
    checkpoint = {
        "epoch": epoch,
        "encoder": model.state_dict(),
        "head": boundary_head.state_dict(),
        "optimizer": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_macro_f1": val_f1,
        "labels": labels
    }

    torch.save(checkpoint, f"{checkpoint_dir}/latest.pt")

    # Save best only when validation F1 improves
    if val_f1 > best_f1:
        best_f1 = val_f1

        torch.save({
            "epoch": epoch,
            "encoder": model.state_dict(),
            "head": boundary_head.state_dict(),
            "val_macro_f1": val_f1,
            "labels": labels
        }, f"{checkpoint_dir}/best.pt")

        print("New best checkpoint saved.")

print(f"\nBest validation macro F1: {best_f1:.4f}")


Trainable encoder parameters: 14173440


Training:   0%|          | 0/586 [00:00<?, ?chunk/s]

Evaluating:   0%|          | 0/49 [00:00<?, ?chunk/s]

Train loss: 0.00475 | Val macro F1: 0.3277


Training:   0%|          | 0/586 [00:00<?, ?chunk/s]

Evaluating:   0%|          | 0/49 [00:00<?, ?chunk/s]

Train loss: 0.00435 | Val macro F1: 0.3326


Training:   0%|          | 0/586 [00:00<?, ?chunk/s]

Evaluating:   0%|          | 0/49 [00:00<?, ?chunk/s]

Train loss: 0.00405 | Val macro F1: 0.3167

Best validation macro F1: 0.3414


In [157]:
checkpoint = torch.load(
    "/kaggle/working/lner_finetune/best.pt",
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["encoder"])
boundary_head.load_state_dict(checkpoint["head"])

model.to(device).eval()
boundary_head.to(device).eval()

print("Loaded best checkpoint")
print("Epoch:", checkpoint["epoch"])
print("Best validation macro F1:", checkpoint["val_macro_f1"])

Loaded best checkpoint
Epoch: 0
Best validation macro F1: 0.341418479148807


In [158]:
import json
import os
import torch
from tqdm.auto import tqdm

model.eval()
boundary_head.eval()

test_predictions = {}

with torch.no_grad():
    for doc_id, text in tqdm(fold3_text.items(), desc="Predicting test", unit="doc"):
        chunks = make_document_chunks(
            text, tokenizer,
            chunk_size=CHUNK_SIZE,
            overlap=CHUNK_OVERLAP
        )

        entities = set()

        for item in chunks:
            ids = torch.tensor(
                [item["input_ids"]],
                dtype=torch.long,
                device=device
            )

            hidden = model.get_hidden_states(ids)
            start_logits, end_logits = boundary_head(
                hidden, item["text"], item["offsets"]
            )

            spans = decode_entities(
                start_logits, end_logits,
                len(item["text"]), labels
            )

            for start, end, label in spans:
                global_start = item["char_start"] + start
                global_end = item["char_start"] + end

                if 0 <= global_start < global_end <= len(text):
                    entities.add((global_start, global_end, label))

        test_predictions[doc_id] = [
            [start, end, label]
            for start, end, label in sorted(entities)
        ]

output_path = "/kaggle/working/submission.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(test_predictions, f, ensure_ascii=False)

print("Saved submission:", output_path)

Predicting test:   0%|          | 0/35 [00:00<?, ?doc/s]

Saved submission: /kaggle/working/submission.json


In [159]:
import os
import zipfile

json_path = "/kaggle/working/submission.json"
zip_path = "/kaggle/working/submission_M1.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname="prediction.json")

# Verify the archive structure
with zipfile.ZipFile(zip_path, "r") as z:
    print("ZIP contents:", z.namelist())
    assert z.namelist() == ["prediction.json"]

print("Submission ZIP created:", zip_path)
print("Size:", round(os.path.getsize(zip_path) / 1024, 2), "KB")

ZIP contents: ['prediction.json']
Submission ZIP created: /kaggle/working/submission_M1.zip
Size: 22.16 KB


Phase 3

In [2]:

import os
import json
import random
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

PHASE3_FILES = {
    "CL_train": "Phase_3_RR/Train/CL_train.json",
    "CL_dev": "Phase_3_RR/Train/CL_dev.json",
    "IT_train": "Phase_3_RR/Train/IT_train.json",
    "IT_dev": "Phase_3_RR/Train/IT_dev.json",
    "label_vocab": "Phase_3_RR/Train/label_vocab.json",
    "CL_test": "Phase_3_RR/Test/CL_test.json",
    "IT_test": "Phase_3_RR/Test/IT_test.json"
}

SEED = 42
random.seed(SEED)

source_paths = {}

for source_name, file_name in PHASE3_FILES.items():
    print("Downloading:", source_name)

    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=file_name,
        repo_type="dataset",
        token=HF_TOKEN
    )

    source_paths[source_name] = path

    print("Path:", path)
    print()

print("All Phase 3 files downloaded/accessed successfully.")


Downloading: CL_train


CL_train.json: 0.00B [00:00, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/CL_train.json

Downloading: CL_dev


CL_dev.json: 0.00B [00:00, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/CL_dev.json

Downloading: IT_train


IT_train.json: 0.00B [00:00, ?B/s]

... (5 outputs left out)

CL_test.json: 0.00B [00:00, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Test/CL_test.json

Downloading: IT_test


IT_test.json: 0.00B [00:00, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Test/IT_test.json

All Phase 3 files downloaded/accessed successfully.


In [3]:

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

CL_train = load_json(source_paths["CL_train"])
CL_dev = load_json(source_paths["CL_dev"])

IT_train = load_json(source_paths["IT_train"])
IT_dev = load_json(source_paths["IT_dev"])

label_vocab = load_json(source_paths["label_vocab"])

CL_test = load_json(source_paths["CL_test"])
IT_test = load_json(source_paths["IT_test"])

print("CL train documents:", len(CL_train))
print("CL dev documents:", len(CL_dev))
print("IT train documents:", len(IT_train))
print("IT dev documents:", len(IT_dev))
print("CL test documents:", len(CL_test))
print("IT test documents:", len(IT_test))

print("\nLabel vocabulary:")
print(label_vocab)


CL train documents: 40
CL dev documents: 5
IT train documents: 40
IT dev documents: 5
CL test documents: 5
IT test documents: 5

Label vocabulary:
{'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2, 'ArgumentRespondent': 3, 'Statute': 4, 'Dissent': 5, 'PrecedentReliedUpon': 6, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 8, 'RulingByLowerCourt': 9, 'RatioOfTheDecision': 10, 'RulingByPresentCourt': 11, 'None': 12}


In [4]:

print("First CL training document:")
print(json.dumps(CL_train[0], indent=2, ensure_ascii=False))


First CL training document:
{
  "id": "CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733",
  "text": [
    "Consequent upon the repeal of the MRTP Act this complaint has been received by transfer on 04.03.2010 from the office of DGIR, MRTP Commission under Section 66(6) of the Competition Act, 2002.",
    "The facts as stated in the complaint, in brief, are: 2 .1 This complaint was filed by the All India Distillers Association against glass bottle manufacturers in India alleging restrictive and monopolistic trade practices being adopted by the four major manufacturers namely: Haldyn Glass Gujarat Limited, Hindustan National Glass and Industries, AGI Glasspac and Mohan Crystal Glass Work who together control 90 of the glass bottle market.",
    "2.2 It has been stated that Potable Alcohol Industry of India comprises of Indian Made Foreign Liquor (IMFL), Country Liquor, Beer and Wine.",
    "The packaging material for the potable alcohol is primarily glass bottles."

In [18]:
import os
import pickle

M1_CODE_PATH = "/kaggle/working/legal_m1.py"
TOKENIZER_CODE_PATH = "/kaggle/working/legal_bpe.py"
TOKENIZER_PATH = (
    "/kaggle/input/datasets/spikie/capstone-split/"
    "legal_bpe_32k.pkl"
)

# Load the original ByteLevelBPE definition into __main__.
with open(TOKENIZER_CODE_PATH, "r", encoding="utf-8") as f:
    tokenizer_source = f.read()

exec(compile(tokenizer_source, TOKENIZER_CODE_PATH, "exec"), globals())

# Now the pickle can resolve __main__.ByteLevelBPE.
with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

print("Tokenizer class:", type(tokenizer))
print("Vocabulary size:", tokenizer.vocab_size)
print("EOS token ID:", tokenizer.eos_id)

# Test encoding and decoding support.
sample = "The petitioner filed an appeal."
token_ids = tokenizer.encode_text(sample)

print("Sample:", sample)
print("Encoded length:", len(token_ids))
print("First token IDs:", token_ids[:20])

Tokenizer class: <class '__main__.ByteLevelBPE'>
Vocabulary size: 32000
EOS token ID: 0
Sample: The petitioner filed an appeal.
Encoded length: 10
First token IDs: [323, 33, 368, 33, 556, 33, 262, 33, 2987, 0]


In [21]:
from collections import Counter
import numpy as np

for name, data in [("CL_train",CL_train),("CL_dev",CL_dev),("IT_train",IT_train),("IT_dev",IT_dev)]:
    counts = Counter()
    lengths = []
    for doc in data:
        for sentence,label in zip(doc["text"],doc["labels"]):
            counts[label] += 1
            lengths.append(len(tokenizer.encode_text(sentence,add_eos=True)))
    print(f"\n{name}: {len(data)} documents, {len(lengths)} sentences")
    print("Labels:",dict(counts))
    print("Token lengths min/median/p90/p95/max:",np.min(lengths),int(np.median(lengths)),int(np.percentile(lengths,90)),int(np.percentile(lengths,95)),np.max(lengths))


CL_train: 40 documents, 10593 sentences
Labels: {'Fact': 2979, 'ArgumentPetitioner': 976, 'ArgumentRespondent': 996, 'RatioOfTheDecision': 2856, 'RulingByPresentCourt': 113, 'RulingByLowerCourt': 438, 'None': 53, 'PrecedentNotReliedUpon': 209, 'PrecedentReliedUpon': 1305, 'Statute': 367, 'Issue': 17, 'Dissent': 284}
Token lengths min/median/p90/p95/max: 2 58 123 151 1542

CL_dev: 5 documents, 1396 sentences
Labels: {'Fact': 249, 'ArgumentPetitioner': 344, 'None': 2, 'ArgumentRespondent': 236, 'RatioOfTheDecision': 288, 'RulingByPresentCourt': 18, 'RulingByLowerCourt': 56, 'Issue': 3, 'Statute': 58, 'PrecedentReliedUpon': 142}
Token lengths min/median/p90/p95/max: 2 61 122 155 744

IT_train: 40 documents, 6252 sentences
Labels: {'Fact': 1119, 'RulingByLowerCourt': 436, 'ArgumentPetitioner': 637, 'Issue': 90, 'Statute': 732, 'RatioOfTheDecision': 1445, 'RulingByPresentCourt': 194, 'PrecedentReliedUpon': 640, 'ArgumentRespondent': 352, 'PrecedentNotReliedUpon': 360, 'None': 244, 'Precede

In [30]:
import torch
import torch.nn as nn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 256
NUM_LABELS = len(label_vocab)
config = ckpt["config"]

model = GPTModel(
    vocab_size=config["vocab_size"],
    d_model=config["d_model"],
    n_layers=config["n_layers"],
    n_heads=config["n_heads"],
    ffn_dim=config["ffn_dim"],
    dropout=config["dropout"]
)
model.load_state_dict(ckpt["model"], strict=True)
model = model.to(DEVICE)

class M1RhetoricalClassifier(nn.Module):
    def __init__(self, backbone, d_model, num_labels, dropout=0.2):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, num_labels))

    def forward(self, input_ids, attention_mask):
        x = self.backbone.token_embedding(input_ids)
        x = self.backbone.dropout(x)
        for block in self.backbone.blocks:
            x = block(x)
        x = self.backbone.norm(x)
        last_idx = attention_mask.sum(dim=1) - 1
        pooled = x[torch.arange(x.size(0), device=x.device), last_idx]
        return self.classifier(pooled)

rr_model = M1RhetoricalClassifier(model, config["d_model"], NUM_LABELS).to(DEVICE)
print("Device:", DEVICE)
print("M1 parameters:", round(sum(p.numel() for p in rr_model.parameters()) / 1e6, 2), "M")
print("Output labels:", NUM_LABELS)
print("Initialized from Phase 1 checkpoint:", M1_PHASE1_CHECKPOINT)

Device: cuda
M1 parameters: 109.62 M
Output labels: 13
Initialized from Phase 1 checkpoint: /kaggle/working/m1_750_checkpoints/latest.pt


In [40]:
from torch.utils.data import Dataset, DataLoader
import torch

PAD_ID = 0
EOS_ID = tokenizer.eos_id

class RRDataset(Dataset):
    def __init__(self, documents, label_vocab, tokenizer, max_len=256):
        self.examples = []
        self.label_vocab = label_vocab
        self.tokenizer = tokenizer
        self.max_len = max_len
        for doc in documents:
            for sentence, label in zip(doc["text"], doc["labels"]):
                ids = tokenizer.encode_text(sentence, add_eos=True)
                if len(ids) > max_len:
                    ids = ids[:max_len-1] + [EOS_ID]
                self.examples.append((ids, label_vocab[label]))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ids, label = self.examples[idx]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

def rr_collate(batch):
    lengths = torch.tensor([len(item[0]) for item in batch], dtype=torch.long)
    max_len = int(lengths.max())
    input_ids = torch.full((len(batch), max_len), PAD_ID, dtype=torch.long)
    labels = torch.empty(len(batch), dtype=torch.long)
    for i, (ids, label) in enumerate(batch):
        input_ids[i, :len(ids)] = ids
        labels[i] = label
    attention_mask = torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1)
    return input_ids, attention_mask.long(), labels

cl_train_ds = RRDataset(CL_train, label_vocab, tokenizer, MAX_LEN)
cl_dev_ds = RRDataset(CL_dev, label_vocab, tokenizer, MAX_LEN)
it_train_ds = RRDataset(IT_train, label_vocab, tokenizer, MAX_LEN)
it_dev_ds = RRDataset(IT_dev, label_vocab, tokenizer, MAX_LEN)

print("CL train/dev:", len(cl_train_ds), len(cl_dev_ds))
print("IT train/dev:", len(it_train_ds), len(it_dev_ds))

sample_loader = DataLoader(cl_train_ds, batch_size=4, shuffle=False, collate_fn=rr_collate)
sample_ids, sample_mask, sample_labels = next(iter(sample_loader))
print("Batch shapes:", sample_ids.shape, sample_mask.shape, sample_labels.shape)
print("Lengths:", sample_mask.sum(dim=1).tolist())
print("Labels:", sample_labels.tolist())

CL train/dev: 10593 1396
IT train/dev: 6252 746
Batch shapes: torch.Size([4, 146]) torch.Size([4, 146]) torch.Size([4])
Lengths: [70, 146, 53, 25]
Labels: [0, 0, 0, 0]


In [41]:
import torch.nn.functional as F

if isinstance(rr_model, nn.DataParallel):
    rr_model = rr_model.module

DEVICE = "cuda:0"
rr_model = rr_model.to(DEVICE)
rr_model.zero_grad(set_to_none=True)
rr_model.train()

input_ids, attention_mask, labels = next(iter(sample_loader))
input_ids = input_ids.to(DEVICE)
attention_mask = attention_mask.to(DEVICE)
labels = labels.to(DEVICE)

logits = rr_model(input_ids, attention_mask)
loss = nn.CrossEntropyLoss()(logits, labels)
loss.backward()

print("GPU count:", torch.cuda.device_count())
print("Sanity-check GPU:", torch.cuda.get_device_name(0))
print("Logits shape:", logits.shape)
print("Loss:", round(loss.item(), 4))
print("Logits finite:", torch.isfinite(logits).all().item())
print("Classifier gradient:", rr_model.classifier[1].weight.grad is not None)
print("Backbone gradient:", rr_model.backbone.blocks[0].attn.qkv.weight.grad is not None)

rr_model.zero_grad(set_to_none=True)

GPU count: 2
Sanity-check GPU: Tesla T4
Logits shape: torch.Size([4, 13])
Loss: 3.5082
Logits finite: True
Classifier gradient: True
Backbone gradient: True


In [42]:
import os, json, random, numpy as np, torch
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MAX_LEN = 256
BATCH_SIZE = 4
GRAD_ACCUM = 4
EPOCHS = 5
LR = 1e-5
WEIGHT_DECAY = 0.01
NUM_LABELS = len(label_vocab)

PHASE3_DIR = "/kaggle/working/phase3_rr/experiment_1_m1_full"
os.makedirs(PHASE3_DIR, exist_ok=True)

train_config = {
    "source_checkpoint": M1_PHASE1_CHECKPOINT,
    "max_len": MAX_LEN,
    "batch_size_per_gpu": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "epochs": EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "num_labels": NUM_LABELS,
    "seed": SEED,
    "gpu_count": 2,
    "gpu_type": "Tesla T4",
    "multi_gpu": "DistributedDataParallel",
    "label_vocab": label_vocab
}

with open(os.path.join(PHASE3_DIR, "config.json"), "w") as f:
    json.dump(train_config, f, indent=2)

for name, docs in [
    ("CL_train", CL_train), ("CL_dev", CL_dev),
    ("IT_train", IT_train), ("IT_dev", IT_dev)
]:
    with open(os.path.join(PHASE3_DIR, name + ".json"), "w", encoding="utf-8") as f:
        json.dump(docs, f, ensure_ascii=False)

print("CL train/dev:", len(cl_train_ds), len(cl_dev_ds))
print("IT train/dev:", len(it_train_ds), len(it_dev_ds))
print("Total training sentences:", len(cl_train_ds) + len(it_train_ds))
print("Checkpoint directory:", PHASE3_DIR)
print("DDP GPUs:", torch.cuda.device_count())
print("Per-GPU batch size:", BATCH_SIZE)
print("Nominal effective batch size:", BATCH_SIZE * GRAD_ACCUM * 2)

CL train/dev: 10593 1396
IT train/dev: 6252 746
Total training sentences: 16845
Checkpoint directory: /kaggle/working/phase3_rr/experiment_1_m1_full
DDP GPUs: 2
Per-GPU batch size: 4
Nominal effective batch size: 32


In [51]:

%%writefile /kaggle/working/train_rr_m1.py
import os, sys, json, pickle, random, argparse
from contextlib import nullcontext
import numpy as np
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from sklearn.metrics import f1_score, precision_score, recall_score

sys.path.insert(0, "/kaggle/working")
from legal_m1 import GPTModel

PHASE3_DIR = "/kaggle/working/phase3_rr/experiment_1_m1_full"
CHECKPOINT_PATH = "/kaggle/working/m1_750_checkpoints/latest.pt"
TOKENIZER_PATH = "/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl"
TOKENIZER_CODE = "/kaggle/working/legal_bpe.py"

with open(os.path.join(PHASE3_DIR, "config.json")) as f:
    cfg = json.load(f)

LABEL_VOCAB = cfg["label_vocab"]
NUM_LABELS = cfg["num_labels"]
MAX_LEN = cfg["max_len"]
BATCH_SIZE = cfg["batch_size_per_gpu"]
GRAD_ACCUM = cfg["grad_accum"]
EPOCHS = cfg["epochs"]
LR = cfg["lr"]
WEIGHT_DECAY = cfg["weight_decay"]
SEED = cfg["seed"]
PAD_ID = 0

def setup():
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend="nccl", device_id=local_rank)
    return local_rank, torch.device("cuda", local_rank)

def cleanup():
    if dist.is_initialized():
        dist.destroy_process_group()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

class RRDataset(Dataset):
    def __init__(self, documents, tokenizer, max_len):
        self.examples = []
        for doc in documents:
            for sentence, label in zip(doc["text"], doc["labels"]):
                ids = tokenizer.encode_text(sentence, add_eos=True)
                if len(ids) > max_len:
                    ids = ids[:max_len-1] + [tokenizer.eos_id]
                self.examples.append((ids, LABEL_VOCAB[label]))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ids, label = self.examples[idx]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

def rr_collate(batch):
    lengths = torch.tensor([len(x[0]) for x in batch], dtype=torch.long)
    max_len = int(lengths.max())
    input_ids = torch.full((len(batch), max_len), PAD_ID, dtype=torch.long)
    labels = torch.empty(len(batch), dtype=torch.long)
    for i, (ids, label) in enumerate(batch):
        input_ids[i, :len(ids)] = ids
        labels[i] = label
    mask = torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1)
    return input_ids, mask.long(), labels

class M1RhetoricalClassifier(nn.Module):
    def __init__(self, backbone, d_model, num_labels, dropout=0.2):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        x = self.backbone.token_embedding(input_ids)
        x = self.backbone.dropout(x)
        for block in self.backbone.blocks:
            x = block(x)
        x = self.backbone.norm(x)
        last_idx = attention_mask.sum(dim=1) - 1
        pooled = x[torch.arange(x.size(0), device=x.device), last_idx]
        return self.classifier(pooled)

def evaluate(model, loader, criterion, device, max_batches=None):
    model.eval()
    all_labels, all_preds = [], []
    total_loss, total = 0.0, 0

    with torch.no_grad():
        for batch_idx, (input_ids, mask, labels) in enumerate(loader):
            if max_batches is not None and batch_idx >= max_batches:
                break
            input_ids = input_ids.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(input_ids, mask)
                loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total += labels.size(0)
            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())

    metrics = {
        "loss": total_loss / max(total, 1),
        "macro_f1": f1_score(
            all_labels, all_preds, labels=list(range(NUM_LABELS)),
            average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            all_labels, all_preds, average="weighted", zero_division=0
        ),
        "macro_precision": precision_score(
            all_labels, all_preds, labels=list(range(NUM_LABELS)),
            average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            all_labels, all_preds, labels=list(range(NUM_LABELS)),
            average="macro", zero_division=0
        )
    }
    model.train()
    return metrics

def main():
    parser = argparse.ArgumentParser()
    mode = parser.add_mutually_exclusive_group()
    mode.add_argument("--smoke-test", action="store_true")
    mode.add_argument("--sanity-check", action="store_true")
    args = parser.parse_args()

    local_rank, device = setup()
    rank = dist.get_rank()
    world_size = dist.get_world_size()

    try:
        set_seed(SEED + rank)

        with open(TOKENIZER_CODE, "r", encoding="utf-8") as f:
            exec(compile(f.read(), TOKENIZER_CODE, "exec"), globals())
        with open(TOKENIZER_PATH, "rb") as f:
            tokenizer = pickle.load(f)

        def load_docs(name):
            with open(os.path.join(PHASE3_DIR, name + ".json"), encoding="utf-8") as f:
                return json.load(f)

        cl_train = RRDataset(load_docs("CL_train"), tokenizer, MAX_LEN)
        it_train = RRDataset(load_docs("IT_train"), tokenizer, MAX_LEN)
        cl_dev = RRDataset(load_docs("CL_dev"), tokenizer, MAX_LEN)
        it_dev = RRDataset(load_docs("IT_dev"), tokenizer, MAX_LEN)
        train_ds = ConcatDataset([cl_train, it_train])

        train_sampler = DistributedSampler(
            train_ds, num_replicas=world_size, rank=rank,
            shuffle=True, seed=SEED
        )
        train_loader = DataLoader(
            train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
            collate_fn=rr_collate, num_workers=2, pin_memory=True
        )
        cl_dev_loader = DataLoader(
            cl_dev, batch_size=8, shuffle=False,
            collate_fn=rr_collate, num_workers=2, pin_memory=True
        )
        it_dev_loader = DataLoader(
            it_dev, batch_size=8, shuffle=False,
            collate_fn=rr_collate, num_workers=2, pin_memory=True
        )

        ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
        model_cfg = ckpt["config"]
        backbone = GPTModel(
            vocab_size=model_cfg["vocab_size"],
            d_model=model_cfg["d_model"],
            n_layers=model_cfg["n_layers"],
            n_heads=model_cfg["n_heads"],
            ffn_dim=model_cfg["ffn_dim"],
            dropout=model_cfg["dropout"]
        )
        backbone.load_state_dict(ckpt["model"], strict=True)
        model = M1RhetoricalClassifier(
            backbone, model_cfg["d_model"], NUM_LABELS
        ).to(device)
        model = DDP(model, device_ids=[local_rank], output_device=local_rank)

        counts = np.zeros(NUM_LABELS, dtype=np.int64)
        for dataset in [cl_train, it_train]:
            for _, label in dataset.examples:
                counts[label] += 1
        weights = torch.tensor(
            [1.0 / np.sqrt(max(int(counts[i]), 1)) for i in range(NUM_LABELS)],
            dtype=torch.float32
        )
        weights = (weights / weights.mean()).to(device)
        criterion = nn.CrossEntropyLoss(weight=weights)
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
        )
        scaler = torch.amp.GradScaler("cuda")

        if args.smoke_test:
            epochs_to_run = 1
            max_train_batches = 2
            max_eval_batches = 1
            save_name = None
            history_name = None
            mode_name = "SMOKE TEST"
        elif args.sanity_check:
            epochs_to_run = 1
            max_train_batches = None
            max_eval_batches = None
            save_name = "sanity_best.pt"
            history_name = "sanity_history.json"
            mode_name = "FULL-EPOCH SANITY CHECK"
        else:
            epochs_to_run = EPOCHS
            max_train_batches = None
            max_eval_batches = None
            save_name = "best.pt"
            history_name = "history.json"
            mode_name = "FULL TRAINING"

        if rank == 0:
            print("DDP world size:", world_size)
            print("GPU:", torch.cuda.get_device_name(local_rank))
            print("Train examples:", len(train_ds))
            print("CL/IT validation:", len(cl_dev), len(it_dev))
            print("Initialized from Phase 1:", CHECKPOINT_PATH)
            print("Mode:", mode_name)

        best_macro_f1 = -1.0
        history = []

        for epoch in range(epochs_to_run):
            model.train()
            train_sampler.set_epoch(epoch)
            optimizer.zero_grad(set_to_none=True)

            batch_limit = min(max_train_batches, len(train_loader)) if max_train_batches is not None else len(train_loader)
            running_loss, seen = 0.0, 0
            num_groups = (batch_limit + GRAD_ACCUM - 1) // GRAD_ACCUM

            for step, (input_ids, mask, labels) in enumerate(train_loader):
                if step >= batch_limit:
                    break

                group_start = (step // GRAD_ACCUM) * GRAD_ACCUM
                group_size = min(GRAD_ACCUM, batch_limit - group_start)
                update_now = ((step + 1) % GRAD_ACCUM == 0) or (step + 1 == batch_limit)

                input_ids = input_ids.to(device, non_blocking=True)
                mask = mask.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                sync_context = nullcontext() if update_now else model.no_sync()
                with sync_context:
                    with torch.autocast(device_type="cuda", dtype=torch.float16):
                        logits = model(input_ids, mask)
                        raw_loss = criterion(logits, labels)
                        loss = raw_loss / group_size
                    scaler.scale(loss).backward()

                running_loss += raw_loss.detach().item() * labels.size(0)
                seen += labels.size(0)

                if update_now:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

            loss_stats = torch.tensor([running_loss, seen], dtype=torch.float64, device=device)
            dist.all_reduce(loss_stats, op=dist.ReduceOp.SUM)
            train_loss = (loss_stats[0] / loss_stats[1].clamp_min(1)).item()

            # Rank 0 evaluates and saves; rank 1 waits until evaluation is done.
            if rank == 0:
                cl_metrics = evaluate(
                    model.module, cl_dev_loader, criterion, device,
                    max_batches=max_eval_batches
                )
                it_metrics = evaluate(
                    model.module, it_dev_loader, criterion, device,
                    max_batches=max_eval_batches
                )
                mean_macro_f1 = (cl_metrics["macro_f1"] + it_metrics["macro_f1"]) / 2
                record = {
                    "epoch": epoch + 1,
                    "train_loss": train_loss,
                    "cl_validation": cl_metrics,
                    "it_validation": it_metrics,
                    "mean_domain_macro_f1": mean_macro_f1
                }
                history.append(record)

                print(
                    f"Epoch {epoch+1}/{epochs_to_run} | train loss {train_loss:.4f} | "
                    f"CL macro-F1 {cl_metrics['macro_f1']:.4f} | "
                    f"IT macro-F1 {it_metrics['macro_f1']:.4f} | "
                    f"Mean {mean_macro_f1:.4f}"
                )

                if history_name:
                    with open(os.path.join(PHASE3_DIR, history_name), "w") as f:
                        json.dump(history, f, indent=2)

                if save_name and mean_macro_f1 > best_macro_f1:
                    best_macro_f1 = mean_macro_f1
                    torch.save({
                        "model": model.module.state_dict(),
                        "epoch": epoch + 1,
                        "best_mean_domain_macro_f1": best_macro_f1,
                        "cl_validation": cl_metrics,
                        "it_validation": it_metrics,
                        "config": cfg,
                        "source_checkpoint": CHECKPOINT_PATH
                    }, os.path.join(PHASE3_DIR, save_name))
                    print("Saved checkpoint:", save_name)

            dist.barrier()

        if rank == 0:
            if args.smoke_test:
                print("Smoke test completed. No checkpoint saved.")
            elif args.sanity_check:
                print("Sanity check completed.")
                print("Checkpoint:", os.path.join(PHASE3_DIR, "sanity_best.pt"))
                print("History:", os.path.join(PHASE3_DIR, "sanity_history.json"))
            else:
                print("Training complete.")
                print("Best mean domain macro-F1:", round(best_macro_f1, 4))
                print("Best checkpoint:", os.path.join(PHASE3_DIR, "best.pt"))

    finally:
        cleanup()

if __name__ == "__main__":
    main()


Overwriting /kaggle/working/train_rr_m1.py


In [56]:
#Adding tqdm
script_path = "/kaggle/working/train_rr_m1.py"

with open(script_path, "r") as f:
    script = f.read()

script = script.replace(
    "import os, sys, json, pickle, random, argparse\n",
    "import os, sys, json, pickle, random, argparse, time\n"
    "from tqdm.auto import tqdm\n",
    1
)

old = '''        best_macro_f1 = -1.0
        history = []

        for epoch in range(epochs_to_run):
            model.train()
            train_sampler.set_epoch(epoch)
            optimizer.zero_grad(set_to_none=True)

            batch_limit = min(max_train_batches, len(train_loader)) if max_train_batches is not None else len(train_loader)
            running_loss, seen = 0.0, 0
            num_groups = (batch_limit + GRAD_ACCUM - 1) // GRAD_ACCUM

            for step, (input_ids, mask, labels) in enumerate(train_loader):
                if step >= batch_limit:
                    break
'''

new = '''        best_macro_f1 = -1.0
        history = []
        run_start = time.time()
        epoch_times = []

        for epoch in range(epochs_to_run):
            epoch_start = time.time()
            model.train()
            train_sampler.set_epoch(epoch)
            optimizer.zero_grad(set_to_none=True)

            batch_limit = min(max_train_batches, len(train_loader)) if max_train_batches is not None else len(train_loader)
            running_loss, seen = 0.0, 0
            num_groups = (batch_limit + GRAD_ACCUM - 1) // GRAD_ACCUM
            progress = tqdm(
                train_loader,
                total=batch_limit,
                desc=f"Epoch {epoch+1}/{epochs_to_run}",
                disable=(rank != 0),
                dynamic_ncols=True
            )

            for step, (input_ids, mask, labels) in enumerate(progress):
                if step >= batch_limit:
                    break
'''

assert old in script, "Could not find the expected training-loop start."
script = script.replace(old, new, 1)

old = '''                running_loss += raw_loss.detach().item() * labels.size(0)
                seen += labels.size(0)

                if update_now:
'''
new = '''                running_loss += raw_loss.detach().item() * labels.size(0)
                seen += labels.size(0)

                if rank == 0:
                    elapsed = time.time() - epoch_start
                    eta = (elapsed / (step + 1)) * max(batch_limit - step - 1, 0)
                    progress.set_postfix(
                        loss=f"{raw_loss.detach().item():.3f}",
                        elapsed=f"{elapsed/60:.1f}m",
                        eta=f"{eta/60:.1f}m"
                    )

                if update_now:
'''
assert old in script, "Could not find the expected training-loop metrics section."
script = script.replace(old, new, 1)

old = '''            train_loss = (loss_stats[0] / loss_stats[1].clamp_min(1)).item()

            # Rank 0 evaluates and saves; rank 1 waits until evaluation is done.
            if rank == 0:
'''
new = '''            train_loss = (loss_stats[0] / loss_stats[1].clamp_min(1)).item()
            epoch_duration = time.time() - epoch_start
            epoch_times.append(epoch_duration)

            # Rank 0 evaluates and saves; rank 1 waits until evaluation is done.
            if rank == 0:
                elapsed_total = time.time() - run_start
                avg_epoch = sum(epoch_times) / len(epoch_times)
                remaining_epochs = epochs_to_run - epoch - 1
                estimated_remaining = avg_epoch * remaining_epochs
                print(
                    f"Epoch training duration: {epoch_duration/60:.2f} min | "
                    f"Total elapsed: {elapsed_total/60:.2f} min | "
                    f"Estimated run time remaining: {estimated_remaining/60:.2f} min"
                )
'''
assert old in script, "Could not find the expected post-training section."
script = script.replace(old, new, 1)

with open(script_path, "w") as f:
    f.write(script)

import py_compile
py_compile.compile(script_path, doraise=True)
print("Progress bar and ETA added. Syntax check passed.")

Progress bar and ETA added. Syntax check passed.


In [57]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_rr_m1.py --smoke-test

W1016 13:44:31.921000 488 torch/distributed/run.py:852] 
W1016 13:44:31.921000 488 torch/distributed/run.py:852] *****************************************
W1016 13:44:31.921000 488 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1016 13:44:31.921000 488 torch/distributed/run.py:852] *****************************************
[W1016 13:44:32.356037499 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1016 13:44:35.545233903 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1016 13:44:35.557281910 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
DDP world size: 2
GPU: Tesla T4
Train examples: 16845
CL/IT validation: 1396 746
Initialized from Phase 1: /kaggle/working/m1_750_checkpoints/latest

In [58]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_rr_m1.py --sanity-check

W1016 13:44:53.524000 589 torch/distributed/run.py:852] 
W1016 13:44:53.524000 589 torch/distributed/run.py:852] *****************************************
W1016 13:44:53.524000 589 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1016 13:44:53.524000 589 torch/distributed/run.py:852] *****************************************
[W1016 13:44:53.688998649 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1016 13:44:56.833691809 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1016 13:44:56.863366860 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
DDP world size: 2
GPU: Tesla T4
Train examples: 16845
CL/IT validation: 1396 746
Initialized from Phase 1: /kaggle/working/m1_750_checkpoints/latest

In [59]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_rr_m1.py

W1016 13:48:49.588000 692 torch/distributed/run.py:852] 
W1016 13:48:49.588000 692 torch/distributed/run.py:852] *****************************************
W1016 13:48:49.588000 692 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1016 13:48:49.588000 692 torch/distributed/run.py:852] *****************************************
[W1016 13:48:49.806763507 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1016 13:48:52.972677872 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1016 13:48:52.982294193 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
DDP world size: 2
GPU: Tesla T4
Train examples: 16845
CL/IT validation: 1396 746
Initialized from Phase 1: /kaggle/working/m1_750_checkpoints/latest

In [77]:
import os, sys, json, pickle, zipfile
import torch
import torch.nn as nn

sys.path.insert(0, "/kaggle/working")
from legal_m1 import GPTModel

BASE = "/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR"
OUT = "/kaggle/working/phase3_rr/experiment_1_m1_full"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 256
PAD_ID = 0

with open(os.path.join(BASE, "Train/label_vocab.json"), encoding="utf-8") as f:
    label_vocab = json.load(f)
id_to_label = {v: k for k, v in label_vocab.items()}

with open("/kaggle/working/legal_bpe.py", encoding="utf-8") as f:
    exec(compile(f.read(), "legal_bpe.py", "exec"), globals())
with open("/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl", "rb") as f:
    tokenizer = pickle.load(f)

class M1RhetoricalClassifier(nn.Module):
    def __init__(self, backbone, d_model, num_labels, dropout=0.2):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        x = self.backbone.token_embedding(input_ids)
        x = self.backbone.dropout(x)
        for block in self.backbone.blocks:
            x = block(x)
        x = self.backbone.norm(x)
        last_idx = attention_mask.sum(dim=1) - 1
        pooled = x[torch.arange(x.size(0), device=x.device), last_idx]
        return self.classifier(pooled)

# Load the Phase 1 architecture configuration
phase1 = torch.load(
    "/kaggle/working/m1_750_checkpoints/latest.pt",
    map_location="cpu",
    weights_only=False
)
cfg = phase1["config"]

backbone = GPTModel(
    vocab_size=cfg["vocab_size"],
    d_model=cfg["d_model"],
    n_layers=cfg["n_layers"],
    n_heads=cfg["n_heads"],
    ffn_dim=cfg["ffn_dim"],
    dropout=cfg["dropout"]
)

# Build classifier and load the Phase 3 best checkpoint
ckpt = torch.load(
    os.path.join(OUT, "best.pt"),
    map_location="cpu",
    weights_only=False
)
model = M1RhetoricalClassifier(
    backbone, cfg["d_model"], len(label_vocab)
)
model.load_state_dict(ckpt["model"], strict=True)
model.to(DEVICE).eval()

@torch.no_grad()
def predict_sentences(sentences, batch_size=16):
    predictions = []
    for start in range(0, len(sentences), batch_size):
        batch = sentences[start:start + batch_size]
        encoded = []

        for sentence in batch:
            ids = tokenizer.encode_text(sentence, add_eos=True)
            if len(ids) > MAX_LEN:
                ids = ids[:MAX_LEN - 1] + [tokenizer.eos_id]
            encoded.append(ids)

        lengths = [len(ids) for ids in encoded]
        max_len = max(lengths)
        input_ids = torch.full((len(batch), max_len), PAD_ID, dtype=torch.long)
        mask = torch.zeros((len(batch), max_len), dtype=torch.long)

        for i, ids in enumerate(encoded):
            input_ids[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
            mask[i, :len(ids)] = 1

        logits = model(input_ids.to(DEVICE), mask.to(DEVICE))
        predictions.extend(logits.argmax(dim=1).cpu().tolist())

    return [id_to_label[i] for i in predictions]

prediction = {}
for domain, filename in [("CL", "CL_test.json"), ("IT", "IT_test.json")]:
    with open(os.path.join(BASE, "Test", filename), encoding="utf-8") as f:
        documents = json.load(f)

    for doc_id, sentences in documents.items():
        prediction[doc_id] = predict_sentences(sentences)
        assert len(prediction[doc_id]) == len(sentences), doc_id

    print(domain, "documents:", len(documents),
          "| sentences:", sum(len(x) for x in documents.values()))

json_path = "/kaggle/working/prediction.json"
zip_path = "/kaggle/working/submission.zip"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(prediction, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname="prediction.json")

with zipfile.ZipFile(zip_path, "r") as z:
    assert z.namelist() == ["prediction.json"]
    check = json.loads(z.read("prediction.json").decode("utf-8"))

assert all(
    len(check[doc_id]) == len(sentences)
    for filename in ["CL_test.json", "IT_test.json"]
    for documents in [json.load(open(os.path.join(BASE, "Test", filename), encoding="utf-8"))]
    for doc_id, sentences in documents.items()
)

print("\nSubmission ready:", zip_path)
print("ZIP contents:", zipfile.ZipFile(zip_path).namelist())
print("Total documents:", len(prediction))
print("Total predictions:", sum(len(v) for v in prediction.values()))

CL documents: 5 | sentences: 1339
IT documents: 5 | sentences: 858

Submission ready: /kaggle/working/submission.zip
ZIP contents: ['prediction.json']
Total documents: 10
Total predictions: 2197


In [79]:
import os, json, zipfile

BASE = "/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR"

with zipfile.ZipFile("/kaggle/working/submission.zip") as z:
    prediction = json.loads(z.read("prediction.json"))

expected = {}
for domain in ["CL", "IT"]:
    with open(f"{BASE}/Test/{domain}_test.json", encoding="utf-8") as f:
        expected.update(json.load(f))

print("Document IDs match:", set(expected) == set(prediction))
print("All sentence counts match:", all(
    len(expected[k]) == len(prediction[k]) for k in expected
))
print("All labels valid:", all(
    p in label_vocab for labels in prediction.values() for p in labels
))

for doc_id, sentences in list(expected.items())[:2]:
    print(f"\n--- {doc_id} ---")
    for sentence, label in list(zip(sentences, prediction[doc_id]))[:12]:
        print(f"[{label}] {sentence[:220]}")

Document IDs match: True
All sentence counts match: True
All labels valid: True

--- HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489 ---
[None] Manmohan, J. 1 .
[Fact] Present writ petition has been filed, challenging the notice dated 26th February, 2016 and respondent Number 1s order dated 18th February, 2016 whereby the respondent-CCI has directed the opposite parties in Reference Ca
[Fact] The CCI has further directed individuals who have been identified in the DGs investigation report as key persons of the parties to appear before the CCI for oral hearing on the investigation report of DG.
[Fact] It is pertinent to mention that the DG had found five leading tyre manufacturing companies under aegis of their association being Automotive Tyre Manufacturers Association (ATMA) who have indulged indirectly in carteliza
[ArgumentPetitioner] Mr. Rajshekhar Rao, learned Counsel for the petitioner submits that the respondent No. 1 is acting contrary to Sect

PHASE 4


In [2]:

import os
import json
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE4_FILES = {
    "train": "Phase_4_CJPE/Train/train.json",
    "dev": "Phase_4_CJPE/Train/dev.json",
    "test": "Phase_4_CJPE/Test/test.json",
}

phase4_paths = {}

for split, filename in PHASE4_FILES.items():
    print(f"Accessing {split}: {filename}")

    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
    )

    phase4_paths[split] = path
    print("Cached path:", path)

with open(phase4_paths["train"], "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(phase4_paths["dev"], "r", encoding="utf-8") as f:
    dev_data = json.load(f)

with open(phase4_paths["test"], "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("\nDataset sizes:")
print("Train:", len(train_data))
print("Dev:", len(dev_data))
print("Test:", len(test_data))

print("\nFirst training example keys:", train_data[0].keys())
print("First test example keys:", test_data[0].keys())

Accessing train: Phase_4_CJPE/Train/train.json


Phase_4_CJPE/Train/train.json:   0%|          | 0.00/737M [00:00<?, ?B/s]

Cached path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/train.json
Accessing dev: Phase_4_CJPE/Train/dev.json


Phase_4_CJPE/Train/dev.json:   0%|          | 0.00/45.0M [00:00<?, ?B/s]

Cached path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/dev.json
Accessing test: Phase_4_CJPE/Test/test.json


Phase_4_CJPE/Test/test.json:   0%|          | 0.00/35.3M [00:00<?, ?B/s]

Cached path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Test/test.json

Dataset sizes:
Train: 37387
Dev: 1988
Test: 1517

First training example keys: dict_keys(['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5'])
First test example keys: dict_keys(['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5'])


In [4]:

from collections import Counter
import numpy as np

# 1. Inspect the binary label distribution
print("TRAIN LABEL DISTRIBUTION")
train_labels = [row["label"] for row in train_data]
print(Counter(train_labels))

print("\nDEV LABEL DISTRIBUTION")
dev_labels = [row["label"] for row in dev_data]
print(Counter(dev_labels))

# 2. Inspect document lengths in characters
# We will measure token lengths later using each model's tokenizer.
def print_length_stats(name, data):
    lengths = [len(row["text"]) for row in data]
    print(f"\n{name} document lengths (characters)")
    print("Min:", int(np.min(lengths)))
    print("Median:", int(np.median(lengths)))
    print("90th percentile:", int(np.percentile(lengths, 90)))
    print("95th percentile:", int(np.percentile(lengths, 95)))
    print("99th percentile:", int(np.percentile(lengths, 99)))
    print("Max:", int(np.max(lengths)))

print_length_stats("TRAIN", train_data)
print_length_stats("DEV", dev_data)
print_length_stats("TEST", test_data)

# 3. Inspect expert fields without using them as model inputs
expert_fields = [
    "expert_1", "expert_2", "expert_3",
    "expert_4", "expert_5"
]

print("\nEXPERT FIELD INSPECTION")
for field in expert_fields:
    present = sum(
        field in row and row[field] is not None
        for row in train_data
    )
    examples = [
        type(row[field]).__name__
        for row in train_data
        if field in row and row[field] is not None
    ]
    print(field, "| present:", present,
          "| example type:", examples[0] if examples else "None")

# 4. Check whether training and development IDs overlap
train_ids = set(row["id"] for row in train_data)
dev_ids = set(row["id"] for row in dev_data)

print("\nDATA SPLIT CHECK")
print("Train IDs:", len(train_ids))
print("Dev IDs:", len(dev_ids))
print("Train-dev ID overlap:", len(train_ids & dev_ids))

# 5. Verify required fields
print("\nFIELD VALIDATION")
for name, data in [
    ("train", train_data),
    ("dev", dev_data),
    ("test", test_data)
]:
    missing_id = sum("id" not in row for row in data)
    missing_text = sum(
        "text" not in row or not isinstance(row["text"], str)
        for row in data
    )
    print(name, "| missing IDs:", missing_id,
          "| missing/invalid text:", missing_text)


TRAIN LABEL DISTRIBUTION
Counter({0: 22067, 1: 15320})

DEV LABEL DISTRIBUTION
Counter({0: 994, 1: 994})

TRAIN document lengths (characters)
Min: 217
Median: 13023
90th percentile: 38996
95th percentile: 54753
99th percentile: 115746
Max: 2492701

DEV document lengths (characters)
Min: 904
Median: 16700
90th percentile: 40495
95th percentile: 54452
99th percentile: 117842
Max: 260198

TEST document lengths (characters)
Min: 434
Median: 17320
90th percentile: 42859
95th percentile: 58170
99th percentile: 108673
Max: 233000

EXPERT FIELD INSPECTION
expert_1 | present: 0 | example type: None
expert_2 | present: 0 | example type: None
expert_3 | present: 0 | example type: None
expert_4 | present: 0 | example type: None
expert_5 | present: 0 | example type: None

DATA SPLIT CHECK
Train IDs: 32305
Dev IDs: 994
Train-dev ID overlap: 0

FIELD VALIDATION
train | missing IDs: 0 | missing/invalid text: 0
dev | missing IDs: 0 | missing/invalid text: 0
test | missing IDs: 0 | missing/invalid text:

In [5]:

from collections import Counter, defaultdict

id_counts = Counter(row["id"] for row in train_data)

duplicate_ids = [
    case_id for case_id, count in id_counts.items()
    if count > 1
]

print("Total training records:", len(train_data))
print("Unique training IDs:", len(id_counts))
print("IDs appearing multiple times:", len(duplicate_ids))
print("Extra records due to repeated IDs:",
      sum(count - 1 for count in id_counts.values() if count > 1))

# Inspect repeated-ID label consistency
labels_by_id = defaultdict(set)
texts_by_id = defaultdict(set)

for row in train_data:
    labels_by_id[row["id"]].add(row["label"])
    texts_by_id[row["id"]].add(row["text"])

conflicting_labels = [
    case_id for case_id in duplicate_ids
    if len(labels_by_id[case_id]) > 1
]

different_texts = [
    case_id for case_id in duplicate_ids
    if len(texts_by_id[case_id]) > 1
]

print("\nRepeated IDs with conflicting labels:",
      len(conflicting_labels))
print("Repeated IDs with different text:",
      len(different_texts))

print("\nExamples of repeated IDs:")
for case_id in duplicate_ids[:5]:
    rows = [row for row in train_data if row["id"] == case_id]
    print("\nID:", case_id)
    print("Number of records:", len(rows))
    print("Labels:", [row["label"] for row in rows])
    print("Text lengths:", [len(row["text"]) for row in rows])


Total training records: 37387
Unique training IDs: 32305
IDs appearing multiple times: 5082
Extra records due to repeated IDs: 5082

Repeated IDs with conflicting labels: 0
Repeated IDs with different text: 0

Examples of repeated IDs:

ID: 2019_890
Number of records: 2
Labels: [1, 1]
Text lengths: [28354, 28354]

ID: 2014_170
Number of records: 2
Labels: [0, 0]
Text lengths: [31637, 31637]

ID: 2010_721
Number of records: 2
Labels: [1, 1]
Text lengths: [18402, 18402]

ID: 2008_1460
Number of records: 2
Labels: [1, 1]
Text lengths: [20133, 20133]

ID: 2008_188
Number of records: 2
Labels: [1, 1]
Text lengths: [28326, 28326]


In [6]:

train_data_original = train_data

train_data_unique = []
seen_ids = set()

for row in train_data_original:
    if row["id"] not in seen_ids:
        train_data_unique.append(row)
        seen_ids.add(row["id"])

print("Original records:", len(train_data_original))
print("Unique training cases:", len(train_data_unique))

train_data = train_data_unique

print("Training records after deduplication:", len(train_data))
print("Unique IDs after deduplication:",
      len(set(row["id"] for row in train_data)))


Original records: 37387
Unique training cases: 32305
Training records after deduplication: 32305
Unique IDs after deduplication: 32305


In [11]:
import torch
import torch.nn as nn

class M1ForCJPE(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):
        x = self.backbone.token_embedding(input_ids)
        x = self.backbone.dropout(x)

        for block in self.backbone.blocks:
            x = block(x)

        x = self.backbone.norm(x)

        # Mean pooling over non-padding tokens
        mask = attention_mask.unsqueeze(-1).float()
        x = (x * mask).sum(dim=1)
        x = x / mask.sum(dim=1).clamp(min=1.0)

        return self.classifier(x)

In [12]:
import torch

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory allocated (GB):",
      round(torch.cuda.memory_allocated() / 1024**3, 2))

Total parameters: 109612800
Trainable parameters: 109612800
GPU: Tesla T4
GPU memory allocated (GB): 1.24


In [28]:
import torch, numpy as np, random
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm

SEED, MAX_LEN, PAD_ID = 42, 1024, 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

class CJDataset(Dataset):
    def __init__(self, records, tokenizer, max_len=1024, is_test=False):
        self.ids, self.labels, self.case_ids = [], [], []
        self.is_test = is_test
        for row in tqdm(records, desc="Tokenizing", unit="case"):
            tokens = tokenizer.encode_text(row["text"])[:max_len]
            self.ids.append(torch.tensor(tokens, dtype=torch.long))
            self.case_ids.append(row["id"])
            if not is_test:
                self.labels.append(int(row["label"]))
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, idx):
        if self.is_test:
            return self.ids[idx], idx
        return self.ids[idx], self.labels[idx]

def collate_cj(batch):
    sequences = [item[0] for item in batch]
    lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
    input_ids = pad_sequence(sequences, batch_first=True, padding_value=PAD_ID)
    attention_mask = (torch.arange(input_ids.size(1)).unsqueeze(0) < lengths.unsqueeze(1)).long()
    labels = torch.tensor([item[1] for item in batch], dtype=torch.long)
    return input_ids, attention_mask, labels

train_ds = CJDataset(train_data, tokenizer, MAX_LEN)
dev_ds = CJDataset(dev_data, tokenizer, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate_cj, num_workers=0, pin_memory=True)
dev_loader = DataLoader(dev_ds, batch_size=4, shuffle=False, collate_fn=collate_cj, num_workers=0, pin_memory=True)

print("Train:", len(train_ds), "| Dev:", len(dev_ds))
input_ids, attention_mask, labels = next(iter(train_loader))
print("Input shape:", input_ids.shape, "| Mask shape:", attention_mask.shape)
print("Labels:", labels.tolist(), "| Real-token counts:", attention_mask.sum(dim=1).tolist())
print("EOS tokens:", (input_ids == tokenizer.eos_id).sum(dim=1).tolist())
print("Padding positions:", (attention_mask == 0).sum(dim=1).tolist())

Tokenizing:   0%|          | 0/32305 [00:00<?, ?case/s]

Tokenizing:   0%|          | 0/1988 [00:00<?, ?case/s]

Train: 32305 | Dev: 1988
Input shape: torch.Size([2, 1024]) | Mask shape: torch.Size([2, 1024])
Labels: [0, 1] | Real-token counts: [1024, 1024]
EOS tokens: [0, 0]
Padding positions: [0, 0]


In [8]:
import os, torch

DATA_DIR = "/kaggle/working/cjpe_m1"
os.makedirs(DATA_DIR, exist_ok=True)

torch.save(
    {"ids": train_ds.ids, "labels": train_ds.labels, "case_ids": train_ds.case_ids},
    f"{DATA_DIR}/train.pt"
)
torch.save(
    {"ids": dev_ds.ids, "labels": dev_ds.labels, "case_ids": dev_ds.case_ids},
    f"{DATA_DIR}/dev.pt"
)

print("Saved tokenized datasets:", DATA_DIR)

Saved tokenized datasets: /kaggle/working/cjpe_m1


In [13]:
%%writefile /kaggle/working/cjpe_m1/train_m1_cjpe.py
import os, sys, random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torch.nn.utils.rnn import pad_sequence
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score
sys.path.insert(0, "/kaggle/working")
import legal_m1

DATA_DIR = "/kaggle/working/cjpe_m1"
CKPT = "/kaggle/working/m1_750_checkpoints/latest.pt"
OUT = DATA_DIR + "/checkpoints"
MAX_LEN, BATCH_SIZE, ACCUM, EPOCHS = 1024, 2, 4, 3
LR, PAD_ID, SEED = 1e-5, 0, 42

class CJDataset(Dataset):
    def __init__(self, path):
        d = torch.load(path, map_location="cpu", weights_only=False)
        self.ids, self.labels = d["ids"], d["labels"]
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return self.ids[i][:MAX_LEN], int(self.labels[i])

def collate(batch):
    seqs = [x[0] for x in batch]
    lengths = torch.tensor([len(x) for x in seqs])
    ids = pad_sequence(seqs, batch_first=True, padding_value=PAD_ID)
    mask = (torch.arange(ids.size(1))[None, :] < lengths[:, None]).long()
    labels = torch.tensor([x[1] for x in batch], dtype=torch.long)
    return ids, mask, labels

class M1ForCJPE(nn.Module):
    def __init__(self, backbone, dim):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(dim, 2)
    def forward(self, ids, mask):
        x = self.backbone.dropout(self.backbone.token_embedding(ids))
        for block in self.backbone.blocks:
            x = block(x)
        x = self.backbone.norm(x)
        m = mask.unsqueeze(-1).to(x.dtype)
        x = (x * m).sum(1) / m.sum(1).clamp(min=1)
        return self.classifier(x)

def main():
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = int(os.environ["RANK"])
    world = int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(local_rank)
    dist.init_process_group("nccl")
    device = torch.device("cuda", local_rank)
    random.seed(SEED + rank)
    np.random.seed(SEED + rank)
    torch.manual_seed(SEED + rank)
    os.makedirs(OUT, exist_ok=True)

    train_ds, dev_ds = CJDataset(DATA_DIR + "/train.pt"), CJDataset(DATA_DIR + "/dev.pt")
    sampler = DistributedSampler(train_ds, num_replicas=world, rank=rank, shuffle=True, seed=SEED)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, collate_fn=collate, num_workers=0, pin_memory=True)
    dev_loader = DataLoader(dev_ds, batch_size=8, shuffle=False, collate_fn=collate, num_workers=0, pin_memory=True)

    raw = torch.load(CKPT, map_location="cpu", weights_only=False)
    cfg = raw["config"]
    backbone = legal_m1.GPTModel(cfg["vocab_size"], cfg["d_model"], cfg["n_layers"], cfg["n_heads"], cfg["ffn_dim"], cfg["dropout"])
    backbone.load_state_dict(raw["model"], strict=True)
    model = M1ForCJPE(backbone, cfg["d_model"]).to(device)
    model = DDP(model, device_ids=[local_rank])
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scaler = GradScaler("cuda")
    start_epoch, best_f1 = 0, -1.0
    resume_path = OUT + "/resume.pt"

    if os.path.exists(resume_path):
        state = torch.load(resume_path, map_location="cpu", weights_only=False)
        model.module.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        start_epoch, best_f1 = state["epoch"], state["best_f1"]
        if rank == 0: print("Resuming at epoch", start_epoch + 1)

    for epoch in range(start_epoch, EPOCHS):
        sampler.set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        iterator = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", disable=(rank != 0))
        for step, (ids, mask, labels) in enumerate(iterator):
            ids, mask, labels = ids.to(device, non_blocking=True), mask.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            sync_now = ((step + 1) % ACCUM == 0) or (step + 1 == len(train_loader))
            ctx = model.no_sync() if not sync_now else torch.enable_grad()
            with ctx:
                with autocast("cuda", dtype=torch.float16):
                    logits = model(ids, mask)
                    loss = F.cross_entropy(logits, labels) / ACCUM
                scaler.scale(loss).backward()
            if sync_now:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            total_loss += loss.item() * ACCUM
            if rank == 0: iterator.set_postfix(loss=f"{loss.item()*ACCUM:.4f}")
        loss_tensor = torch.tensor(total_loss / len(train_loader), device=device)
        dist.all_reduce(loss_tensor, op=dist.ReduceOp.SUM)
        train_loss = loss_tensor.item() / world
        dist.barrier()

        if rank == 0:
            model.module.eval()
            ys, ps, val_loss, n = [], [], 0.0, 0
            with torch.no_grad():
                for ids, mask, labels in tqdm(dev_loader, desc="Validation"):
                    ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
                    with autocast("cuda", dtype=torch.float16):
                        logits = model.module(ids, mask)
                        loss = F.cross_entropy(logits, labels)
                    val_loss += loss.item() * len(labels)
                    n += len(labels)
                    ps.extend(logits.argmax(1).cpu().tolist())
                    ys.extend(labels.cpu().tolist())
            mf1 = f1_score(ys, ps, average="macro")
            prec = precision_score(ys, ps, average="macro", zero_division=0)
            rec = recall_score(ys, ps, average="macro", zero_division=0)
            print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss/n:.4f} macro_f1={mf1:.4f} macro_precision={prec:.4f} macro_recall={rec:.4f}")
            if mf1 > best_f1:
                best_f1 = mf1
                torch.save({"model": model.module.state_dict(), "config": cfg, "epoch": epoch+1, "macro_f1": mf1}, OUT + "/best.pt")
                print("Saved best checkpoint.")
            torch.save({"model": model.module.state_dict(), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch+1, "best_f1": best_f1}, resume_path)
        dist.barrier()
    dist.destroy_process_group()

if __name__ == "__main__":
    main()

Writing /kaggle/working/cjpe_m1/train_m1_cjpe.py


In [14]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/cjpe_m1/train_m1_cjpe.py

W1021 15:59:06.133000 132 torch/distributed/run.py:852] 
W1021 15:59:06.133000 132 torch/distributed/run.py:852] *****************************************
W1021 15:59:06.133000 132 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1021 15:59:06.133000 132 torch/distributed/run.py:852] *****************************************
[W1021 15:59:06.944198919 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1021 15:59:09.257725530 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1021 15:59:09.346017819 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Epoch 1/3: 100%|███████████████| 8077/8077 [22:50<00:00,  5.89it/s, loss=0.4546]
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logg

In [21]:

import os
import torch
from tqdm.auto import tqdm

OUT_DIR = "/kaggle/working/cjpe_test_tokens"
os.makedirs(OUT_DIR, exist_ok=True)

MAX_LEN = min(int(MAX_LEN), int(BLOCK_SIZE))
test_ids = []
test_case_ids = []

for row in tqdm(test_data, desc="Tokenizing CJPE test"):
    ids = tokenizer.encode_text(row["text"], add_eos=True)
    ids = ids[:MAX_LEN]

    if len(ids) == 0:
        ids = [0]

    test_ids.append(torch.tensor(ids, dtype=torch.long))
    test_case_ids.append(str(row["id"]))

torch.save({
    "ids": test_ids,
    "case_ids": test_case_ids
}, os.path.join(OUT_DIR, "test_m1.pt"))

print("Saved:", os.path.join(OUT_DIR, "test_m1.pt"))
print("Cases:", len(test_ids))
print("Max token length:", max(len(x) for x in test_ids))
print("First case ID:", test_case_ids[0])


Tokenizing CJPE test:   0%|          | 0/1517 [00:00<?, ?it/s]

Saved: /kaggle/working/cjpe_test_tokens/test_m1.pt
Cases: 1517
Max token length: 1024
First case ID: 1986_397


In [25]:

import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm
import legal_m1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_PATH = "/kaggle/working/cjpe_m1/checkpoints/best.pt"
TEST_PATH = "/kaggle/working/cjpe_test_tokens/test_m1.pt"
OUT_PATH = "/kaggle/working/submission_m1_p4.json"

assert os.path.exists(CKPT_PATH), f"Missing trained checkpoint: {CKPT_PATH}"
assert os.path.exists(TEST_PATH), f"Missing test tokens: {TEST_PATH}"

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
print("Checkpoint keys:", list(ckpt.keys()))
print("Epoch:", ckpt.get("epoch"), "Macro-F1:", ckpt.get("macro_f1"))

cfg = ckpt["config"]
backbone = legal_m1.GPTModel(
    cfg["vocab_size"], cfg["d_model"], cfg["n_layers"],
    cfg["n_heads"], cfg["ffn_dim"], cfg["dropout"]
)

# Match the training script's classifier architecture.
class M1ForCJPE(nn.Module):
    def __init__(self, backbone, hidden_size):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, lengths):
        x = self.backbone.token_embedding(input_ids)
        for block in self.backbone.blocks:
            x = block(x)
        if hasattr(self.backbone, "norm_f"):
            x = self.backbone.norm_f(x)

        positions = torch.arange(input_ids.size(1), device=input_ids.device)
        mask = (positions.unsqueeze(0) < lengths.unsqueeze(1)).unsqueeze(-1)
        mask = mask.to(x.dtype)
        pooled = (x * mask).sum(dim=1) / lengths.unsqueeze(1).clamp(min=1)
        return self.classifier(self.dropout(pooled))

model = M1ForCJPE(backbone, cfg["d_model"])
model.load_state_dict(ckpt["model"], strict=True)
model.to(device).eval()

test = torch.load(TEST_PATH, map_location="cpu", weights_only=False)
assert len(test["ids"]) == len(test["case_ids"]) == 1517

class TestDataset(Dataset):
    def __len__(self):
        return len(test["ids"])

    def __getitem__(self, idx):
        return test["ids"][idx], test["case_ids"][idx]

def collate_test(batch):
    ids = [item[0] for item in batch]
    lengths = torch.tensor([len(x) for x in ids], dtype=torch.long)
    ids = pad_sequence(ids, batch_first=True, padding_value=0)
    return ids, lengths, [item[1] for item in batch]

loader = DataLoader(
    TestDataset(), batch_size=4, shuffle=False, collate_fn=collate_test
)

predictions = []
with torch.no_grad():
    for input_ids, lengths, case_ids in tqdm(loader, desc="M1 predictions"):
        logits = model(input_ids.to(device), lengths.to(device))
        preds = logits.argmax(dim=1).cpu().tolist()
        for case_id, pred in zip(case_ids, preds):
            predictions.append({"id": case_id, "pred": int(pred)})

assert len(predictions) == 1517
assert all(set(row.keys()) == {"id", "pred"} and row["pred"] in [0, 1]
           for row in predictions)

with open(OUT_PATH, "w") as f:
    json.dump(predictions, f)

print("Saved:", OUT_PATH)
print("Total predictions:", len(predictions))
print("Class counts:", {
    0: sum(x["pred"] == 0 for x in predictions),
    1: sum(x["pred"] == 1 for x in predictions)
})
print("Preview:", predictions[:3])


Checkpoint keys: ['model', 'config', 'epoch', 'macro_f1']
Epoch: 2 Macro-F1: 0.5665121610845295


M1 predictions:   0%|          | 0/380 [00:00<?, ?it/s]

Saved: /kaggle/working/submission_m1_p4.json
Total predictions: 1517
Class counts: {0: 530, 1: 987}
Preview: [{'id': '1986_397', 'pred': 1}, {'id': '1977_183', 'pred': 0}, {'id': '1993_98', 'pred': 1}]


In [27]:
import os
import json
import zipfile
import glob
import shutil

WORK = "/kaggle/working"

# Known M1 submission and possible names for M2/M3.
candidates = {
    "M1": [
        f"{WORK}/submission_m1_p4.json",
    ]
}

for model_name, paths in candidates.items():
    existing = [p for p in paths if os.path.isfile(p)]

    if not existing:
        print(f"{model_name}: prediction JSON not found yet.")
        continue

    source = existing[0]

    with open(source, "r") as f:
        predictions = json.load(f)

    assert isinstance(predictions, list), (
        f"{model_name}: expected a JSON list"
    )
    assert all(
        isinstance(row, dict)
        and "id" in row
        and "pred" in row
        and row["pred"] in (0, 1)
        for row in predictions
    ), f"{model_name}: invalid prediction format"

    zip_path = f"{WORK}/submission_{model_name.lower()}_p4.zip"

    with zipfile.ZipFile(
        zip_path, "w", zipfile.ZIP_DEFLATED
    ) as zf:
        zf.write(source, arcname="prediction.json")

    print(
        f"{model_name}: {len(predictions)} predictions -> "
        f"{zip_path}"
    )

M1: 1517 predictions -> /kaggle/working/submission_m1_p4.zip


Phase 5

In [3]:

import os
import json
from huggingface_hub import HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
REPO_ID = "Exploration-Lab/CS-781-Capstone"

api = HfApi(token=HF_TOKEN)

# Confirm the exact Phase 5 filenames in the dataset repository.
all_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
)

phase5_files = [
    path for path in all_files
    if "Phase_5_LSI" in path
]

print("Phase 5 files:")
for path in phase5_files:
    print(path)


Phase 5 files:
Phase_5_LSI/Test/test.json
Phase_5_LSI/Train/dev.json
Phase_5_LSI/Train/train.json


In [7]:

# Download/access the official Phase 5 splits.
# Adjust filenames only if the listing above shows different paths.

TRAIN_PATH = hf_hub_download(
    repo_id=REPO_ID,
    filename="Phase_5_LSI/Train/train.json",
    repo_type="dataset",
    token=HF_TOKEN
)

DEV_PATH = hf_hub_download(
    repo_id=REPO_ID,
    filename="Phase_5_LSI/Train/dev.json",
    repo_type="dataset",
    token=HF_TOKEN
)

TEST_PATH = hf_hub_download(
    repo_id=REPO_ID,
    filename="Phase_5_LSI/Test/test.json",
    repo_type="dataset",
    token=HF_TOKEN
)

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(DEV_PATH, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Train documents:", len(train_data))
print("Development documents:", len(dev_data))
print("Test documents:", len(test_data))


Train documents: 42750
Development documents: 10181
Test documents: 13019


In [9]:
import pickle
import os
TOKENIZER_PATH = "/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl"

with open(TOKENIZER_PATH, "rb") as f:
    bpe = pickle.load(f)

print("Tokenizer loaded:", type(bpe).__name__)
print("Vocabulary size:", bpe.vocab_size)
print("EOS ID:", bpe.eos_id)
print("Test encoding:", bpe.encode_text("A legal case", add_eos=True))

Tokenizer loaded: ByteLevelBPE
Vocabulary size: 32000
EOS ID: 0
Test encoding: [66, 33, 1320, 33, 422, 0]


In [14]:
import torch
import legal_m1

cfg = ckpt["config"]

model = legal_m1.GPTModel(
    cfg["vocab_size"],
    cfg["d_model"],
    cfg["n_layers"],
    cfg["n_heads"],
    cfg["ffn_dim"],
    cfg["dropout"]
)

result = model.load_state_dict(ckpt["model"], strict=True)

print("Phase 1 weights loaded successfully.")
print("Missing keys:", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)
print("Total parameters:", sum(p.numel() for p in model.parameters()))

Phase 1 weights loaded successfully.
Missing keys: []
Unexpected keys: []
Total parameters: 109612800


In [15]:

import os
import json
import torch
from tqdm.auto import tqdm

WORK = "/kaggle/working/lsi_m1"
os.makedirs(WORK, exist_ok=True)

# Use whichever variable holds your already-loaded tokenizer.
tok = globals().get("bpe", globals().get("tokenizer"))
assert tok is not None, "Load the saved Phase 1 tokenizer first."

CHUNK_SIZE = 1024
OVERLAP = 128
STRIDE = CHUNK_SIZE - OVERLAP

# The output label space comes from training labels only.
LABEL_IDS = sorted(set(
    int(label)
    for row in train_data
    for label in row["labels"]
))

with open(WORK + "/label_ids.json", "w") as f:
    json.dump(LABEL_IDS, f)

def encode_documents(records, is_test=False):
    encoded = []

    for row in tqdm(records, desc="Encoding documents"):
        token_ids = tok.encode_text(row["text"], add_eos=True)

        # Preserve at least one token for an empty document.
        if not token_ids:
            token_ids = [tok.eos_id]

        chunks = []
        for start in range(0, len(token_ids), STRIDE):
            chunk = token_ids[start:start + CHUNK_SIZE]
            if chunk:
                chunks.append(torch.tensor(chunk, dtype=torch.int32))

            if start + CHUNK_SIZE >= len(token_ids):
                break

        item = {
            "id": str(row["id"]),
            "chunks": chunks
        }

        if not is_test:
            item["labels"] = [
                int(label) for label in row["labels"]
            ]

        encoded.append(item)
    return encoded

train_encoded = encode_documents(train_data)
dev_encoded = encode_documents(dev_data)
test_encoded = encode_documents(test_data, is_test=True)

torch.save(train_encoded, WORK + "/train.pt")
torch.save(dev_encoded, WORK + "/dev.pt")
torch.save(test_encoded, WORK + "/test.pt")

print("Saved chunked documents.")
print("Train/dev/test:",
      len(train_encoded), len(dev_encoded), len(test_encoded))
print("Chunks in first training document:",
      len(train_encoded[0]["chunks"]))


Encoding documents:   0%|          | 0/42750 [00:00<?, ?it/s]

Encoding documents:   0%|          | 0/10181 [00:00<?, ?it/s]

Encoding documents:   0%|          | 0/13019 [00:00<?, ?it/s]

Saved chunked documents.
Train/dev/test: 42750 10181 13019
Chunks in first training document: 4


In [22]:

%%writefile /kaggle/working/train_lsi_m1.py
import os
import sys
import json
import random
from datetime import timedelta
import numpy as np
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist

from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.nn.utils.rnn import pad_sequence
from torch.amp import autocast, GradScaler
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm

sys.path.insert(0, "/kaggle/working")
import legal_m1

WORK = "/kaggle/working/lsi_m1"
CKPT = "/kaggle/working/m1_750_checkpoints/latest.pt"
OUT = WORK + "/checkpoints"
os.makedirs(OUT, exist_ok=True)

MAX_CHUNK_LEN = 1024
OVERLAP = 128
MAX_TRAIN_CHUNKS = 2
CHUNK_BATCH = 4          # Larger inference microbatches reduce Python/GPU launch overhead
DOC_BATCH = 2
EVAL_DOC_BATCH = 4       # Evaluate several documents per DataLoader batch
ACCUM = 4
EPOCHS = 2
LR = 2e-4
LORMA_RANK = 8
LORMA_ALPHA = 16
SEED = 42


class DocumentDataset(Dataset):
    def __init__(self, path, label_to_idx, training):
        self.docs = torch.load(
            path, map_location="cpu", weights_only=False
        )
        self.label_to_idx = label_to_idx
        self.training = training

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        doc = self.docs[idx]
        chunks = doc["chunks"]

        # Sample different chunks over training accesses.
        if self.training and len(chunks) > MAX_TRAIN_CHUNKS:
            chosen = sorted(random.sample(
                range(len(chunks)), MAX_TRAIN_CHUNKS
            ))
            chunks = [chunks[i] for i in chosen]

        target = torch.zeros(len(self.label_to_idx))
        if self.training or "labels" in doc:
            for label in doc.get("labels", []):
                if label in self.label_to_idx:
                    target[self.label_to_idx[label]] = 1.0

        return doc["id"], [x.long() for x in chunks], target


def collate_docs(batch):
    doc_ids = []
    flat_chunks = []
    chunk_to_doc = []
    targets = []

    for doc_index, (doc_id, chunks, target) in enumerate(batch):
        doc_ids.append(doc_id)
        targets.append(target)

        for chunk in chunks:
            flat_chunks.append(chunk[:MAX_CHUNK_LEN])
            chunk_to_doc.append(doc_index)

    lengths = torch.tensor([len(x) for x in flat_chunks])
    ids = pad_sequence(
        flat_chunks, batch_first=True, padding_value=0
    )
    mask = (
        torch.arange(ids.size(1))[None, :] < lengths[:, None]
    ).long()

    return (
        ids, mask, doc_ids,
        torch.tensor(chunk_to_doc, dtype=torch.long),
        torch.stack(targets)
    )


# Validation runs only on rank 0. The process-group timeout is extended because
# rank 1 waits at a barrier while rank 0 evaluates all dev documents.
# LoRMA+:
# W_eff = (I + alpha/r * B A) W0.
# Compute the multiplicative transformation on the layer output
# to avoid materializing the full effective weight matrix.
class LoRMALinear(nn.Module):
    def __init__(self, base, rank=LORMA_RANK, alpha=LORMA_ALPHA):
        super().__init__()
        self.base = base
        self.scale = alpha / rank

        out_dim = base.out_features
        self.A = nn.Parameter(torch.empty(rank, out_dim))
        self.B = nn.Parameter(torch.zeros(out_dim, rank))
        nn.init.kaiming_uniform_(self.A, a=np.sqrt(5))

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        y = self.base(x)
        update = F.linear(F.linear(y, self.A), self.B)
        return y + self.scale * update


def apply_lorma_plus(backbone):
    for p in backbone.parameters():
        p.requires_grad = False

    for block in backbone.blocks:
        block.attn.qkv = LoRMALinear(block.attn.qkv)
        block.attn.out_proj = LoRMALinear(block.attn.out_proj)
        block.ffn.fc1 = LoRMALinear(block.ffn.fc1)
        block.ffn.fc2 = LoRMALinear(block.ffn.fc2)

    return backbone


class M1ForLSI(nn.Module):
    def __init__(self, backbone, num_labels):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, num_labels)

    def encode_chunks(self, ids, mask):
        x = self.backbone.dropout(
            self.backbone.token_embedding(ids)
        )

        for block in self.backbone.blocks:
            x = block(x)

        x = self.backbone.norm(x)

        m = mask.unsqueeze(-1).to(x.dtype)
        pooled = (x * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)
        return pooled

    def forward(self, ids, mask, chunk_to_doc, num_docs):
        chunk_vectors = self.encode_chunks(ids, mask)

        # Average chunk representations into one representation
        # per original document, then compute one document-level loss.
        doc_vectors = chunk_vectors.new_zeros(
            num_docs, chunk_vectors.size(-1)
        )
        doc_vectors.index_add_(0, chunk_to_doc, chunk_vectors)

        counts = torch.bincount(
            chunk_to_doc, minlength=num_docs
        ).to(chunk_vectors.dtype).unsqueeze(1)

        doc_vectors = doc_vectors / counts.clamp(min=1)
        return self.classifier(self.dropout(doc_vectors))


def evaluate(model, loader, device, label_ids, rank):
    model.eval()
    all_probs = []
    all_targets = []
    all_doc_ids = []

    with torch.inference_mode():
        iterator = tqdm(
            loader, desc="Dev evaluation", disable=(rank != 0)
        )

        for ids, mask, doc_ids, chunk_to_doc, targets in iterator:
            ids = ids.to(device)
            mask = mask.to(device)
            chunk_to_doc = chunk_to_doc.to(device)

            # Evaluation batches may contain many chunks per document.
            # Split chunks into small forward batches, then aggregate
            # their vectors before predicting the document's labels.
            chunk_vectors = []

            for start in range(0, ids.size(0), CHUNK_BATCH):
                end = min(start + CHUNK_BATCH, ids.size(0))

                with autocast("cuda", dtype=torch.float16):
                    vec = model.module.backbone.token_embedding(
                        ids[start:end]
                    )
                    vec = model.module.backbone.dropout(vec)

                    for block in model.module.backbone.blocks:
                        vec = block(vec)

                    vec = model.module.backbone.norm(vec)
                    m = mask[start:end].unsqueeze(-1).to(vec.dtype)
                    pooled = (vec * m).sum(dim=1) / m.sum(
                        dim=1
                    ).clamp(min=1)

                chunk_vectors.append(pooled.float())

            chunk_vectors = torch.cat(chunk_vectors, dim=0)
            doc_vectors = chunk_vectors.new_zeros(
                len(doc_ids), chunk_vectors.size(-1)
            )
            doc_vectors.index_add_(0, chunk_to_doc, chunk_vectors)

            counts = torch.bincount(
                chunk_to_doc, minlength=len(doc_ids)
            ).to(doc_vectors.dtype).unsqueeze(1)

            doc_vectors = doc_vectors / counts.clamp(min=1)
            logits = model.module.classifier(
                model.module.dropout(doc_vectors)
            )

            probs = torch.sigmoid(logits.float()).cpu().numpy()
            all_probs.append(probs)
            all_targets.append(targets.numpy())
            all_doc_ids.extend(doc_ids)

    return (
        np.concatenate(all_probs),
        np.concatenate(all_targets),
        all_doc_ids
    )


def main():
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = int(os.environ["RANK"])
    world = int(os.environ["WORLD_SIZE"])

    torch.cuda.set_device(local_rank)
    # Rank 0 evaluates the full dev set while rank 1 waits at the next
    # barrier. Evaluation can exceed the default 10-minute NCCL timeout.
    dist.init_process_group("nccl", timeout=timedelta(hours=3))
    device = torch.device("cuda", local_rank)

    random.seed(SEED + rank)
    np.random.seed(SEED + rank)
    torch.manual_seed(SEED + rank)

    with open(WORK + "/label_ids.json") as f:
        label_ids = [int(x) for x in json.load(f)]

    label_to_idx = {label: i for i, label in enumerate(label_ids)}

    train_ds = DocumentDataset(
        WORK + "/train.pt", label_to_idx, training=True
    )
    dev_ds = DocumentDataset(
        WORK + "/dev.pt", label_to_idx, training=False
    )

    train_sampler = DistributedSampler(
        train_ds, num_replicas=world, rank=rank,
        shuffle=True, seed=SEED
    )

    train_loader = DataLoader(
        train_ds, batch_size=DOC_BATCH,
        sampler=train_sampler, collate_fn=collate_docs,
        num_workers=0, pin_memory=True
    )

    # Every rank participates in training; rank 0 evaluates.
    dev_loader = DataLoader(
        dev_ds, batch_size=EVAL_DOC_BATCH, shuffle=False,
        collate_fn=collate_docs, num_workers=0
    )

    checkpoint = torch.load(
        CKPT, map_location="cpu", weights_only=False
    )
    cfg = checkpoint["config"]

    backbone = legal_m1.GPTModel(
        cfg["vocab_size"], cfg["d_model"], cfg["n_layers"],
        cfg["n_heads"], cfg["ffn_dim"], cfg["dropout"]
    )
    backbone.load_state_dict(checkpoint["model"], strict=True)
    backbone = apply_lorma_plus(backbone)

    model = M1ForLSI(backbone, len(label_ids)).to(device)
    model = DDP(model, device_ids=[local_rank])

    trainable = [
        p for p in model.parameters() if p.requires_grad
    ]
    optimizer = torch.optim.AdamW(
        trainable, lr=LR, weight_decay=0.01
    )
    scaler = GradScaler("cuda")

    if rank == 0:
        print("Train documents:", len(train_ds))
        print("Development documents:", len(dev_ds))
        print("Statute labels:", len(label_ids))
        print("Trainable parameters:",
              sum(p.numel() for p in trainable))

    best_f1 = -1.0
    start_epoch = 0

    # Resume from the most recent epoch-training snapshot if present.
    # A snapshot is written before validation so a long validation pass
    # cannot destroy the completed training epoch.
    resume_path = OUT + "/latest.pt"
    if os.path.exists(resume_path):
        saved = torch.load(resume_path, map_location="cpu", weights_only=False)
        model.module.load_state_dict(saved["model"], strict=True)
        optimizer.load_state_dict(saved["optimizer"])
        if "scaler" in saved:
            scaler.load_state_dict(saved["scaler"])
        start_epoch = int(saved.get("epoch", 0))
        best_f1 = float(saved.get("best_f1", -1.0))
        if rank == 0:
            print(f"Resuming from latest checkpoint; next training epoch: {start_epoch + 1}", flush=True)

    for epoch in range(start_epoch, EPOCHS):
        train_sampler.set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss_total = 0.0

        iterator = tqdm(
            train_loader,
            desc="Epoch " + str(epoch + 1),
            disable=(rank != 0)
        )

        for step, (ids, mask, doc_ids, chunk_to_doc, targets) in enumerate(iterator):
            ids = ids.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            chunk_to_doc = chunk_to_doc.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            sync_now = (
                (step + 1) % ACCUM == 0
                or (step + 1) == len(train_loader)
            )

            context = (
                model.no_sync() if not sync_now
                else torch.enable_grad()
            )

            with context:
                with autocast("cuda", dtype=torch.float16):
                    logits = model(
                        ids, mask, chunk_to_doc, len(doc_ids)
                    )
                    loss = F.binary_cross_entropy_with_logits(
                        logits, targets
                    ) / ACCUM

                scaler.scale(loss).backward()

            loss_total += loss.item() * ACCUM

            if sync_now:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            if rank == 0:
                iterator.set_postfix(
                    loss=f"{loss.item() * ACCUM:.4f}"
                )

        # Save immediately after training, BEFORE the expensive dev pass.
        # If evaluation crashes, the trained backbone/adapters are retained.
        if rank == 0:
            torch.save({
                "model": model.module.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(),
                "config": cfg,
                "label_ids": label_ids,
                "epoch": epoch + 1,
                "best_f1": best_f1,
            }, resume_path)
        dist.barrier()

        if rank == 0:
            probs, targets, _ = evaluate(
                model, dev_loader, device, label_ids, rank
            )

            best_threshold = 0.5
            epoch_f1 = -1.0

            for threshold in [
                0.10, 0.15, 0.20, 0.25, 0.30,
                0.35, 0.40, 0.45, 0.50
            ]:
                preds = (probs >= threshold).astype(np.int32)
                score = f1_score(
                    targets, preds, average="macro",
                    zero_division=0
                )

                if score > epoch_f1:
                    epoch_f1 = score
                    best_threshold = threshold

            preds = (probs >= best_threshold).astype(np.int32)
            micro_f1 = f1_score(
                targets, preds, average="micro", zero_division=0
            )
            subset = accuracy_score(targets, preds)

            print(
                f"Epoch {epoch + 1} | "
                f"Train loss {loss_total / len(train_loader):.4f} | "
                f"Macro-F1 {epoch_f1:.5f} | "
                f"Micro-F1 {micro_f1:.5f} | "
                f"Subset accuracy {subset:.5f} | "
                f"Threshold {best_threshold:.2f}",
                flush=True
            )

            if epoch_f1 > best_f1:
                best_f1 = epoch_f1
                torch.save({
                    "model": model.module.state_dict(),
                    "config": cfg,
                    "label_ids": label_ids,
                    "threshold": best_threshold,
                    "epoch": epoch + 1,
                    "macro_f1": epoch_f1,
                    "micro_f1": micro_f1,
                    "subset_accuracy": subset,
                }, OUT + "/best.pt")

            # Refresh latest checkpoint with the best score after evaluation.
            latest = torch.load(resume_path, map_location="cpu", weights_only=False)
            latest["best_f1"] = best_f1
            latest["threshold"] = best_threshold
            latest["macro_f1"] = epoch_f1
            latest["micro_f1"] = micro_f1
            latest["subset_accuracy"] = subset
            torch.save(latest, resume_path)

        dist.barrier()

    dist.destroy_process_group()


if __name__ == "__main__":
    main()


Overwriting /kaggle/working/train_lsi_m1.py


In [ ]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_lsi_m1.py

W1028 14:03:40.722000 255 torch/distributed/run.py:852] 
W1028 14:03:40.722000 255 torch/distributed/run.py:852] *****************************************
W1028 14:03:40.722000 255 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1028 14:03:40.722000 255 torch/distributed/run.py:852] *****************************************
[W1028 14:03:41.446335679 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1028 14:03:44.757167021 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1028 14:03:44.773180310 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Train documents: 42750
Development documents: 10181
Statute labels: 100
Trainable parameters: 1404004
Epoch 1: 100%|███████████████| 10688/10688 [35

Phase 6

In [2]:
import os
import json
import random
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILE = "Phase_6_Summ/Train/train.json"
TEST_FILE = "Phase_6_Summ/Test/test.json"

SEED = 42
random.seed(SEED)


train_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Training dataset:",train_path)

test_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

Phase_6_Summ/Train/train.json:   0%|          | 0.00/211M [00:00<?, ?B/s]

Training dataset: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ/Train/train.json


test.json: 0.00B [00:00, ?B/s]

In [3]:
print("Test dataset:",test_path)
with open(train_path,"r",encoding="utf-8") as f:
    train_data = json.load(f)

with open(test_path,"r",encoding="utf-8") as f:
    test_data = json.load(f)
print("Training examples:",len(train_data))
print("Test examples:",len(test_data))
print("ID:",train_data[0]["id"])
print("Document:",train_data[0]["document"][:500])
print("Summary:",train_data[0]["summary"][:500])

print("ID:",test_data[0]["id"])
print("Document:",test_data[0]["document"][:500])

Test dataset: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ/Test/test.json
Training examples: 7030
Test examples: 100
ID: 6712
Document: IVIL Appeal No.768 (NT) of 1977 etc. From the judgment Order dated 9.10.1975 of the Madhya Pradesh High Court in M.C.C. No.144 of 1966. V.A. Bobde, B.R.Aggarwala and U.A.Rana for the Appellant. S.V.Deshpande and S.K.Agnihotri for the Respondents. The Judgment of the Court was delivered by RANGANATH MISHRA, CJ. These are appeals by special leave 252 and are directed against the separate decisions of the Madhya Pradesh High Court in references under the Madhya Pradesh Sales Tax Act. Civil Appeal N
Summary: The appellant, a manufacturer of cement, entered into an agreement with the Cement Manufacturing Company of India Limited, for sale of cement. Under the agreement, the appellant was to sell its cement only through the marketing company, and certain sums would be 

In [6]:
import numpy as np
from tqdm.auto import tqdm

train_doc_lens = []
train_sum_lens = []
test_doc_lens = []

for row in tqdm(train_data,desc="Tokenizing training documents"):
    train_doc_lens.append(len(bpe.encode_text(row["document"],add_eos=False)))
    train_sum_lens.append(len(bpe.encode_text(row["summary"],add_eos=False)))

for row in tqdm(test_data,desc="Tokenizing test documents"):
    test_doc_lens.append(len(bpe.encode_text(row["document"],add_eos=False)))

print("Document token lengths:")
print("Train:",np.percentile(train_doc_lens,[50,75,90,95,99]))
print("Test :",np.percentile(test_doc_lens,[50,75,90,95,99]))

print("\nSummary token lengths:")
print(np.percentile(train_sum_lens,[50,75,90,95,99]))

Tokenizing training documents:   0%|          | 0/7030 [00:00<?, ?it/s]

Tokenizing test documents:   0%|          | 0/100 [00:00<?, ?it/s]

Document token lengths:
Train: [ 6545.5  10369.   16461.   23763.95 50970.97]
Test : [ 6621.5 11523.  18098.8 28812.3 61380.9]

Summary token lengths:
[1338.   2023.5  3154.1  4249.65 9005.3 ]


In [9]:
import re
import math
from collections import Counter
from tqdm.auto import tqdm

MAX_SEQ_LEN = 2048
MAX_SOURCE_LEN = 1400
MAX_TARGET_LEN = 500
CHUNK_OVERLAP = 128

PROMPT = (
    "Summarize the following legal judgment segment. "
    "Include only information supported by the text.\n\n"
    "JUDGMENT:\n" )

EOS_ID = tokenizer.stoi["<eos>"]
PROMPT_IDS = tokenizer.encode_text(PROMPT,add_eos=False)
assert len(PROMPT_IDS) + MAX_SOURCE_LEN + MAX_TARGET_LEN + 1 <= MAX_SEQ_LEN
def split_sentences(text):
    text = re.sub(r'\s+', ' ', str(text)).strip()

    if not text:
        return []

    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

def encode_without_eos(text):
    return list(tokenizer.encode_text(text,add_eos=False))

def make_chunks(document):
    sentences = split_sentences(document)
    sentence_ids = []
    for sentence in sentences:
        ids = encode_without_eos(sentence)
        if len(ids) > 0:
            sentence_ids.append(ids)

    chunks = []
    start = 0

    while start < len(sentence_ids):
        chunk_ids = []
        end = start
        # Add complete sentences while within the source limit.
        while end < len(sentence_ids):
            next_len = len(sentence_ids[end])

            if len(chunk_ids) + next_len > MAX_SOURCE_LEN:
                break

            chunk_ids.extend(sentence_ids[end])
            end += 1

        # Handle a single exceptionally long sentence.
        if end == start:
            long_sentence = sentence_ids[start]
            chunk_ids = long_sentence[:MAX_SOURCE_LEN]
            remaining = long_sentence[MAX_SOURCE_LEN:]
            if remaining:
                sentence_ids.insert(start + 1, remaining)

            end = start + 1

        if chunk_ids:
            chunks.append(chunk_ids)

        if end >= len(sentence_ids):
            break

        # Keep whole trailing sentences as overlap where possible.
        overlap_start = end
        overlap_len = 0
        while overlap_start > start:
            candidate_len = len(sentence_ids[overlap_start - 1])
            if overlap_len + candidate_len > CHUNK_OVERLAP:
                break

            overlap_len += candidate_len
            overlap_start -= 1
# If the final sentence itself is longer than the overlap budget, retain only its final CHUNK_OVERLAP tokens.
        if overlap_start == end:
            last_sentence = sentence_ids[end - 1]
            if len(last_sentence) > CHUNK_OVERLAP:
                sentence_ids[end - 1] = last_sentence[-CHUNK_OVERLAP:]
                overlap_start = end - 1
            else:
                overlap_start = end - 1
        start = max(overlap_start, start + 1)

    return chunks

def make_summary_sentences(summary):
    sentences = split_sentences(summary)
    result = []
    for sentence in sentences:
        ids = encode_without_eos(sentence)
        if not ids:
            continue
        # Split unusually long sentences without dropping tokens.
        for i in range(0,len(ids),MAX_TARGET_LEN):
            result.append(ids[i:i + MAX_TARGET_LEN])

    return result

def align_summary_to_chunks(chunks, summary):
    summary_sentences = make_summary_sentences(summary)
    if not chunks or not summary_sentences:
        return []

    # Document-frequency statistics over chunks.
    chunk_sets = [set(chunk) for chunk in chunks]
    df = Counter()
    for token_set in chunk_sets:
        df.update(token_set)

    num_chunks = len(chunks)
    idf = {}
    for token,count in df.items():
        idf[token] = math.log((num_chunks + 1) / (count + 1)) + 1.0

    aligned = [[] for _ in chunks]
    for sentence_ids in summary_sentences:
        sentence_tokens = set(sentence_ids)
        total_weight = sum(idf.get(t,1.0) for t in sentence_tokens)
        best_chunk = 0
        best_score = -1.0
        for chunk_idx,chunk_tokens in enumerate(chunk_sets):
            common = sentence_tokens.intersection(chunk_tokens)
            score = sum(idf.get(t,1.0) for t in common)
            score /= max(total_weight,1e-8)
            if score > best_score:
                best_score = score
                best_chunk = chunk_idx

        aligned[best_chunk].append(sentence_ids)

    examples = []
    for chunk_idx, sentence_groups in enumerate(aligned):
        current_target = []

        for sentence_ids in sentence_groups:
            if len(current_target) + len(sentence_ids) > MAX_TARGET_LEN:
                if current_target:
                    examples.append({
                        "source_ids": chunks[chunk_idx],
                        "target_ids": current_target + [EOS_ID],
                        "chunk_idx": chunk_idx
                    })

                current_target = []

            current_target.extend(sentence_ids)

        if current_target:
            examples.append({"source_ids":chunks[chunk_idx],"target_ids":current_target+[EOS_ID],
                "chunk_idx": chunk_idx})

    return examples

train_examples = []
train_chunk_counts = []
for row in tqdm(train_data,desc="Preparing training chunks"):
    document = row["document"]
    summary = row["summary"]
    chunks = make_chunks(document)
    examples = align_summary_to_chunks(chunks,summary)
    for example in examples:
        example["id"] = row["id"]
        train_examples.append(example)

    train_chunk_counts.append(len(chunks))

test_chunks = []
for row in tqdm(test_data,desc="Preparing test chunks"):
    chunks = make_chunks(row["document"])
    for chunk_idx,chunk_ids in enumerate(chunks):
        test_chunks.append({"id": row["id"],"chunk_idx": chunk_idx,"source_ids": chunk_ids})

print("Training examples:",len(train_examples))
print("Test chunks:",len(test_chunks))
print("Average chunks per training document:",
      sum(train_chunk_counts) / max(len(train_chunk_counts),1))

if train_examples:
    print("Maximum source length:",max(len(x["source_ids"]) for x in train_examples))
    print("Maximum target length:", max(len(x["target_ids"]) for x in train_examples))
    assert all(len(x["source_ids"]) <= MAX_SOURCE_LEN for x in train_examples)
    assert all(len(x["target_ids"]) <= MAX_TARGET_LEN + 1 for x in train_examples)
    assert all(
        len(PROMPT_IDS)+len(x["source_ids"])+len(x["target_ids"]) <= MAX_SEQ_LEN for x in train_examples
    )

print("Chunking and alignment completed.")

Preparing training chunks:   0%|          | 0/7030 [00:00<?, ?it/s]

Preparing test chunks:   0%|          | 0/100 [00:00<?, ?it/s]

Training examples: 50530
Test chunks: 820
Average chunks per training document: 7.517354196301564
Maximum source length: 1400
Maximum target length: 501
Chunking and alignment completed.


In [10]:
import random
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SEED = 42
VAL_RATIO = 0.10

random.seed(SEED)

doc_ids = list(set(x["id"] for x in train_examples))
random.shuffle(doc_ids)

num_val = max(1,int(len(doc_ids) * VAL_RATIO))
val_ids = set(doc_ids[:num_val])
train_ids = set(doc_ids[num_val:])

train_rows = [x for x in train_examples if x["id"] in train_ids]
val_rows = [x for x in train_examples if x["id"] in val_ids]

print("Training documents:",len(train_ids))
print("Validation documents:",len(val_ids))
print("Training examples:",len(train_rows))
print("Validation examples:",len(val_rows))

assert train_ids.isdisjoint(val_ids)

SUMMARY_PREFIX_IDS = tokenizer.encode_text("\n\nSUMMARY:\n",add_eos=False)

assert len(PROMPT_IDS) + MAX_SOURCE_LEN + len(SUMMARY_PREFIX_IDS) + MAX_TARGET_LEN + 1 <= MAX_SEQ_LEN

class LegalSummaryDataset(Dataset):
    def __init__(self,examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self,idx):
        item = self.examples[idx]
        prefix_ids = PROMPT_IDS + item["source_ids"] + SUMMARY_PREFIX_IDS
        input_ids = prefix_ids + item["target_ids"]
        labels = [-100] * len(prefix_ids) + item["target_ids"]

        assert len(input_ids) == len(labels)
        assert len(input_ids) <= MAX_SEQ_LEN

        return torch.tensor(input_ids,dtype=torch.long),torch.tensor(labels,dtype=torch.long)


def collate_summary(batch):
    input_ids = [x[0] for x in batch]
    labels = [x[1] for x in batch]

    input_ids = pad_sequence(input_ids,batch_first=True,padding_value=EOS_ID)
    labels = pad_sequence(labels,batch_first=True,padding_value=-100)

    return input_ids,labels


train_dataset = LegalSummaryDataset(train_rows)
val_dataset = LegalSummaryDataset(val_rows)

print("Train dataset:",len(train_dataset))
print("Validation dataset:",len(val_dataset))

check_loader = DataLoader(train_dataset,batch_size=2,shuffle=False,collate_fn=collate_summary)
sample_x,sample_y = next(iter(check_loader))
print("Input shape:",sample_x.shape)
print("Label shape:",sample_y.shape)
print("Non-masked label tokens:",(sample_y != -100).sum().item())
assert sample_x.shape == sample_y.shape
assert sample_x.shape[1] <= MAX_SEQ_LEN
assert (sample_y != -100).any()

first_prefix_len = len(PROMPT_IDS) + len(train_rows[0]["source_ids"]) + len(SUMMARY_PREFIX_IDS)
assert (sample_y[0,:first_prefix_len] == -100).all()

Training documents: 6326
Validation documents: 702
Training examples: 45450
Validation examples: 5080
Train dataset: 45450
Validation dataset: 5080
Input shape: torch.Size([2, 1846])
Label shape: torch.Size([2, 1846])
Non-masked label tokens: 737


In [11]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

CHECKPOINT_PATH = "/kaggle/working/m1_750_checkpoints/latest.pt"

if not os.path.exists(CHECKPOINT_PATH):
    CHECKPOINT_PATH = "/kaggle/working/m1_checkpoints/best.pt"

assert os.path.exists(CHECKPOINT_PATH),"M1 checkpoint not found."

checkpoint = torch.load(CHECKPOINT_PATH,map_location="cpu",weights_only=False)
cfg = checkpoint["config"]

assert cfg["vocab_size"] == tokenizer.vocab_size
assert cfg["block_size"] == MAX_SEQ_LEN

backbone = GPTModel(
    cfg["vocab_size"],
    cfg["d_model"],
    cfg["n_layers"],
    cfg["n_heads"],
    cfg["ffn_dim"],
    cfg["dropout"]
)

backbone.load_state_dict(checkpoint["model"],strict=True)

LORMA_RANK = 8
LORMA_ALPHA = 16

class LoRMALinear(nn.Module):
    def __init__(self,base,rank=LORMA_RANK,alpha=LORMA_ALPHA):
        super().__init__()
        self.base = base
        self.scale = alpha / rank

        out_dim = base.out_features
        self.A = nn.Parameter(torch.empty(rank,out_dim))
        self.B = nn.Parameter(torch.zeros(out_dim,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self,x):
        y = self.base(x)
        update = F.linear(F.linear(y,self.A),self.B)
        return y + self.scale * update


def apply_lorma_plus(backbone):
    for p in backbone.parameters():
        p.requires_grad = False

    for block in backbone.blocks:
        block.attn.qkv = LoRMALinear(block.attn.qkv)
        block.attn.out_proj = LoRMALinear(block.attn.out_proj)
        block.ffn.fc1 = LoRMALinear(block.ffn.fc1)
        block.ffn.fc2 = LoRMALinear(block.ffn.fc2)

    return backbone


backbone = apply_lorma_plus(backbone).cuda()
trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in backbone.parameters())

print("Checkpoint:",CHECKPOINT_PATH)
print("Pretraining tokens:",checkpoint.get("tokens_seen","not recorded"))
print("Model parameters:",round(total_params / 1e6,2),"M")
print("Trainable LoRMA+ parameters:",trainable_params)
print("Trainable percentage:",round(100 * trainable_params / total_params,2),"%")
print("GPU count:",torch.cuda.device_count())

assert trainable_params > 0
assert all(
    torch.count_nonzero(block.attn.qkv.B).item() == 0
    and torch.count_nonzero(block.attn.out_proj.B).item() == 0
    and torch.count_nonzero(block.ffn.fc1.B).item() == 0
    and torch.count_nonzero(block.ffn.fc2.B).item() == 0
    for block in backbone.blocks
)


Checkpoint: /kaggle/working/m1_750_checkpoints/latest.pt
Pretraining tokens: 750000330
Model parameters: 110.94 M
Trainable LoRMA+ parameters: 1327104
Trainable percentage: 1.2 %
GPU count: 2


In [12]:
import os
import numpy as np
import torch
from tqdm.auto import tqdm

DATA_DIR = "/kaggle/working/phase6_m1_data"
os.makedirs(DATA_DIR,exist_ok=True)

def save_packed_data(examples,path):
    lengths = [
        len(PROMPT_IDS) + len(x["source_ids"]) +
        len(SUMMARY_PREFIX_IDS) + len(x["target_ids"])
        for x in examples
    ]

    offsets = np.zeros(len(examples) + 1,dtype=np.int64)
    offsets[1:] = np.cumsum(lengths)

    total_len = int(offsets[-1])
    input_ids = np.empty(total_len,dtype=np.int32)
    labels = np.full(total_len,-100,dtype=np.int32)

    for i,item in enumerate(tqdm(examples,desc="Packing data")):
        prefix = PROMPT_IDS + item["source_ids"] + SUMMARY_PREFIX_IDS
        sequence = prefix + item["target_ids"]

        start = offsets[i]
        end = offsets[i + 1]

        input_ids[start:end] = sequence
        labels[start + len(prefix):end] = item["target_ids"]

    torch.save({
        "input_ids":torch.from_numpy(input_ids),
        "labels":torch.from_numpy(labels),
        "offsets":torch.from_numpy(offsets)
    },path)

    print(path,"saved | Examples:",len(examples),
          "| Tokens:",total_len)

save_packed_data(train_rows,DATA_DIR + "/train.pt")
save_packed_data(val_rows,DATA_DIR + "/valid.pt")

Packing data:   0%|          | 0/45450 [00:00<?, ?it/s]

/kaggle/working/phase6_m1_data/train.pt saved | Examples: 45450 | Tokens: 71922931


Packing data:   0%|          | 0/5080 [00:00<?, ?it/s]

/kaggle/working/phase6_m1_data/valid.pt saved | Examples: 5080 | Tokens: 8047205


In [15]:
%%writefile /kaggle/working/train_phase6_m1.py

import os,math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from contextlib import nullcontext
from torch.utils.data import Dataset,DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.amp import autocast,GradScaler
from tqdm.auto import tqdm

from legal_m1 import GPTModel

BASE_CKPT="/kaggle/working/m1_750_checkpoints/latest.pt"
DATA_DIR="/kaggle/working/phase6_m1_data"
OUT="/kaggle/working/phase6_m1_lorma_checkpoints"
os.makedirs(OUT,exist_ok=True)

ACCUM=9
EPOCHS=2
LR=2e-4
WEIGHT_DECAY=0.01
GRAD_CLIP=1.0
LORMA_RANK=8
LORMA_ALPHA=16
SMOKE_TEST=False
SMOKE_STEPS=2


class PackedDataset(Dataset):
    def __init__(self,path):
        data=torch.load(path,map_location="cpu",weights_only=False)
        self.input_ids=data["input_ids"]
        self.labels=data["labels"]
        self.offsets=data["offsets"]

    def __len__(self):
        return len(self.offsets)-1

    def __getitem__(self,idx):
        start=int(self.offsets[idx])
        end=int(self.offsets[idx+1])
        return self.input_ids[start:end].long(),self.labels[start:end].long()


class LoRMALinear(nn.Module):
    def __init__(self,base,rank=LORMA_RANK,alpha=LORMA_ALPHA):
        super().__init__()
        self.base=base
        self.scale=alpha/rank
        out_dim=base.out_features
        self.A=nn.Parameter(torch.empty(rank,out_dim))
        self.B=nn.Parameter(torch.zeros(out_dim,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

        for p in self.base.parameters():
            p.requires_grad=False

    def forward(self,x):
        y=self.base(x)
        update=F.linear(F.linear(y,self.A),self.B)
        return y+self.scale*update


def apply_lorma_plus(model):
    for p in model.parameters():
        p.requires_grad=False

    for block in model.blocks:
        block.attn.qkv=LoRMALinear(block.attn.qkv)
        block.attn.out_proj=LoRMALinear(block.attn.out_proj)
        block.ffn.fc1=LoRMALinear(block.ffn.fc1)
        block.ffn.fc2=LoRMALinear(block.ffn.fc2)

    return model


def get_adapter_state(model):
    return {
        k:v.detach().cpu()
        for k,v in model.module.state_dict().items()
        if k.endswith(".A") or k.endswith(".B")
    }


def save_checkpoint(path,model,optimizer,scaler,epoch,best_val_loss,cfg):
    torch.save({
        "adapters":get_adapter_state(model),
        "optimizer":optimizer.state_dict(),
        "scaler":scaler.state_dict(),
        "epoch":epoch,
        "best_val_loss":best_val_loss,
        "config":cfg,
        "base_checkpoint":BASE_CKPT
    },path)


def get_loss(model,x,y,vocab_size):
    logits=model(x)
    shift_logits=logits[:,:-1,:].contiguous()
    shift_labels=y[:,1:].contiguous()

    loss=F.cross_entropy(
        shift_logits.view(-1,vocab_size),
        shift_labels.view(-1),
        ignore_index=-100
    )
    return loss,shift_labels


def main():
    dist.init_process_group("nccl")
    rank=int(os.environ["RANK"])
    local_rank=int(os.environ["LOCAL_RANK"])
    world=int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(local_rank)
    device=torch.device("cuda",local_rank)

    torch.manual_seed(42+rank)
    torch.backends.cuda.matmul.allow_tf32=True

    train_ds=PackedDataset(DATA_DIR+"/train.pt")
    val_ds=PackedDataset(DATA_DIR+"/valid.pt")

    train_sampler=DistributedSampler(
        train_ds,num_replicas=world,rank=rank,
        shuffle=True,drop_last=True,seed=42
    )
    val_sampler=DistributedSampler(
        val_ds,num_replicas=world,rank=rank,
        shuffle=False,drop_last=True
    )

    train_loader=DataLoader(
        train_ds,batch_size=1,sampler=train_sampler,
        num_workers=0,pin_memory=True
    )
    val_loader=DataLoader(
        val_ds,batch_size=1,sampler=val_sampler,
        num_workers=0,pin_memory=True
    )

    checkpoint=torch.load(BASE_CKPT,map_location="cpu",weights_only=False)
    cfg=checkpoint["config"]

    assert cfg["vocab_size"]==32000
    assert cfg["block_size"]==2048
    assert len(train_loader)%ACCUM==0

    model=GPTModel(
        cfg["vocab_size"],cfg["d_model"],cfg["n_layers"],
        cfg["n_heads"],cfg["ffn_dim"],cfg["dropout"]
    )
    model.load_state_dict(checkpoint["model"],strict=True)
    model=apply_lorma_plus(model).to(device)
    model=DDP(model,device_ids=[local_rank],output_device=local_rank)

    trainable=[p for p in model.parameters() if p.requires_grad]
    optimizer=torch.optim.AdamW(
        trainable,lr=LR,weight_decay=WEIGHT_DECAY,fused=True
    )
    scaler=GradScaler("cuda")

    if rank==0:
        print("GPUs:",world)
        print("Training examples:",len(train_ds))
        print("Validation examples:",len(val_ds))
        print("Trainable parameters:",sum(p.numel() for p in trainable))
        print("Effective batch size:",ACCUM*world)

    if SMOKE_TEST:
        model.train()
        optimizer.zero_grad(set_to_none=True)

        for step,(x,y) in enumerate(train_loader):
            x=x.to(device,non_blocking=True)
            y=y.to(device,non_blocking=True)

            context=model.no_sync() if step<SMOKE_STEPS-1 else nullcontext()

            with context:
                with autocast("cuda",dtype=torch.float16):
                    loss,_=get_loss(model,x,y,cfg["vocab_size"])
                    loss=loss/SMOKE_STEPS

                scaler.scale(loss).backward()

            if step==SMOKE_STEPS-1:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable,GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                break

        dist.barrier()
        if rank==0:
            print("DDP smoke test passed.")
        dist.destroy_process_group()
        return

    latest_path=OUT+"/latest.pt"
    best_val_loss=float("inf")
    start_epoch=0

    if os.path.exists(latest_path):
        saved=torch.load(latest_path,map_location="cpu",weights_only=False)
        expected={
            k for k in model.module.state_dict()
            if k.endswith(".A") or k.endswith(".B")
        }
        assert set(saved["adapters"])==expected

        model.module.load_state_dict(saved["adapters"],strict=False)
        optimizer.load_state_dict(saved["optimizer"])
        scaler.load_state_dict(saved["scaler"])
        start_epoch=saved["epoch"]
        best_val_loss=saved["best_val_loss"]

        if rank==0:
            print("Resuming from epoch:",start_epoch)

    for epoch in range(start_epoch,EPOCHS):
        train_sampler.set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)

        loss_total=0.0
        iterator=tqdm(
            train_loader,desc="Epoch "+str(epoch+1),
            disable=(rank!=0)
        )

        for step,(x,y) in enumerate(iterator):
            x=x.to(device,non_blocking=True)
            y=y.to(device,non_blocking=True)

            sync_now=(step+1)%ACCUM==0
            context=nullcontext() if sync_now else model.no_sync()

            with context:
                with autocast("cuda",dtype=torch.float16):
                    loss,_=get_loss(model,x,y,cfg["vocab_size"])
                    scaled_loss=loss/ACCUM

                scaler.scale(scaled_loss).backward()

            loss_total+=loss.item()

            if sync_now:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable,GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            if rank==0:
                iterator.set_postfix(loss=f"{loss.item():.4f}")

        train_stats=torch.tensor(
            [loss_total,len(train_loader)],device=device
        )
        dist.all_reduce(train_stats,op=dist.ReduceOp.SUM)
        train_loss=(train_stats[0]/train_stats[1]).item()

        dist.barrier()
        if rank==0:
            save_checkpoint(
                latest_path,model,optimizer,scaler,
                epoch+1,best_val_loss,cfg
            )

        dist.barrier()
        model.eval()
        val_loss_sum=torch.tensor(0.0,device=device)
        val_tokens=torch.tensor(0.0,device=device)

        with torch.no_grad():
            for x,y in tqdm(
                val_loader,desc="Validation",
                disable=(rank!=0)
            ):
                x=x.to(device,non_blocking=True)
                y=y.to(device,non_blocking=True)

                with autocast("cuda",dtype=torch.float16):
                    logits=model(x)
                    shift_logits=logits[:,:-1,:].contiguous()
                    shift_labels=y[:,1:].contiguous()

                    loss=F.cross_entropy(
                        shift_logits.view(-1,cfg["vocab_size"]),
                        shift_labels.view(-1),
                        ignore_index=-100,reduction="sum"
                    )

                val_loss_sum+=loss
                val_tokens+=(shift_labels!=-100).sum()

        dist.all_reduce(val_loss_sum,op=dist.ReduceOp.SUM)
        dist.all_reduce(val_tokens,op=dist.ReduceOp.SUM)
        val_loss=(val_loss_sum/val_tokens).item()

        if rank==0:
            print(
                f"Epoch {epoch+1} | Train loss: {train_loss:.4f} | "
                f"Validation loss: {val_loss:.4f}"
            )

            if val_loss<best_val_loss:
                best_val_loss=val_loss
                save_checkpoint(
                    OUT+"/best.pt",model,optimizer,scaler,
                    epoch+1,best_val_loss,cfg
                )

            save_checkpoint(
                latest_path,model,optimizer,scaler,
                epoch+1,best_val_loss,cfg
            )

        dist.barrier()

    if rank==0:
        save_checkpoint(
            OUT+"/final.pt",model,optimizer,scaler,
            EPOCHS,best_val_loss,cfg
        )
        print("Training completed. Best validation loss:",best_val_loss)

    dist.destroy_process_group()


if __name__=="__main__":
    main()

Overwriting /kaggle/working/train_phase6_m1.py


In [ ]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_phase6_m1.py

W1103 04:38:06.236000 383 torch/distributed/run.py:852] 
W1103 04:38:06.236000 383 torch/distributed/run.py:852] *****************************************
W1103 04:38:06.236000 383 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1103 04:38:06.236000 383 torch/distributed/run.py:852] *****************************************
[W1103 04:38:06.961659952 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1103 04:38:08.758342274 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1103 04:38:08.808520031 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
GPUs: 2
Training examples: 45450
Validation examples: 5080
Trainable parameters: 1327104
Effective batch size: 18
Epoch 1: 100%|███████████████| 227

In [17]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_phase6_m1.py

W1103 06:26:07.499000 471 torch/distributed/run.py:852] 
W1103 06:26:07.499000 471 torch/distributed/run.py:852] *****************************************
W1103 06:26:07.499000 471 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1103 06:26:07.499000 471 torch/distributed/run.py:852] *****************************************
[W1103 06:26:07.384618919 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1103 06:26:09.265218478 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1103 06:26:09.273182935 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
GPUs: 2
Training examples: 45450
Validation examples: 5080
Trainable parameters: 1327104
Effective batch size: 18
Resuming from epoch: 2
Training co

In [9]:
import types

def rope_cached(rope,q,k,start_pos):
    seq_len = q.size(-2)

    cos = rope.cos[:,:,start_pos:start_pos+seq_len,:]
    sin = rope.sin[:,:,start_pos:start_pos+seq_len,:]

    q1 = q[...,::2]
    q2 = q[...,1::2]
    k1 = k[...,::2]
    k2 = k[...,1::2]

    q_rot = torch.stack(
        (q1*cos-q2*sin,q1*sin+q2*cos),dim=-1
    ).flatten(-2)

    k_rot = torch.stack(
        (k1*cos-k2*sin,k1*sin+k2*cos),dim=-1
    ).flatten(-2)

    return q_rot,k_rot


def attention_forward_cached(attn,x,past_kv,start_pos):
    B,T,C = x.shape

    qkv = attn.qkv(x)
    q,k,v = qkv.chunk(3,dim=-1)

    q = q.view(B,T,attn.n_heads,attn.head_dim).transpose(1,2)
    k = k.view(B,T,attn.n_heads,attn.head_dim).transpose(1,2)
    v = v.view(B,T,attn.n_heads,attn.head_dim).transpose(1,2)

    q,k = rope_cached(attn.rope,q,k,start_pos)

    if past_kv is not None:
        past_k,past_v = past_kv
        k = torch.cat((past_k,k),dim=2)
        v = torch.cat((past_v,v),dim=2)
        causal = False
    else:
        causal = True

    out = F.scaled_dot_product_attention(
        q,k,v,
        dropout_p=0.0,
        is_causal=causal
    )

    new_kv = (k,v)

    out = out.transpose(1,2).contiguous()
    out = out.view(B,T,C)

    return attn.out_proj(out),new_kv


def block_forward_cached(block,x,past_kv,start_pos):
    attn_out,new_kv = attention_forward_cached(
        block.attn,
        block.norm1(x),
        past_kv,
        start_pos
    )

    x = x+attn_out
    x = x+block.ffn(block.norm2(x))

    return x,new_kv


def gpt_forward_cached(self,x,past_kvs=None,start_pos=0):
    x = self.token_embedding(x)
    x = self.dropout(x)

    if past_kvs is None:
        past_kvs = [None]*len(self.blocks)

    new_kvs = []

    for i,block in enumerate(self.blocks):
        x,new_kv = block_forward_cached(
            block,x,past_kvs[i],start_pos
        )
        new_kvs.append(new_kv)

    x = self.norm(x)
    logits = self.lm_head(x)

    return logits,new_kvs


model.forward_cached = types.MethodType(gpt_forward_cached,model)

print("KV cache methods added.")

KV cache methods added.


In [13]:
model.eval()

test_ids = torch.randint(
    0,tokenizer.vocab_size,(1,65),device=DEVICE
)

with torch.no_grad():
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(DEVICE.type == "cuda")
    ):
        # Normal forward pass
        normal_logits = model(test_ids)

        # Cached prefill: first 64 tokens
        first_logits,past_kvs = model.forward_cached(
            test_ids[:,:64]
        )

        # Cached decoding: token 65
        next_logits,past_kvs = model.forward_cached(
            test_ids[:,64:65],
            past_kvs,
            start_pos=64
        )

    expected = normal_logits[:,-1,:]
    actual = next_logits[:,-1,:]

    max_diff = (expected-actual).abs().max().item()

print("Maximum logit difference:",max_diff)

assert torch.allclose(
    expected,actual,rtol=0.03,atol=0.03
)

print("KV cache verification passed.")

Maximum logit difference: 0.005859375
KV cache verification passed.


In [1]:

import os
import json
import math
import zipfile
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from legal_m1 import GPTModel

# ================= CONFIG =================

BASE_CKPT = "/kaggle/working/m1_750_checkpoints/latest.pt"
ADAPTER_CKPT = "/kaggle/working/phase6_m1_lorma_checkpoints/best.pt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 500

EOS_ID = tokenizer.stoi["<eos>"]

# ================= LORMA+ =================

class LoRMALinear(nn.Module):
    def __init__(self,base,rank=8,alpha=16):
        super().__init__()
        self.base = base
        self.scale = alpha/rank

        out_dim = base.out_features
        self.A = nn.Parameter(torch.empty(rank,out_dim))
        self.B = nn.Parameter(torch.zeros(out_dim,rank))

        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self,x):
        y = self.base(x)
        update = F.linear(F.linear(y,self.A),self.B)
        return y+self.scale*update


def apply_lorma_plus(model):
    for p in model.parameters():
        p.requires_grad = False

    for block in model.blocks:
        block.attn.qkv = LoRMALinear(block.attn.qkv)
        block.attn.out_proj = LoRMALinear(block.attn.out_proj)
        block.ffn.fc1 = LoRMALinear(block.ffn.fc1)
        block.ffn.fc2 = LoRMALinear(block.ffn.fc2)

    return model


# ================= LOAD MODEL =================

print("Loading base checkpoint...")

base = torch.load(
    BASE_CKPT,
    map_location="cpu",
    weights_only=False
)

cfg = base["config"]

model = GPTModel(
    cfg["vocab_size"],
    cfg["d_model"],
    cfg["n_layers"],
    cfg["n_heads"],
    cfg["ffn_dim"],
    cfg["dropout"]
)

model.load_state_dict(base["model"],strict=True)
del base

model = apply_lorma_plus(model)

print("Loading LoRMA+ adapters...")

adapter_ckpt = torch.load(
    ADAPTER_CKPT,
    map_location="cpu",
    weights_only=False
)

adapter_state = adapter_ckpt["adapters"]

# Check that the checkpoint contains only adapter parameters
assert len(adapter_state) > 0
assert all(k.endswith((".A",".B")) for k in adapter_state)

# Load adapters
missing,unexpected = model.load_state_dict(
    adapter_state,
    strict=False
)

# Verify that all adapter parameters were loaded
expected_keys = set(
    k for k in model.state_dict()
    if k.endswith((".A",".B"))
)

assert set(adapter_state.keys()) == expected_keys
assert len(unexpected) == 0

del adapter_ckpt,adapter_state

model = model.to(DEVICE)
model.eval()

print("LoRMA+ adapters loaded successfully.")
print("Model loaded.")
print("Device:",DEVICE)
print("Parameters:",sum(p.numel() for p in model.parameters()))
DEVICE = torch.device(DEVICE)

print("Device:", DEVICE)
print("Device type:", DEVICE.type)

# ================= TOKEN DECODING =================

def decode_tokens(ids):
    output_bytes = bytearray()

    for token_id in ids:
        token_id = int(token_id)

        if token_id == tokenizer.eos_id:
            continue

        token_bytes = tokenizer.itos[token_id]
        output_bytes.extend(token_bytes)

    return output_bytes.decode("utf-8",errors="replace")


@torch.no_grad()
def generate_summary(source_ids,max_new_tokens=500):

    input_ids = PROMPT_IDS+source_ids+SUMMARY_PREFIX_IDS

    assert len(input_ids)<2048

    x = torch.tensor(
        [input_ids],dtype=torch.long,device=DEVICE
    )

    generated = []

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(DEVICE.type == "cuda")
    ):
        # Process the entire prompt only once
        logits,past_kvs = model.forward_cached(x)

        next_token = logits[:,-1,:].argmax(dim=-1).item()

        if next_token == EOS_ID:
            return ""

        generated.append(next_token)

        # Process only the newly generated token each time
        for step in range(max_new_tokens-1):

            if next_token == EOS_ID:
                break

            next_input = torch.tensor(
                [[next_token]],dtype=torch.long,device=DEVICE
            )

            logits,past_kvs = model.forward_cached(
                next_input,
                past_kvs,
                start_pos=len(input_ids)+step
            )

            next_token = logits[:,-1,:].argmax(dim=-1).item()

            if next_token == EOS_ID:
                break

            generated.append(next_token)

    return decode_tokens(generated).strip()


# ================= PREPARE TEST CHUNKS =================

if "test_chunks" not in globals():
    test_chunks = []

    for row in tqdm(test_data,desc="Preparing test chunks"):
        chunks = make_chunks(row["document"])

        for chunk_idx,chunk_ids in enumerate(chunks):
            test_chunks.append({
                "id":row["id"],
                "chunk_idx":chunk_idx,
                "source_ids":chunk_ids
            })

print("Test documents:",len(test_data))
print("Test chunks:",len(test_chunks))


# ================= GENERATE PREDICTIONS =================

chunk_predictions = {}

for item in tqdm(test_chunks,desc="Generating summaries"):
    doc_id = item["id"]
    chunk_idx = item["chunk_idx"]

    summary = generate_summary(item["source_ids"])

    if doc_id not in chunk_predictions:
        chunk_predictions[doc_id] = []

    chunk_predictions[doc_id].append(
        (chunk_idx,summary)
    )


# ================= COMBINE DOCUMENT SUMMARIES =================

predictions = []

for row in tqdm(test_data,desc="Combining summaries"):
    doc_id = row["id"]

    chunks = chunk_predictions.get(doc_id,[])
    chunks.sort(key=lambda x:x[0])

    summary_parts = []

    for chunk_idx,summary in chunks:
        if summary:
            summary_parts.append(summary)

    final_summary = " ".join(summary_parts).strip()

    predictions.append({
        "id":doc_id,
        "summary":final_summary
    })


# ================= SAVE JSON =================

PRED_PATH = "/kaggle/working/predictions.json"
SUBMISSION_PATH = "/kaggle/working/submission.json"
ZIP_PATH = "/kaggle/working/submission.zip"

with open(PRED_PATH,"w",encoding="utf-8") as f:
    json.dump(predictions,f,ensure_ascii=False,indent=2)

# Keep the conventional submission filename inside the ZIP.
with open(SUBMISSION_PATH,"w",encoding="utf-8") as f:
    json.dump(predictions,f,ensure_ascii=False,indent=2)


# ================= CREATE ZIP =================

with zipfile.ZipFile(
    ZIP_PATH,"w",compression=zipfile.ZIP_DEFLATED
) as z:
    z.write(SUBMISSION_PATH,arcname="submission.json")


# ================= VERIFY =================

with open(PRED_PATH,"r",encoding="utf-8") as f:
    check = json.load(f)

expected_ids = [row["id"] for row in test_data]
predicted_ids = [row["id"] for row in check]

assert len(check) == len(test_data)
assert predicted_ids == expected_ids
assert all(isinstance(x["summary"],str) for x in check)

with zipfile.ZipFile(ZIP_PATH,"r") as z:
    assert z.namelist() == ["submission.json"]
    assert z.testzip() is None

print("\nPrediction generation completed.")
print("Documents:",len(predictions))
print("Predictions:",PRED_PATH)
print("Submission ZIP:",ZIP_PATH)
print("ZIP contents:",zipfile.ZipFile(ZIP_PATH).namelist())

print("\nSample prediction:")
print(predictions[0]["id"])
print(predictions[0]["summary"][:1000])


Loading base checkpoint...
Loading LoRMA+ adapters...
LoRMA+ adapters loaded successfully.
Model loaded.
Device: cuda
Parameters: 110939904
Device: cuda
Device type: cuda
Test documents: 100
Test chunks: 820


Generating summaries:   0%|          | 0/820 [00:00<?, ?it/s]

Combining summaries:   0%|          | 0/100 [00:00<?, ?it/s]


Prediction generation completed.
Documents: 100
Predictions: /kaggle/working/predictions.json
Submission ZIP: /kaggle/working/submission.zip
ZIP contents: ['submission.json']

Sample prediction:
4963
The Income Tax Appellate Tribunal upheld the order of the Income Tax Appellate Tribunal and confirmed the order of the Income Tax Appellate Tribunal. The computation of the net wealth of an assessee calls for a determination of his assets and debts as on the valuation date.The definition embodied in the substantive part of cl.(q) of s.2 indicates that broadly Parliament has fixed upon the last day of the "previous year" as defined under the Income Tax Act.The words "value date" in the definition of the expression "value date" in the definition of the expression "value date" in the definition of the expression "value date" in the definition of the expression "value date" in the definition of the expression "value date" in the definition of the expression "value date" in the definition of t

Phase 7

In [9]:
class GPTModel(nn.Module):
    def __init__(self,vocab_size,d_model,n_layers,n_heads,ffn_dim,dropout=0.2):
        super().__init__()
        self.token_embedding=nn.Embedding(vocab_size,d_model)
        nn.init.normal_(self.token_embedding.weight,mean=0.0,std=0.02)
        self.blocks=nn.ModuleList([TransformerBlock(d_model,n_heads,ffn_dim,dropout) for _ in range(n_layers)])
        self.norm=RMSNorm(d_model)
        self.lm_head=nn.Linear(d_model,vocab_size,bias=False)
        self.lm_head.weight=self.token_embedding.weight
        self.dropout=nn.Dropout(dropout)

    def forward(self,x,return_hidden=False):
        x=self.token_embedding(x)
        x=self.dropout(x)

        for block in self.blocks:
            x=block(x)

        x=self.norm(x)

        if return_hidden:
            return x

        return self.lm_head(x)

In [10]:
class LoRMALinear(nn.Module):
    def __init__(self,linear,rank=8,alpha=16):
        super().__init__()

        self.base=linear
        self.rank=rank
        self.scale=alpha/rank

        self.in_features=linear.in_features
        self.out_features=linear.out_features

        self.A=nn.Parameter(torch.empty(rank,self.out_features))
        self.B=nn.Parameter(torch.zeros(self.out_features,rank))

        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

        for p in self.base.parameters():
            p.requires_grad=False

    def forward(self,x):
        y=F.linear(x,self.base.weight,None)

        update=F.linear(y,self.A)
        update=F.linear(update,self.B)

        y=y+self.scale*update

        if self.base.bias is not None:
            y=y+self.base.bias

        return y

In [11]:
def apply_lorma(model,rank=8,alpha=16):
    for block in model.blocks:
        block.attn.qkv=LoRMALinear(block.attn.qkv,rank,alpha)
        block.attn.out_proj=LoRMALinear(block.attn.out_proj,rank,alpha)

        block.ffn.fc1=LoRMALinear(block.ffn.fc1,rank,alpha)
        block.ffn.fc2=LoRMALinear(block.ffn.fc2,rank,alpha)

    return model
    

In [12]:
model=GPTModel(
    VOCAB_SIZE,D_MODEL,N_LAYERS,N_HEADS,FFN_DIM,DROPOUT
).to(DEVICE)

checkpoint=torch.load(
    "/kaggle/working/m1_750_checkpoints/latest.pt",
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model"])

model=apply_lorma(model,rank=8,alpha=16)

for p in model.parameters():
    p.requires_grad=False

for module in model.modules():
    if isinstance(module,LoRMALinear):
        module.A.requires_grad=True
        module.B.requires_grad=True

model=model.to(DEVICE)

total_params=sum(p.numel() for p in model.parameters())
trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total parameters:",total_params)
print("Trainable parameters:",trainable_params)
print("Trainable percentage:",100*trainable_params/total_params)

Total parameters: 110939904
Trainable parameters: 1327104
Trainable percentage: 1.1962368382795787


In [13]:
import os
import json
import random
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

PHASE7_FILES = {
    "train": "Phase_7_PCR/Train/train.json",
    "train_candidates": "Phase_7_PCR/Train/train_candidates.json",
    "dev": "Phase_7_PCR/Dev/dev.json",
    "dev_candidates": "Phase_7_PCR/Dev/dev_candidates.json",
    "test": "Phase_7_PCR/Test/test.json",
    "test_candidates": "Phase_7_PCR/Test/test_candidates.json"
}

SEED = 42
random.seed(SEED)

phase7_paths = {}

for name,file_name in PHASE7_FILES.items():
    print("Downloading:",name)

    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=file_name,
        repo_type="dataset",
        token=HF_TOKEN
    )

    phase7_paths[name] = path
    print("Path:",path)

print("All Phase 7 files downloaded/accessed successfully.")

Downloading: train


Phase_7_PCR/Train/train.json:   0%|          | 0.00/33.2M [00:00<?, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Train/train.json
Downloading: train_candidates


Phase_7_PCR/Train/train_candidates.json:   0%|          | 0.00/160M [00:00<?, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Train/train_candidates.json
Downloading: dev


dev.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

test.json: 0.00B [00:00, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Test/test.json
Downloading: test_candidates


Phase_7_PCR/Test/test_candidates.json:   0%|          | 0.00/71.1M [00:00<?, ?B/s]

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Test/test_candidates.json
All Phase 7 files downloaded/accessed successfully.


In [14]:
with open(phase7_paths["train"],"r") as f:
    train_data=json.load(f)

with open(phase7_paths["train_candidates"],"r") as f:
    train_candidates=json.load(f)

with open(phase7_paths["dev"],"r") as f:
    dev_data=json.load(f)

with open(phase7_paths["dev_candidates"],"r") as f:
    dev_candidates=json.load(f)

with open(phase7_paths["test"],"r") as f:
    test_data=json.load(f)

with open(phase7_paths["test_candidates"],"r") as f:
    test_candidates=json.load(f)

print("Train queries:",len(train_data))
print("Train candidates:",len(train_candidates))
print("Dev queries:",len(dev_data))
print("Dev candidates:",len(dev_candidates))
print("Test queries:",len(test_data))
print("Test candidates:",len(test_candidates))

Train queries: 827
Train candidates: 4320
Dev queries: 118
Dev candidates: 1023
Test queries: 237
Test candidates: 1727


In [18]:
qid=list(train_data.keys())[0]
cid=list(train_candidates.keys())[0]

print("QUERY FIELDS:",train_data[qid].keys())
print("CANDIDATE FIELDS:",train_candidates[cid].keys())

print("\nRELEVANCE LABELS:")
print(train_data[qid].get("relevant_candidates",None))

QUERY FIELDS: dict_keys(['text', 'relevant_candidates'])
CANDIDATE FIELDS: dict_keys(['text'])

RELEVANCE LABELS:
['0001292810', '0001742283']


In [20]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class LoRMALinear(nn.Module):
    def __init__(self,linear,rank=8,alpha=16):
        super().__init__()
        self.base=linear
        self.rank=rank
        self.scale=alpha/rank

        self.A=nn.Parameter(torch.empty(rank,linear.out_features))
        self.B=nn.Parameter(torch.zeros(linear.out_features,rank))

        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

        for p in self.base.parameters():
            p.requires_grad=False

    def forward(self,x):
        y=F.linear(x,self.base.weight,None)
        update=F.linear(y,self.A)
        update=F.linear(update,self.B)
        y=y+self.scale*update

        if self.base.bias is not None:
            y=y+self.base.bias

        return y


def apply_lorma(model,rank=8,alpha=16):
    for p in model.parameters():
        p.requires_grad=False

    for block in model.blocks:
        block.attn.qkv=LoRMALinear(block.attn.qkv,rank,alpha)
        block.attn.out_proj=LoRMALinear(block.attn.out_proj,rank,alpha)
        block.ffn.fc1=LoRMALinear(block.ffn.fc1,rank,alpha)
        block.ffn.fc2=LoRMALinear(block.ffn.fc2,rank,alpha)

    return model


model=apply_lorma(model,rank=8,alpha=16)

# Retrieval projection head
retrieval_head=nn.Sequential(
    nn.Linear(D_MODEL,D_MODEL),
    nn.Tanh(),
    nn.Linear(D_MODEL,256)
).to(DEVICE)

trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable+=sum(p.numel() for p in retrieval_head.parameters() if p.requires_grad)

total=sum(p.numel() for p in model.parameters())

print("M1 loaded with LoRMA+")
print("Trainable M1 parameters:",trainable)
print("Total M1 parameters:",total)

M1 loaded with LoRMA+
Trainable M1 parameters: 2114560
Total M1 parameters: 112267008


In [21]:
def forward(self,x,return_hidden=False):
    x=self.token_embedding(x)
    x=self.dropout(x)

    for block in self.blocks:
        x=block(x)

    x=self.norm(x)

    if return_hidden:
        return x

    return self.lm_head(x)


GPTModel.forward=forward

In [22]:
# Convert each case into a single text string
train_queries={
    qid:" ".join(item["text"])
    for qid,item in train_data.items()
}

train_docs={
    cid:" ".join(item["text"])
    for cid,item in train_candidates.items()
}

train_relevance={
    qid:set(item["relevant_candidates"])
    for qid,item in train_data.items()
}

# Verify that every positive exists in the training candidate pool
missing=[]
for qid,positives in train_relevance.items():
    for cid in positives:
        if cid not in train_docs:
            missing.append((qid,cid))

print("Queries:",len(train_queries))
print("Candidates:",len(train_docs))
print("Queries with positives:",sum(len(v)>0 for v in train_relevance.values()))
print("Total positive pairs:",sum(len(v) for v in train_relevance.values()))
print("Missing positive candidates:",len(missing))
print("Example query characters:",len(train_queries[list(train_queries.keys())[0]]))
print("Example candidate characters:",len(train_docs[list(train_docs.keys())[0]]))

Queries: 827
Candidates: 4320
Queries with positives: 817
Total positive pairs: 5308
Missing positive candidates: 0
Example query characters: 31473
Example candidate characters: 21968


In [23]:
model=model.to(DEVICE)
retrieval_head=retrieval_head.to(DEVICE)

print("Model device:",next(model.parameters()).device)
print("Adapter device:",next(p for p in model.parameters() if p.requires_grad).device)

Model device: cuda:0
Adapter device: cuda:0


In [ ]:
def make_chunks(text):
    tokens=encode_text(bpe,text,add_eos=False)

    if len(tokens)==0:
        return [[EOS_ID]]

    chunks=[]
    for i in range(0,len(tokens),CHUNK_SIZE-1):
        chunk=tokens[i:i+CHUNK_SIZE-1]+[EOS_ID]
        chunks.append(chunk)

    if len(chunks)>MAX_CHUNKS:
        indices=torch.linspace(0,len(chunks)-1,MAX_CHUNKS).long().tolist()
        chunks=[chunks[i] for i in indices]

    return chunks

In [30]:
from tqdm.auto import tqdm

train_query_chunks={}
train_candidate_chunks={}

for qid,text in tqdm(train_queries.items(),total=len(train_queries),desc="Tokenizing queries"):
    train_query_chunks[qid]=make_chunks(text)

for cid,text in tqdm(train_docs.items(),total=len(train_docs),desc="Tokenizing candidates"):
    train_candidate_chunks[cid]=make_chunks(text)

os.makedirs("/kaggle/working/m1_pcr",exist_ok=True)

torch.save({
    "queries":train_query_chunks,
    "candidates":train_candidate_chunks,
    "relevance":train_relevance
},"/kaggle/working/m1_pcr/train_tokens.pt")

print("Tokenization complete")
print("Query documents:",len(train_query_chunks))
print("Candidate documents:",len(train_candidate_chunks))
print("Average query chunks:",sum(len(x) for x in train_query_chunks.values())/len(train_query_chunks))
print("Average candidate chunks:",sum(len(x) for x in train_candidate_chunks.values())/len(train_candidate_chunks))

Tokenizing queries:   0%|          | 0/827 [00:00<?, ?it/s]

Tokenizing candidates:   0%|          | 0/4320 [00:00<?, ?it/s]

Tokenization complete
Query documents: 827
Candidate documents: 4320
Average query chunks: 3.966142684401451
Average candidate chunks: 3.8851851851851853


In [31]:
dev_queries={
    qid:" ".join(item["text"])
    for qid,item in dev_data.items()
}

dev_docs={
    cid:" ".join(item["text"])
    for cid,item in dev_candidates.items()
}

dev_relevance={
    qid:set(item["relevant_candidates"])
    for qid,item in dev_data.items()
}

dev_query_chunks={}
dev_candidate_chunks={}

for qid,text in tqdm(dev_queries.items(),total=len(dev_queries),desc="Tokenizing dev queries"):
    dev_query_chunks[qid]=make_chunks(text)

for cid,text in tqdm(dev_docs.items(),total=len(dev_docs),desc="Tokenizing dev candidates"):
    dev_candidate_chunks[cid]=make_chunks(text)

torch.save({
    "queries":dev_query_chunks,
    "candidates":dev_candidate_chunks,
    "relevance":dev_relevance
},"/kaggle/working/m1_pcr/dev_tokens.pt")

print("Dev tokenization complete")
print("Queries:",len(dev_query_chunks))
print("Candidates:",len(dev_candidate_chunks))
print("Positive pairs:",sum(len(x) for x in dev_relevance.values()))

Tokenizing dev queries:   0%|          | 0/118 [00:00<?, ?it/s]

Tokenizing dev candidates:   0%|          | 0/1023 [00:00<?, ?it/s]

Dev tokenization complete
Queries: 118
Candidates: 1023
Positive pairs: 973


In [ ]:
%%writefile /kaggle/working/train_m1_pcr.py

In [40]:
%%writefile /kaggle/working/train_m1_pcr.py
import os
import math
import random
import argparse
from contextlib import nullcontext

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm


# ---------------------------- Configuration ----------------------------

D_MODEL = 768
N_LAYERS = 12
N_HEADS = 12
FFN_DIM = 3072
VOCAB_SIZE = 32000
BLOCK_SIZE = 2048
DROPOUT = 0.2

DEFAULT_BASE_CKPT = "/kaggle/working/m1_750_checkpoints/latest.pt"
DEFAULT_TRAIN_TOKENS = "/kaggle/working/m1_pcr/train_tokens.pt"
DEFAULT_DEV_TOKENS = "/kaggle/working/m1_pcr/dev_tokens.pt"
DEFAULT_OUTPUT = "/kaggle/working/m1_pcr/m1_lorma_best.pt"


# ---------------------------- M1 architecture ----------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(rms + self.eps)
        return self.weight * x


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=BLOCK_SIZE, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        self.register_buffer("cos", freqs.cos()[None, None, :, :])
        self.register_buffer("sin", freqs.sin()[None, None, :, :])

    def forward(self, q, k):
        seq_len = q.size(-2)
        cos = self.cos[:, :, :seq_len, :].to(dtype=q.dtype)
        sin = self.sin[:, :, :seq_len, :].to(dtype=q.dtype)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack((q1 * cos - q2 * sin,
                             q1 * sin + q2 * cos), dim=-1).flatten(-2)
        k_rot = torch.stack((k1 * cos - k2 * sin,
                             k1 * sin + k2 * cos), dim=-1).flatten(-2)
        return q_rot, k_rot


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.2):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.rope = RotaryEmbedding(self.head_dim)
        self.dropout = dropout

    def forward(self, x, valid_mask):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        q, k = self.rope(q, k)

        causal = torch.ones(T, T, device=x.device, dtype=torch.bool).tril()
        allowed = causal[None, None, :, :] & valid_mask[:, None, None, :]
        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=allowed,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False
        )

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, ffn_dim, dropout=0.2):
        super().__init__()
        self.fc1 = nn.Linear(d_model, ffn_dim)
        self.fc2 = nn.Linear(ffn_dim, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        return self.dropout(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout=0.2):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = FeedForward(d_model, ffn_dim, dropout)

    def forward(self, x, valid_mask):
        x = x + self.attn(self.norm1(x), valid_mask)
        x = x + self.ffn(self.norm2(x))
        return x


class GPTEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, dropout=0.2):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        nn.init.normal_(self.token_embedding.weight, mean=0.0, std=0.02)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_layers)
        ])
        self.norm = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.gradient_checkpointing = True

    def forward(self, input_ids, lengths):
        x = self.token_embedding(input_ids)
        x = self.dropout(x)

        valid_mask = (
            torch.arange(input_ids.size(1), device=input_ids.device)[None, :]
            < lengths[:, None]
        )

        for block in self.blocks:
            if self.gradient_checkpointing and self.training:
                x = checkpoint(block, x, valid_mask, use_reentrant=False)
            else:
                x = block(x, valid_mask)

        x = self.norm(x)
        eos_positions = lengths - 1
        pooled = x[torch.arange(x.size(0), device=x.device), eos_positions]
        return pooled


# ---------------------------- LoRMA+ ----------------------------

class LoRMALinear(nn.Module):
    def __init__(self, linear, rank=8, alpha=16):
        super().__init__()
        self.base = linear
        self.scale = alpha / rank
        self.A = nn.Parameter(torch.empty(rank, linear.out_features))
        self.B = nn.Parameter(torch.zeros(linear.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        y = F.linear(x, self.base.weight, None)
        update = F.linear(y, self.A)
        update = F.linear(update, self.B)
        y = y + self.scale * update
        if self.base.bias is not None:
            y = y + self.base.bias
        return y


def apply_lorma(model, rank=8, alpha=16):
    for p in model.parameters():
        p.requires_grad = False

    for block in model.blocks:
        block.attn.qkv = LoRMALinear(block.attn.qkv, rank, alpha)
        block.attn.out_proj = LoRMALinear(block.attn.out_proj, rank, alpha)
        block.ffn.fc1 = LoRMALinear(block.ffn.fc1, rank, alpha)
        block.ffn.fc2 = LoRMALinear(block.ffn.fc2, rank, alpha)

    return model


class RetrievalModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.projection = nn.Sequential(
            nn.Linear(D_MODEL, D_MODEL),
            nn.Tanh(),
            nn.Linear(D_MODEL, 256)
        )

    def forward(self, input_ids, lengths):
        hidden = self.encoder(input_ids, lengths)
        z = self.projection(hidden)
        return F.normalize(z.float(), p=2, dim=-1)


# ---------------------------- Data helpers ----------------------------

def load_token_data(path):
    return torch.load(path, map_location="cpu", weights_only=False)


def pad_chunks(chunks, device):
    lengths = torch.tensor([len(x) for x in chunks], dtype=torch.long, device=device)
    max_len = int(lengths.max().item())
    input_ids = torch.zeros((len(chunks), max_len), dtype=torch.long, device=device)

    for i, chunk in enumerate(chunks):
        input_ids[i, :len(chunk)] = torch.as_tensor(chunk, dtype=torch.long, device=device)

    return input_ids, lengths


def choose_chunk(chunks):
    return chunks[random.randrange(len(chunks))]


def build_training_batch(qid, token_data, candidate_ids, negatives_per_query=6):
    query_chunks = token_data["queries"]
    candidate_chunks = token_data["candidates"]
    relevance = token_data["relevance"]

    positives = list(relevance[qid])
    random.shuffle(positives)
    positives = positives[:2]

    positive_set = set(positives)
    available = [cid for cid in candidate_ids if cid not in positive_set]
    negatives = random.sample(available, min(negatives_per_query, len(available)))

    selected = positives + negatives
    chunks = [choose_chunk(query_chunks[qid])]
    chunks.extend(choose_chunk(candidate_chunks[cid]) for cid in selected)

    return chunks, len(positives)


@torch.no_grad()
def encode_documents(model, documents, device, chunk_batch_size=8, desc="Encoding"):
    model.eval()

    all_chunk_pairs = []
    for doc_id, chunks in documents.items():
        for chunk in chunks:
            all_chunk_pairs.append((doc_id, chunk))

    sums = {}
    counts = {}

    iterator = range(0, len(all_chunk_pairs), chunk_batch_size)
    for start in tqdm(iterator, desc=desc, disable=not is_main_process()):
        batch = all_chunk_pairs[start:start + chunk_batch_size]
        chunks = [x[1] for x in batch]
        input_ids, lengths = pad_chunks(chunks, device)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            vectors = model(input_ids, lengths)

        vectors = vectors.float()
        for i, (doc_id, _) in enumerate(batch):
            if doc_id not in sums:
                sums[doc_id] = vectors[i].clone()
                counts[doc_id] = 1
            else:
                sums[doc_id] += vectors[i]
                counts[doc_id] += 1

    result = {}
    for doc_id in documents:
        result[doc_id] = F.normalize(sums[doc_id] / counts[doc_id], dim=0)
    return result


def evaluate(model, dev_data, device):
    q_vectors = encode_documents(
        model, dev_data["queries"], device, chunk_batch_size=8,
        desc="Dev query chunks"
    )
    c_vectors = encode_documents(
        model, dev_data["candidates"], device, chunk_batch_size=8,
        desc="Dev candidate chunks"
    )

    candidate_ids = list(dev_data["candidates"].keys())
    candidate_matrix = torch.stack([c_vectors[cid] for cid in candidate_ids]).to(device)

    hits = 0
    predicted_total = 0
    gold_total = 0

    for qid, qvec in tqdm(q_vectors.items(), desc="Dev ranking"):
        scores = candidate_matrix @ qvec.to(device)
        top_indices = torch.topk(scores, k=min(7, len(candidate_ids))).indices.tolist()
        predicted = {candidate_ids[i] for i in top_indices}
        gold = dev_data["relevance"].get(qid, set())

        hits += len(predicted & gold)
        predicted_total += 7
        gold_total += len(gold)

    precision = hits / max(1, predicted_total)
    recall = hits / max(1, gold_total)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)

    return f1, precision, recall


# ---------------------------- Distributed setup ----------------------------

def is_main_process():
    return not dist.is_initialized() or dist.get_rank() == 0


def setup_ddp():
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend="nccl")
    return local_rank, dist.get_rank(), dist.get_world_size()


def cleanup_ddp():
    if dist.is_initialized():
        dist.destroy_process_group()


def load_model(base_checkpoint, device, rank=8, alpha=16):
    encoder = GPTEncoder(
        VOCAB_SIZE, D_MODEL, N_LAYERS, N_HEADS, FFN_DIM, DROPOUT
    )

    checkpoint_data = torch.load(
        base_checkpoint, map_location="cpu", weights_only=False
    )
    state = checkpoint_data["model"] if "model" in checkpoint_data else checkpoint_data
    encoder.load_state_dict(state, strict=True)

    encoder = apply_lorma(encoder, rank=rank, alpha=alpha)
    model = RetrievalModel(encoder).to(device)
    return model


# ---------------------------- Training ----------------------------

def run(args):
    local_rank, rank, world_size = setup_ddp()
    device = torch.device("cuda", local_rank)

    torch.manual_seed(args.seed + rank)
    random.seed(args.seed + rank)
    torch.cuda.manual_seed_all(args.seed + rank)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    train_data = load_token_data(args.train_tokens)
    dev_data = load_token_data(args.dev_tokens)

    model = load_model(args.base_checkpoint, device, args.rank, args.alpha)
    model = DDP(
        model,
        device_ids=[local_rank],
        output_device=local_rank,
        find_unused_parameters=False
    )

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=args.lr,
        weight_decay=args.weight_decay
    )
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    if args.smoke_test:
        model.train()
        qid = next(q for q, rel in train_data["relevance"].items() if len(rel) > 0)
        candidate_ids = list(train_data["candidates"].keys())
        chunks, npos = build_training_batch(
            qid, train_data, candidate_ids, args.negatives
        )
        input_ids, lengths = pad_chunks(chunks, device)

        optimizer.zero_grad(set_to_none=True)

        # Run the smoke test in FP32 to separate implementation/gradient
        # problems from FP16 loss-scaling overflow.
        with torch.autocast(device_type="cuda", enabled=False):
            vectors = model(input_ids, lengths)
            logits = vectors[0:1] @ vectors[1:].T / args.temperature
            loss = (
                torch.logsumexp(logits, dim=1)
                - torch.logsumexp(logits[:, :npos], dim=1)
            ).mean()

        loss.backward()

        grad_tensors = [p.grad for p in trainable_params if p.grad is not None]
        finite_grad_count = sum(
            1 for grad in grad_tensors if torch.isfinite(grad).all().item()
        )
        nonfinite_grad_count = len(grad_tensors) - finite_grad_count

        if rank == 0:
            print(
                f"Smoke diagnostics | loss={loss.item():.6f} | "
                f"output_finite={torch.isfinite(vectors).all().item()} | "
                f"gradient_tensors={len(grad_tensors)} | "
                f"finite_gradients={finite_grad_count} | "
                f"nonfinite_gradients={nonfinite_grad_count}"
            )

        if not torch.isfinite(loss).item() or finite_grad_count == 0 or nonfinite_grad_count > 0:
            cleanup_ddp()
            raise RuntimeError(
                "Smoke test failed. See diagnostics above; full training was not started."
            )

        torch.nn.utils.clip_grad_norm_(trainable_params, args.grad_clip)
        optimizer.step()

        dist.barrier()
        if rank == 0:
            os.makedirs(os.path.dirname(args.output), exist_ok=True)
            temp_path = args.output + ".smoke"
            torch.save({"model": model.module.state_dict()}, temp_path)
            check = torch.load(temp_path, map_location="cpu", weights_only=False)
            assert "model" in check and len(check["model"]) > 0
            os.remove(temp_path)
            print(f"SMOKE TEST PASSED | loss={loss.item():.4f} | finite trainable gradients={finite_grad_count}/{len(grad_tensors)}")
        dist.barrier()
        cleanup_ddp()
        return

    best_f1 = -1.0
    candidate_ids = list(train_data["candidates"].keys())
    valid_query_ids = [
        qid for qid, rel in train_data["relevance"].items()
        if len(rel) > 0
    ]

    for epoch in range(args.epochs):
        model.train()
        random.Random(args.seed + epoch).shuffle(valid_query_ids)

        remainder = len(valid_query_ids) % world_size
        if remainder:
            valid_query_ids.extend(valid_query_ids[:world_size - remainder])

        local_query_ids = valid_query_ids[rank::world_size]
        progress = tqdm(
            local_query_ids,
            desc=f"Epoch {epoch+1}/{args.epochs}",
            disable=(rank != 0)
        )

        total_loss = 0.0
        optimizer.zero_grad(set_to_none=True)

        for step, qid in enumerate(progress):
            chunks, npos = build_training_batch(
                qid, train_data, candidate_ids, args.negatives
            )
            input_ids, lengths = pad_chunks(chunks, device)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                vectors = model(input_ids, lengths)
                query_vector = vectors[0:1]
                candidate_vectors = vectors[1:]
                logits = query_vector @ candidate_vectors.T / args.temperature
                loss = (
                    torch.logsumexp(logits, dim=1)
                    - torch.logsumexp(logits[:, :npos], dim=1)
                ).mean()

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            total_loss += loss.item()
            if rank == 0:
                progress.set_postfix(loss=f"{loss.item():.4f}")

        # Keep all ranks synchronized before rank-zero evaluation.
        dist.barrier()

        if rank == 0:
            train_loss = total_loss / max(1, len(local_query_ids))
            print(f"\nEpoch {epoch+1}: rank-0 mean training loss = {train_loss:.4f}")

            dev_f1, dev_precision, dev_recall = evaluate(
                model.module, dev_data, device
            )
            print(
                f"Dev Micro-F1@7={dev_f1:.5f} | "
                f"Precision@7={dev_precision:.5f} | "
                f"Recall@7={dev_recall:.5f}"
            )

            if dev_f1 > best_f1:
                best_f1 = dev_f1
                os.makedirs(os.path.dirname(args.output), exist_ok=True)
                torch.save({
                    "model": model.module.state_dict(),
                    "epoch": epoch + 1,
                    "best_dev_f1": best_f1,
                    "dev_precision": dev_precision,
                    "dev_recall": dev_recall,
                    "config": vars(args)
                }, args.output)
                print("Saved best checkpoint:", args.output)

        dist.barrier()

    if rank == 0:
        print(f"Training complete. Best Dev Micro-F1@7: {best_f1:.5f}")

    cleanup_ddp()


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--base-checkpoint", default=DEFAULT_BASE_CKPT)
    parser.add_argument("--train-tokens", default=DEFAULT_TRAIN_TOKENS)
    parser.add_argument("--dev-tokens", default=DEFAULT_DEV_TOKENS)
    parser.add_argument("--output", default=DEFAULT_OUTPUT)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--weight-decay", type=float, default=0.01)
    parser.add_argument("--rank", type=int, default=8)
    parser.add_argument("--alpha", type=int, default=16)
    parser.add_argument("--negatives", type=int, default=6)
    parser.add_argument("--temperature", type=float, default=0.05)
    parser.add_argument("--grad-clip", type=float, default=1.0)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--smoke-test", action="store_true")
    return parser.parse_args()


if __name__ == "__main__":
    args = parse_args()
    run(args)


Overwriting /kaggle/working/train_m1_pcr.py


In [37]:
!python -m torch.distributed.run --standalone --nproc_per_node=2 /kaggle/working/train_m1_pcr.py --smoke-test


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[W1108 13:41:10.902489049 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1108 13:41:12.019393687 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1108 13:41:12.045348405 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Smoke diagnostics | loss=1.162313 | output_finite=True | gradient_tensors=100 | finite_gradients=100 | nonfinite_gradients=0
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
S

In [41]:
!python -m torch.distributed.run --standalone --nproc_per_node=2 /kaggle/working/train_m1_pcr.py


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[W1108 13:42:54.082697353 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1108 13:42:57.287738922 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1108 13:42:57.311985789 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Epoch 1/5: 100%|█████████████████| 409/409 [03:23<00:00,  2.01it/s, loss=0.3032]
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)

Epoch 1: rank-0 mean training loss = 0.9244


In [42]:
import os
import json
import pickle
import zipfile
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
import importlib.util

# -------------------- Paths and configuration --------------------

DEVICE = torch.device("cuda:0")
TOKENIZER_PATH = "/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl"
BASE_CKPT = "/kaggle/working/m1_750_checkpoints/latest.pt"
BEST_CKPT = "/kaggle/working/m1_pcr/m1_lorma_best.pt"

OUTPUT_DIR = "/kaggle/working/m1_pcr"
OUTPUT_JSON = os.path.join(OUTPUT_DIR, "prediction.json")
OUTPUT_ZIP = "/kaggle/working/submission_m1_p7.zip"

CHUNK_SIZE = 512
MAX_CHUNKS = 4
EOS_ID = 0
TOP_K = 7

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------- Download test data --------------------

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

test_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="Phase_7_PCR/Test/test.json",
    repo_type="dataset",
    token=True
)

test_candidates_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="Phase_7_PCR/Test/test_candidates.json",
    repo_type="dataset",
    token=True
)

with open(test_path, "r") as f:
    test_data = json.load(f)

with open(test_candidates_path, "r") as f:
    test_candidates = json.load(f)

print("Test queries:", len(test_data))
print("Test candidates:", len(test_candidates))

# -------------------- Load tokenizer --------------------

with open(TOKENIZER_PATH, "rb") as f:
    bpe = pickle.load(f)

def make_chunks(text):
    tokens = bpe.encode_text(text)

    # Remove EOS if the tokenizer already appended it.
    if len(tokens) > 0 and tokens[-1] == EOS_ID:
        tokens = tokens[:-1]

    if len(tokens) == 0:
        return [[EOS_ID]]

    chunks = []

    for i in range(0, len(tokens), CHUNK_SIZE - 1):
        chunk = tokens[i:i + CHUNK_SIZE - 1] + [EOS_ID]
        chunks.append(chunk)

    if len(chunks) > MAX_CHUNKS:
        indices = torch.linspace(
            0, len(chunks) - 1, MAX_CHUNKS
        ).long().tolist()
        chunks = [chunks[i] for i in indices]

    return chunks

# -------------------- Tokenize test documents --------------------

test_queries_tokens = {}

for qid, item in tqdm(test_data.items(), desc="Tokenizing test queries"):
    text = " ".join(item["text"])
    test_queries_tokens[qid] = make_chunks(text)

test_candidates_tokens = {}

for cid, item in tqdm(test_candidates.items(), desc="Tokenizing test candidates"):
    text = " ".join(item["text"])
    test_candidates_tokens[cid] = make_chunks(text)

print("Tokenization completed.")

# -------------------- Import M1 model implementation --------------------

spec = importlib.util.spec_from_file_location(
    "train_m1_pcr",
    "/kaggle/working/train_m1_pcr.py"
)
pcr = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pcr)

# -------------------- Load best trained model --------------------

model = pcr.load_model(
    BASE_CKPT,
    DEVICE,
    rank=8,
    alpha=16
)

checkpoint = torch.load(
    BEST_CKPT,
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(checkpoint["model"], strict=True)
model.eval()

print("Loaded best checkpoint from epoch:", checkpoint["epoch"])
print("Best Dev Micro-F1@7:", checkpoint["best_dev_f1"])

# -------------------- Encode test queries and candidates --------------------

test_query_vectors = pcr.encode_documents(
    model,
    test_queries_tokens,
    DEVICE,
    chunk_batch_size=8,
    desc="Encoding test queries"
)

test_candidate_vectors = pcr.encode_documents(
    model,
    test_candidates_tokens,
    DEVICE,
    chunk_batch_size=8,
    desc="Encoding test candidates"
)

# -------------------- Rank candidates --------------------

candidate_ids = list(test_candidates_tokens.keys())

candidate_matrix = torch.stack([
    test_candidate_vectors[cid] for cid in candidate_ids
]).to(DEVICE)

predictions = {}

for qid, qvec in tqdm(
    test_query_vectors.items(),
    desc="Ranking test candidates"
):
    scores = candidate_matrix @ qvec.to(DEVICE)

    top_indices = torch.topk(
        scores,
        k=TOP_K
    ).indices.tolist()

    predictions[qid] = [
        candidate_ids[i] for i in top_indices
    ]

# -------------------- Validate submission --------------------

assert len(predictions) == len(test_data)

for qid, ranked_ids in predictions.items():
    assert len(ranked_ids) == TOP_K
    assert len(set(ranked_ids)) == TOP_K
    assert all(cid in test_candidates for cid in ranked_ids)

# -------------------- Save JSON and ZIP --------------------

with open(OUTPUT_JSON, "w") as f:
    json.dump(predictions, f, indent=2)

with zipfile.ZipFile(
    OUTPUT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:
    zf.write(OUTPUT_JSON, arcname="prediction.json")

print("\nSubmission generated successfully!")
print("Queries:", len(predictions))
print("Candidates per query:", TOP_K)
print("ZIP:", OUTPUT_ZIP)
print("ZIP size: {:.2f} MB".format(os.path.getsize(OUTPUT_ZIP) / (1024 * 1024)))
print("\nSample prediction:")
print(next(iter(predictions.items())))

Test queries: 237
Test candidates: 1727


Tokenizing test queries:   0%|          | 0/237 [00:00<?, ?it/s]

Tokenizing test candidates:   0%|          | 0/1727 [00:00<?, ?it/s]

Tokenization completed.
Loaded best checkpoint from epoch: 2
Best Dev Micro-F1@7: 0.2123401889938855


Encoding test queries:   0%|          | 0/117 [00:00<?, ?it/s]

Encoding test candidates:   0%|          | 0/839 [00:00<?, ?it/s]

Ranking test candidates:   0%|          | 0/237 [00:00<?, ?it/s]


Submission generated successfully!
Queries: 237
Candidates per query: 7
ZIP: /kaggle/working/submission_m1_p7.zip
ZIP size: 0.01 MB

Sample prediction:
('0000021652', ['0000021652', '0001481813', '0000590455', '0001505859', '0001000760', '0001486969', '0001496005'])
